deepseek

In [1]:
"""
synthetic_reservoir_init.py

Research-grade synthetic 3D petroleum reservoir initialization.

This module generates a physically plausible initial reservoir model:
- stratigraphic facies
- correlated porosity/permeability
- explicit faults and equivalent-continuum fractures
- hydrostatic pressure
- initial oil-water saturations
- vertical wells

It is intended to be consumed later by a finite-volume black-oil simulator.
"""

import os
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from scipy.ndimage import gaussian_filter


# ---------------------------------------------------------------------------
# Facies definitions
# ---------------------------------------------------------------------------
# All permeability values are in millidarcy (mD).
# Porosity is fraction.
# These ranges are SYNTHETIC and ILLUSTRATIVE, not field-calibrated.
FACIES_DEFS = {
    0: {
        "name": "shale",
        "phi_mean": 0.04, "phi_std": 0.015, "phi_min": 0.01, "phi_max": 0.10,
        "logk_mean": -3.0, "logk_std": 0.8, "logk_phi_slope": 10.0,
        "k_min": 1e-5, "k_max": 1e-2,
        "ky_kx": 0.80, "kz_kx": 0.10,
        "color": "gray"
    },
    1: {
        "name": "shaly_sandstone",
        "phi_mean": 0.12, "phi_std": 0.025, "phi_min": 0.06, "phi_max": 0.20,
        "logk_mean": 0.0, "logk_std": 0.7, "logk_phi_slope": 15.0,
        "k_min": 1e-3, "k_max": 10.0,
        "ky_kx": 0.80, "kz_kx": 0.15,
        "color": "tan"
    },
    2: {
        "name": "sandstone",
        "phi_mean": 0.21, "phi_std": 0.03, "phi_min": 0.12, "phi_max": 0.30,
        "logk_mean": 2.0, "logk_std": 0.5, "logk_phi_slope": 20.0,
        "k_min": 1.0, "k_max": 1000.0,
        "ky_kx": 0.90, "kz_kx": 0.30,
        "color": "yellow"
    },
    3: {
        "name": "channel_sandstone",
        "phi_mean": 0.28, "phi_std": 0.03, "phi_min": 0.20, "phi_max": 0.38,
        "logk_mean": 3.0, "logk_std": 0.4, "logk_phi_slope": 25.0,
        "k_min": 100.0, "k_max": 10000.0,
        "ky_kx": 1.00, "kz_kx": 0.50,
        "color": "orange"
    }
}


# ---------------------------------------------------------------------------
# Grid
# ---------------------------------------------------------------------------
def make_grid(Nx=40, Ny=40, Nz=10,
              Lx=1000.0, Ly=1000.0, Lz=50.0,
              depth_top=1500.0):
    """
    Create a cell-centered 3D grid.

    Depth convention: z is positive downward, measured from a reference datum.
    Cell centers are at (i+0.5)*dx, etc.
    """
    dx = Lx / Nx
    dy = Ly / Ny
    dz = Lz / Nz

    xc = (np.arange(Nx) + 0.5) * dx
    yc = (np.arange(Ny) + 0.5) * dy
    zc = depth_top + (np.arange(Nz) + 0.5) * dz

    X, Y, Z = np.meshgrid(xc, yc, zc, indexing="ij")

    return {
        "Nx": Nx, "Ny": Ny, "Nz": Nz,
        "Lx": Lx, "Ly": Ly, "Lz": Lz,
        "dx": dx, "dy": dy, "dz": dz,
        "xc": xc, "yc": yc, "zc": zc,
        "X": X, "Y": Y, "Z": Z,
        "depth_top": depth_top,
        "depth_bottom": depth_top + Lz,
    }


# ---------------------------------------------------------------------------
# Facies
# ---------------------------------------------------------------------------
def generate_facies(grid, seed=42):
    """
    Generate a layered stratigraphic facies model with meandering channels.

    Facies codes:
        0 = shale
        1 = shaly sandstone
        2 = sandstone
        3 = channel sandstone
    """
    Nx, Ny, Nz = grid["Nx"], grid["Ny"], grid["Nz"]
    xc, yc = grid["xc"], grid["yc"]
    Lx, Ly = grid["Lx"], grid["Ly"]

    rng = np.random.default_rng(seed)
    facies = np.zeros((Nx, Ny, Nz), dtype=np.int8)

    # Simple layer template from top to bottom.
    # For Nz=10 this gives alternating shale/sand/channel architecture.
    layer_template = np.array([0, 1, 1, 2, 3, 2, 2, 1, 1, 0], dtype=np.int8)
    if Nz != len(layer_template):
        # Interpolate or tile if a different Nz is used.
        layer_template = np.interp(
            np.linspace(0, len(layer_template) - 1, Nz),
            np.arange(len(layer_template)),
            layer_template
        ).round().astype(np.int8)

    for k in range(Nz):
        facies[:, :, k] = layer_template[k]

    # Add meandering channel bodies in channel-prone layers.
    channel_layers = [k for k in range(Nz) if layer_template[k] == 3]
    for k in channel_layers:
        phase = rng.uniform(0, 2 * np.pi)
        amp = 0.15 * Ly
        lam = 0.80 * Lx
        y0 = 0.50 * Ly
        width = 80.0
        for i, x in enumerate(xc):
            y_center = y0 + amp * np.sin(2 * np.pi * x / lam + phase)
            dist = np.abs(yc - y_center)
            facies[i, dist < width / 2.0, k] = 3
            facies[i, (dist >= width / 2.0) & (dist < width), k] = 1

    return facies


# ---------------------------------------------------------------------------
# Porosity
# ---------------------------------------------------------------------------
def _correlated_normal(shape, sigma, rng):
    """Return a standardized spatially correlated Gaussian field."""
    white = rng.normal(size=shape)
    corr = gaussian_filter(white, sigma=sigma, mode="reflect")
    corr = (corr - corr.mean()) / (corr.std() + 1e-12)
    return corr


def generate_porosity(facies, grid, seed=42):
    """Generate facies-dependent, spatially correlated porosity."""
    Nx, Ny, Nz = facies.shape
    rng = np.random.default_rng(seed + 1)
    phi = np.zeros((Nx, Ny, Nz), dtype=float)

    for code, props in FACIES_DEFS.items():
        mask = facies == code
        if not np.any(mask):
            continue
        corr = _correlated_normal(facies.shape, sigma=(2.0, 2.0, 1.0), rng=rng)
        phi[mask] = props["phi_mean"] + props["phi_std"] * corr[mask]
        phi[mask] = np.clip(phi[mask], props["phi_min"], props["phi_max"])

    return phi


# ---------------------------------------------------------------------------
# Permeability
# ---------------------------------------------------------------------------
def generate_permeability(phi, facies, seed=42):
    """
    Generate facies-conditioned permeability.

    kx, ky, kz are returned in mD.
    Permeability is related to porosity but NOT uniquely determined by it:
    a correlated heterogeneity term is added.
    """
    rng = np.random.default_rng(seed + 2)
    kx = np.zeros_like(phi, dtype=float)
    ky = np.zeros_like(phi, dtype=float)
    kz = np.zeros_like(phi, dtype=float)

    for code, props in FACIES_DEFS.items():
        mask = facies == code
        if not np.any(mask):
            continue

        corr = _correlated_normal(phi.shape, sigma=(2.0, 2.0, 1.0), rng=rng)
        logk = (
            props["logk_mean"]
            + props["logk_phi_slope"] * (phi[mask] - props["phi_mean"])
            + props["logk_std"] * corr[mask]
        )
        kx[mask] = 10.0 ** logk
        kx[mask] = np.clip(kx[mask], props["k_min"], props["k_max"])

    for code, props in FACIES_DEFS.items():
        mask = facies == code
        if not np.any(mask):
            continue
        n = int(mask.sum())
        ratio_y = props["ky_kx"] * (1.0 + 0.10 * rng.normal(size=n))
        ratio_z = props["kz_kx"] * (1.0 + 0.10 * rng.normal(size=n))
        ky[mask] = kx[mask] * np.clip(ratio_y, 0.05, 1.5)
        kz[mask] = kx[mask] * np.clip(ratio_z, 0.01, 1.0)

    ky = np.maximum(ky, 1e-6)
    kz = np.maximum(kz, 1e-6)
    return kx, ky, kz


# ---------------------------------------------------------------------------
# Faults
# ---------------------------------------------------------------------------
def create_faults(grid, fault_specs=None):
    """
    Create explicit 3D fault planes.

    Each fault stores:
    - cell_mask
    - face masks for x/y/z faces
    - transmissibility multipliers for x/y/z faces

    The matrix permeability is NOT modified. The future simulator should apply
    the face transmissibility multipliers directly.
    """
    if fault_specs is None:
        fault_specs = [
            {
                "name": "F1",
                "type": "plane",
                "point": (500.0, 500.0, 1525.0),
                "normal": (1.0, 0.0, 0.2),
                "thickness": 4.0,
                "transmissibility_multiplier": 0.01,
            }
        ]

    X, Y, Z = grid["X"], grid["Y"], grid["Z"]
    Nx, Ny, Nz = grid["Nx"], grid["Ny"], grid["Nz"]

    faults = []

    for spec in fault_specs:
        nx, ny, nz = np.array(spec["normal"], dtype=float)
        norm = np.sqrt(nx * nx + ny * ny + nz * nz)
        nx, ny, nz = nx / norm, ny / norm, nz / norm
        x0, y0, z0 = spec["point"]

        d = nx * (X - x0) + ny * (Y - y0) + nz * (Z - z0)
        cell_mask = np.abs(d) <= spec["thickness"] / 2.0

        face_mask_x = np.zeros((Nx + 1, Ny, Nz), dtype=bool)
        face_mask_y = np.zeros((Nx, Ny + 1, Nz), dtype=bool)
        face_mask_z = np.zeros((Nx, Ny, Nz + 1), dtype=bool)

        face_mask_x[1:-1, :, :] = (d[:-1, :, :] * d[1:, :, :]) <= 0
        face_mask_y[:, 1:-1, :] = (d[:, :-1, :] * d[:, 1:, :]) <= 0
        face_mask_z[:, :, 1:-1] = (d[:, :, :-1] * d[:, :, 1:]) <= 0

        tmult_x = np.ones((Nx + 1, Ny, Nz))
        tmult_y = np.ones((Nx, Ny + 1, Nz))
        tmult_z = np.ones((Nx, Ny, Nz + 1))

        tmult_x[face_mask_x] = spec["transmissibility_multiplier"]
        tmult_y[face_mask_y] = spec["transmissibility_multiplier"]
        tmult_z[face_mask_z] = spec["transmissibility_multiplier"]

        faults.append({
            "name": spec["name"],
            "type": spec["type"],
            "point": spec["point"],
            "normal": spec["normal"],
            "thickness": spec["thickness"],
            "transmissibility_multiplier": spec["transmissibility_multiplier"],
            "cell_mask": cell_mask,
            "face_mask_x": face_mask_x,
            "face_mask_y": face_mask_y,
            "face_mask_z": face_mask_z,
            "tmult_x": tmult_x,
            "tmult_y": tmult_y,
            "tmult_z": tmult_z,
        })

    return faults


# ---------------------------------------------------------------------------
# Fractures
# ---------------------------------------------------------------------------
def create_fractures(grid, fracture_specs=None):
    """
    Create explicit fracture corridors.

    This is an EQUIVALENT-CONTINUUM representation, not a discrete fracture
    model. High-permeability multiplier cells are stored separately. The future
    simulator can apply them to the matrix permeability or to face
    transmissibilities.
    """
    if fracture_specs is None:
        fracture_specs = [
            {
                "name": "FR1",
                "type": "vertical_plane",
                "point": (300.0, 300.0, 1525.0),
                "normal": (1.0, 1.0, 0.0),
                "effective_width": 10.0,
                "length": 600.0,
                "height": 40.0,
                "aperture": 0.01,
                "perm_multiplier": 100.0,
            },
            {
                "name": "FR2",
                "type": "vertical_plane",
                "point": (700.0, 700.0, 1525.0),
                "normal": (1.0, -1.0, 0.0),
                "effective_width": 8.0,
                "length": 500.0,
                "height": 35.0,
                "aperture": 0.008,
                "perm_multiplier": 80.0,
            },
        ]

    X, Y, Z = grid["X"], grid["Y"], grid["Z"]
    Nx, Ny, Nz = grid["Nx"], grid["Ny"], grid["Nz"]

    fractures = []
    combined_mask = np.zeros((Nx, Ny, Nz), dtype=bool)
    combined_mult = np.ones((Nx, Ny, Nz), dtype=float)

    for spec in fracture_specs:
        nx, ny, nz = np.array(spec["normal"], dtype=float)
        norm = np.sqrt(nx * nx + ny * ny + nz * nz)
        nx, ny, nz = nx / norm, ny / norm, nz / norm
        x0, y0, z0 = spec["point"]

        d = nx * (X - x0) + ny * (Y - y0) + nz * (Z - z0)
        s = -ny * (X - x0) + nx * (Y - y0)

        mask = (
            (np.abs(d) <= spec["effective_width"] / 2.0)
            & (np.abs(s) <= spec["length"] / 2.0)
            & (np.abs(Z - z0) <= spec["height"] / 2.0)
        )

        fractures.append({
            "name": spec["name"],
            "type": spec["type"],
            "point": spec["point"],
            "normal": spec["normal"],
            "effective_width": spec["effective_width"],
            "length": spec["length"],
            "height": spec["height"],
            "aperture": spec["aperture"],
            "perm_multiplier": spec["perm_multiplier"],
            "cell_mask": mask,
        })

        combined_mask |= mask
        combined_mult[mask] = np.maximum(combined_mult[mask], spec["perm_multiplier"])

    return fractures, combined_mask, combined_mult


# ---------------------------------------------------------------------------
# Initial pressure
# ---------------------------------------------------------------------------
def initial_pressure(grid, owc, p_ref=15e6,
                     rho_o=800.0, rho_w=1000.0, g=9.81):
    """
    Hydrostatic pressure field.

    Sign convention: z is positive downward.
    p_ref is the pressure at the OWC (z_ref = owc).

    Above OWC: P = p_ref + rho_o * g * (z - owc)
    Below OWC: P = p_ref + rho_w * g * (z - owc)
    """
    Z = grid["Z"]
    P = np.where(
        Z < owc,
        p_ref + rho_o * g * (Z - owc),
        p_ref + rho_w * g * (Z - owc)
    )
    return P


# ---------------------------------------------------------------------------
# Initial saturations
# ---------------------------------------------------------------------------
def initial_saturations(grid, facies, owc, swc=0.20, transition=2.0):
    """
    Initial oil-water saturations with a smooth OWC transition.

    Sg is zero initially. Sw + So + Sg = 1 exactly.
    """
    Z = grid["Z"]
    Sw = swc + (1.0 - swc) * 0.5 * (1.0 + np.tanh((Z - owc) / max(transition, 1e-6)))
    So = 1.0 - Sw
    Sg = np.zeros_like(Sw)
    return Sw, So, Sg


# ---------------------------------------------------------------------------
# Wells
# ---------------------------------------------------------------------------
def create_wells(grid, facies, So, owc, well_specs=None):
    """
    Create vertical wells and choose perforations.

    Perforations are selected only in non-shale cells with sufficient oil
    saturation. If no such cells exist in the requested interval, all cells in
    the interval are used as a fallback.
    """
    if well_specs is None:
        well_specs = [
            {
                "name": "INJ1", "type": "injector",
                "x": 100.0, "y": 100.0,
                "k_min": 0, "k_max": grid["Nz"] - 1,
                "control": "rate", "value": 200.0, "bhp": None,
            },
            {
                "name": "PROD1", "type": "producer",
                "x": 900.0, "y": 900.0,
                "k_min": 0, "k_max": grid["Nz"] - 1,
                "control": "bhp", "value": 10e6, "bhp": 10e6,
            },
        ]

    wells = []

    for spec in well_specs:
        i = int(np.floor(spec["x"] / grid["dx"]))
        j = int(np.floor(spec["y"] / grid["dy"]))
        i = max(0, min(i, grid["Nx"] - 1))
        j = max(0, min(j, grid["Ny"] - 1))

        perfs = []
        for k in range(spec["k_min"], spec["k_max"] + 1):
            if 0 <= k < grid["Nz"]:
                if facies[i, j, k] != 0 and So[i, j, k] > 0.05:
                    perfs.append(k)

        if not perfs:
            perfs = [k for k in range(spec["k_min"], spec["k_max"] + 1)
                     if 0 <= k < grid["Nz"]]

        wells.append({
            "name": spec["name"],
            "type": spec["type"],
            "x": spec["x"],
            "y": spec["y"],
            "i": i,
            "j": j,
            "perforations": perfs,
            "control": spec["control"],
            "value": spec.get("value"),
            "bhp": spec.get("bhp"),
        })

    return wells


# ---------------------------------------------------------------------------
# Build reservoir
# ---------------------------------------------------------------------------
def build_reservoir(Nx=40, Ny=40, Nz=10,
                    Lx=1000.0, Ly=1000.0, Lz=50.0,
                    depth_top=1500.0,
                    seed=42,
                    owc=1525.0,
                    p_ref=15e6,
                    swc=0.20,
                    well_specs=None,
                    fault_specs=None,
                    fracture_specs=None):
    """
    Build the full synthetic reservoir initialization object.
    """
    grid = make_grid(Nx, Ny, Nz, Lx, Ly, Lz, depth_top)

    facies = generate_facies(grid, seed=seed)
    phi = generate_porosity(facies, grid, seed=seed)
    kx, ky, kz = generate_permeability(phi, facies, seed=seed)

    faults = create_faults(grid, fault_specs=fault_specs)
    fractures, frac_mask, frac_mult = create_fractures(
        grid, fracture_specs=fracture_specs
    )

    pressure = initial_pressure(grid, owc=owc, p_ref=p_ref)
    Sw, So, Sg = initial_saturations(grid, facies, owc=owc, swc=swc)

    wells = create_wells(grid, facies, So, owc=owc, well_specs=well_specs)

    res = {
        "grid": grid,
        "facies": facies,
        "phi": phi,
        "kx": kx,
        "ky": ky,
        "kz": kz,
        "faults": faults,
        "fractures": fractures,
        "fracture_mask": frac_mask,
        "fracture_perm_multiplier": frac_mult,
        "initial_state": {
            "pressure": pressure,
            "Sw": Sw,
            "So": So,
            "Sg": Sg,
        },
        "wells": wells,
        "metadata": {
            "seed": seed,
            "owc": owc,
            "p_ref": p_ref,
            "z_ref": owc,
            "swc": swc,
            "facies_defs": FACIES_DEFS,
            "note": (
                "Synthetic illustrative model. Not a real field. "
                "All petrophysical ranges are illustrative unless cited."
            ),
        },
    }

    validate_reservoir(res)
    return res


# ---------------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------------
def validate_reservoir(res):
    """Run physical and geometric consistency checks."""
    errors = []

    grid = res["grid"]
    Nx, Ny, Nz = grid["Nx"], grid["Ny"], grid["Nz"]

    # Porosity bounds
    for code, props in FACIES_DEFS.items():
        mask = res["facies"] == code
        if np.any(mask):
            if (np.any(res["phi"][mask] < props["phi_min"] - 1e-10) or
                    np.any(res["phi"][mask] > props["phi_max"] + 1e-10)):
                errors.append(f"Porosity out of bounds for facies {props['name']}")

    # Permeability positivity
    for name in ["kx", "ky", "kz"]:
        if np.any(res[name] <= 0):
            errors.append(f"{name} <= 0")

    # Saturation sum and bounds
    Sw = res["initial_state"]["Sw"]
    So = res["initial_state"]["So"]
    Sg = res["initial_state"]["Sg"]
    if not np.allclose(Sw + So + Sg, 1.0, atol=1e-8):
        errors.append("Saturations do not sum to 1")
    if np.any(Sw < -1e-10) or np.any(So < -1e-10) or np.any(Sg < -1e-10):
        errors.append("Negative saturation detected")

    # Pressure increases with depth
    P = res["initial_state"]["pressure"]
    if np.any(np.diff(P[:, 0, 0]) < -1e-6):
        errors.append("Pressure decreases with depth in test column")

    # Wells inside grid and perforations valid
    for w in res["wells"]:
        if not (0 <= w["i"] < Nx and 0 <= w["j"] < Ny):
            errors.append(f"Well {w['name']} outside grid")
        for k in w["perforations"]:
            if not (0 <= k < Nz):
                errors.append(f"Well {w['name']} perforation {k} outside grid")

    # Fault mask shapes
    for f in res["faults"]:
        if f["cell_mask"].shape != (Nx, Ny, Nz):
            errors.append(f"Fault {f['name']} cell mask shape mismatch")
        if f["tmult_x"].shape != (Nx + 1, Ny, Nz):
            errors.append(f"Fault {f['name']} tmult_x shape mismatch")
        if f["tmult_y"].shape != (Nx, Ny + 1, Nz):
            errors.append(f"Fault {f['name']} tmult_y shape mismatch")
        if f["tmult_z"].shape != (Nx, Ny, Nz + 1):
            errors.append(f"Fault {f['name']} tmult_z shape mismatch")

    # Fracture mask shape
    if res["fracture_mask"].shape != (Nx, Ny, Nz):
        errors.append("Fracture mask shape mismatch")

    if errors:
        raise ValueError("Reservoir validation failed:\n" + "\n".join(errors))

    return True


# ---------------------------------------------------------------------------
# Visualization
# ---------------------------------------------------------------------------
def visualize_reservoir(res, outdir="reservoir_output"):
    """Generate a set of diagnostic figures."""
    os.makedirs(outdir, exist_ok=True)

    grid = res["grid"]
    Nx, Ny, Nz = grid["Nx"], grid["Ny"], grid["Nz"]
    Lx, Ly, Lz = grid["Lx"], grid["Ly"], grid["Lz"]
    X, Y, Z = grid["X"], grid["Y"], grid["Z"]

    # 1. 3D facies
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")
    for code, props in FACIES_DEFS.items():
        mask = res["facies"] == code
        if np.any(mask):
            idx = np.where(mask)
            step = max(1, len(idx[0]) // 2000)
            ax.scatter(X[idx][::step], Y[idx][::step], Z[idx][::step],
                       c=props["color"], label=props["name"], s=2, alpha=0.3)
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_zlabel("Depth (m)")
    ax.invert_zaxis()
    ax.legend(loc="upper right", fontsize=8)
    ax.set_title("3D Facies Model (synthetic)")
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "01_facies_3d.png"), dpi=150)
    plt.close()

    # 2. Porosity slice
    k = Nz // 2
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(res["phi"][:, :, k].T, origin="lower",
                   extent=[0, Lx, 0, Ly], cmap="viridis", aspect="equal")
    plt.colorbar(im, ax=ax, label="Porosity")
    ax.set_title(f"Porosity slice k={k}")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "02_porosity_slice.png"), dpi=150)
    plt.close()

    # 3. Permeability slice
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(np.log10(res["kx"][:, :, k].T), origin="lower",
                   extent=[0, Lx, 0, Ly], cmap="plasma", aspect="equal")
    plt.colorbar(im, ax=ax, label="log10 kx (mD)")
    ax.set_title(f"Permeability slice k={k}")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "03_permeability_slice.png"), dpi=150)
    plt.close()

    # 4. 3D fault geometry
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")
    for fault in res["faults"]:
        mask = fault["cell_mask"]
        if np.any(mask):
            idx = np.where(mask)
            step = max(1, len(idx[0]) // 1000)
            ax.scatter(X[idx][::step], Y[idx][::step], Z[idx][::step],
                       c="red", s=3, alpha=0.5, label=fault["name"])
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_zlabel("Depth (m)")
    ax.invert_zaxis()
    ax.legend()
    ax.set_title("3D Fault Geometry")
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "04_fault_3d.png"), dpi=150)
    plt.close()

    # 5. 3D fracture geometry
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")
    mask = res["fracture_mask"]
    if np.any(mask):
        idx = np.where(mask)
        step = max(1, len(idx[0]) // 1000)
        ax.scatter(X[idx][::step], Y[idx][::step], Z[idx][::step],
                   c="cyan", s=3, alpha=0.5, label="fractures")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_zlabel("Depth (m)")
    ax.invert_zaxis()
    ax.legend()
    ax.set_title("3D Fracture Corridors (equivalent continuum)")
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "05_fracture_3d.png"), dpi=150)
    plt.close()

    # 6. Wells
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")
    for w in res["wells"]:
        x, y = w["x"], w["y"]
        perfs = w["perforations"]
        if perfs:
            k_min, k_max = min(perfs), max(perfs)
            z_top = grid["zc"][k_min] - grid["dz"] / 2.0
            z_bot = grid["zc"][k_max] + grid["dz"] / 2.0
        else:
            z_top = grid["depth_top"]
            z_bot = grid["depth_bottom"]
        color = "blue" if w["type"] == "injector" else "red"
        ax.plot([x, x], [y, y], [z_top, z_bot], color=color, linewidth=3,
                label=w["name"])
        ax.scatter([x], [y], [z_top], color=color, marker="^", s=50)
        ax.scatter([x], [y], [z_bot], color=color, marker="v", s=50)
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_zlabel("Depth (m)")
    ax.invert_zaxis()
    ax.legend()
    ax.set_title("Wells (vertical)")
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "06_wells_3d.png"), dpi=150)
    plt.close()

    # 7. Initial pressure with depth
    fig, ax = plt.subplots(figsize=(6, 7))
    P = res["initial_state"]["pressure"][:, Ny // 2, :]
    Zcol = Z[:, Ny // 2, :]
    ax.scatter(P.flatten(), Zcol.flatten(), s=2, alpha=0.5, label="cells")
    ax.axhline(res["metadata"]["owc"], color="cyan", linestyle="--",
               label="OWC")
    ax.set_xlabel("Pressure (Pa)")
    ax.set_ylabel("Depth (m)")
    ax.invert_yaxis()
    ax.legend()
    ax.set_title("Initial Pressure vs Depth")
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "07_pressure_depth.png"), dpi=150)
    plt.close()

    # 8. Initial saturation / OWC
    fig, ax = plt.subplots(figsize=(8, 5))
    j = Ny // 2
    Sw_slice = res["initial_state"]["Sw"][:, j, :].T
    im = ax.imshow(Sw_slice, origin="lower",
                   extent=[0, Lx, grid["depth_top"], grid["depth_top"] + Lz],
                   cmap="Blues", aspect="auto")
    plt.colorbar(im, ax=ax, label="Sw")
    ax.axhline(res["metadata"]["owc"], color="red", linestyle="--",
               label="OWC")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Depth (m)")
    ax.invert_yaxis()
    ax.legend()
    ax.set_title("Initial Water Saturation Cross-Section")
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "08_saturation_owc.png"), dpi=150)
    plt.close()


# ---------------------------------------------------------------------------
# Output
# ---------------------------------------------------------------------------
def save_reservoir(res, outdir="reservoir_output"):
    """Save arrays to NPZ and metadata to JSON."""
    os.makedirs(outdir, exist_ok=True)

    arrays = {
        "facies": res["facies"],
        "phi": res["phi"],
        "kx": res["kx"],
        "ky": res["ky"],
        "kz": res["kz"],
        "pressure": res["initial_state"]["pressure"],
        "Sw": res["initial_state"]["Sw"],
        "So": res["initial_state"]["So"],
        "Sg": res["initial_state"]["Sg"],
        "fracture_mask": res["fracture_mask"],
        "fracture_perm_multiplier": res["fracture_perm_multiplier"],
    }

    for fi, fault in enumerate(res["faults"]):
        arrays[f"fault_{fi}_cell_mask"] = fault["cell_mask"]
        arrays[f"fault_{fi}_tmult_x"] = fault["tmult_x"]
        arrays[f"fault_{fi}_tmult_y"] = fault["tmult_y"]
        arrays[f"fault_{fi}_tmult_z"] = fault["tmult_z"]

    np.savez_compressed(os.path.join(outdir, "reservoir_arrays.npz"), **arrays)

    meta = {
        "grid": {k: v for k, v in res["grid"].items()
                 if k not in ["X", "Y", "Z"]},
        "seed": res["metadata"]["seed"],
        "owc": res["metadata"]["owc"],
        "p_ref": res["metadata"]["p_ref"],
        "z_ref": res["metadata"]["z_ref"],
        "swc": res["metadata"]["swc"],
        "facies_defs": res["metadata"]["facies_defs"],
        "faults": [
            {k: v for k, v in f.items()
             if k not in ["cell_mask", "face_mask_x", "face_mask_y",
                          "face_mask_z", "tmult_x", "tmult_y", "tmult_z"]}
            for f in res["faults"]
        ],
        "fractures": [
            {k: v for k, v in f.items() if k not in ["cell_mask"]}
            for f in res["fractures"]
        ],
        "wells": res["wells"],
        "note": res["metadata"]["note"],
    }

    def default(o):
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, (np.integer, np.floating)):
            return o.item()
        return str(o)

    with open(os.path.join(outdir, "reservoir_metadata.json"), "w") as f:
        json.dump(meta, f, indent=2, default=default)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    outdir = "reservoir_output"

    reservoir = build_reservoir(
        Nx=40, Ny=40, Nz=10,
        Lx=1000.0, Ly=1000.0, Lz=50.0,
        depth_top=1500.0,
        seed=42,
        owc=1525.0,
        p_ref=15e6,
        swc=0.20,
    )

    save_reservoir(reservoir, outdir=outdir)
    visualize_reservoir(reservoir, outdir=outdir)

    print(f"Saved synthetic reservoir to: {outdir}")
    print("Validation passed.")
    print(f"Grid: {reservoir['grid']['Nx']} x {reservoir['grid']['Ny']} x {reservoir['grid']['Nz']}")
    print(f"Wells: {[w['name'] for w in reservoir['wells']]}")
    print(f"Faults: {[f['name'] for f in reservoir['faults']]}")
    print(f"Fractures: {[f['name'] for f in reservoir['fractures']]}")

C:\Users\dhruv\AppData\Local\Temp\ipykernel_24200\1435239388.py:686: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend()


Saved synthetic reservoir to: reservoir_output
Validation passed.
Grid: 40 x 40 x 10
Wells: ['INJ1', 'PROD1']
Faults: ['F1']
Fractures: ['FR1', 'FR2']


gpt-6-sol

In [2]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # Also works on headless machines.
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
import numpy as np
from scipy.ndimage import gaussian_filter


# All numerical values in this module are illustrative synthetic choices,
# not measurements or calibrated properties from a real field.
FACIES = {
    0: {
        "name": "shale",
        "phi_range": (0.04, 0.12),
        "kx_range_mD": (0.0001, 0.01),
        "sw_above_owc": 0.85,
    },
    1: {
        "name": "shaly sandstone",
        "phi_range": (0.10, 0.20),
        "kx_range_mD": (0.05, 20.0),
        "sw_above_owc": 0.32,
    },
    2: {
        "name": "sandstone",
        "phi_range": (0.16, 0.28),
        "kx_range_mD": (20.0, 600.0),
        "sw_above_owc": 0.23,
    },
    3: {
        "name": "high-quality/channel sandstone",
        "phi_range": (0.22, 0.34),
        "kx_range_mD": (300.0, 3000.0),
        "sw_above_owc": 0.18,
    },
}

MD_TO_M2 = 9.869233e-16
COLORS = ["#55545b", "#a88b71", "#d9bd78", "#ed983b"]


def make_grid(nx=40, ny=40, nz=10, lx=1000.0, ly=1000.0,
              thickness=50.0, top_depth=2000.0):
    """Return a Cartesian grid. Arrays elsewhere have shape (nz, ny, nx).

    x and y are horizontal coordinates in metres; z is positive-down depth
    in metres. The grid origin is the northwest/southwest-independent (0, 0)
    horizontal corner; no geographic coordinates are implied.
    """
    if min(nx, ny, nz) < 2 or min(lx, ly, thickness) <= 0:
        raise ValueError("Grid counts must be >= 2 and dimensions positive.")

    dx, dy, dz = lx / nx, ly / ny, thickness / nz
    return {
        "nx": nx, "ny": ny, "nz": nz,
        "lx_m": lx, "ly_m": ly, "thickness_m": thickness,
        "top_depth_m": top_depth,
        "dx_m": dx, "dy_m": dy, "dz_m": dz,
        "x_m": (np.arange(nx) + 0.5) * dx,
        "y_m": (np.arange(ny) + 0.5) * dy,
        "z_depth_m": top_depth + (np.arange(nz) + 0.5) * dz,
        "array_order": "z,y,x",
        "depth_convention": "positive down",
    }


def cell_coordinates(grid):
    """Broadcastable cell-centre coordinate arrays, each shape (nz,ny,nx)."""
    shape = (grid["nz"], grid["ny"], grid["nx"])
    x = np.broadcast_to(grid["x_m"][None, None, :], shape)
    y = np.broadcast_to(grid["y_m"][None, :, None], shape)
    z = np.broadcast_to(grid["z_depth_m"][:, None, None], shape)
    return x, y, z


def make_facies(grid):
    """Layered stratigraphy with sinuous channel bodies in three layers."""
    nx, ny, nz = grid["nx"], grid["ny"], grid["nz"]
    if nz != 10:
        raise ValueError("This illustrative stratigraphic template requires nz=10.")

    # Top to bottom. Channels locally replace their host sandstone.
    layers = np.array([0, 1, 2, 2, 2, 2, 1, 2, 0, 1], dtype=np.uint8)
    facies = np.broadcast_to(layers[:, None, None], (nz, ny, nx)).copy()

    x = grid["x_m"][None, :]
    y = grid["y_m"][:, None]
    for k, half_width_m in [(3, 45.0), (4, 80.0), (5, 45.0)]:
        centreline_y = (
            0.50 * grid["ly_m"]
            + 105.0 * np.sin(2 * np.pi * 1.15 * x / grid["lx_m"]
                            + 0.13 * (k - 4))
        )
        facies[k, np.abs(y - centreline_y) <= half_width_m] = 3

    return facies


def correlated_field(rng, shape, sigma=(1.0, 3.0, 3.0)):
    """Unit-variance Gaussian-smoothed field; sigma is in (z,y,x) cells."""
    raw = gaussian_filter(rng.standard_normal(shape), sigma=sigma,
                          mode="reflect")
    std = raw.std()
    if std == 0:
        raise RuntimeError("Degenerate correlated random field.")
    return (raw - raw.mean()) / std


def make_petrophysics(facies, rng):
    """Generate bounded porosity and facies-conditioned permeability.

    Baseline kx is in mD. The separate correlated permeability field means
    porosity does not uniquely determine permeability.
    """
    shape = facies.shape
    phi_field = correlated_field(rng, shape)
    k_field = correlated_field(rng, shape)
    anisotropy_field = correlated_field(rng, shape)

    phi = np.empty(shape)
    kx_base = np.empty(shape)

    for code, definition in FACIES.items():
        mask = facies == code
        pmin, pmax = definition["phi_range"]
        kmin, kmax = definition["kx_range_mD"]

        # tanh is bounded, so assigned porosity remains inside facies bounds.
        phi[mask] = pmin + (pmax - pmin) * (
            0.5 + 0.46 * np.tanh(phi_field[mask] / 1.5)
        )
        phi_fraction = (phi[mask] - pmin) / (pmax - pmin)
        independent_fraction = 0.5 + 0.5 * np.tanh(k_field[mask] / 1.5)

        # Work in log(k): porosity is influential, but not determinative.
        log_fraction = (
            0.15
            + 0.70 * (0.55 * phi_fraction + 0.45 * independent_fraction)
        )
        kx_base[mask] = np.exp(
            np.log(kmin) + log_fraction * np.log(kmax / kmin)
        )

    # Illustrative horizontal and vertical anisotropy. Both stay positive;
    # vertical permeability is lower than either horizontal component.
    ky_base = kx_base * (0.65 + 0.50 * (
        0.5 + 0.5 * np.tanh(anisotropy_field / 1.5)
    ))
    kz_base = kx_base * (0.025 + 0.11 * (
        0.5 + 0.5 * np.tanh(anisotropy_field / 1.5)
    ))
    return phi, kx_base, ky_base, kz_base


def fault_signed_distance(grid, geometry):
    """Signed x-distance from the fault surface, evaluated at cell centres."""
    x, y, z = cell_coordinates(grid)
    return (x - geometry["x0_m"]
            - geometry["slope_x_per_y"] * (y - geometry["y0_m"])
            - geometry["slope_x_per_depth"] * (z - geometry["z0_depth_m"]))


def make_fault(grid, multiplier=0.02):
    """A full-height inclined plane; no matrix permeability is changed."""
    if multiplier <= 0:
        raise ValueError("Fault transmissibility multiplier must be positive.")

    geometry = {
        "equation": "x = x0 + slope_x_per_y*(y-y0) "
                    "+ slope_x_per_depth*(z-z0)",
        "x0_m": 510.0,
        "y0_m": 500.0,
        "z0_depth_m": 2025.0,
        "slope_x_per_y": 0.10,
        "slope_x_per_depth": 0.20,
    }
    distance = fault_signed_distance(grid, geometry)

    # A connection is crossed when its adjacent cell centres are on opposite
    # sides of the plane. Shapes match finite-volume x/y/z connections.
    face_masks = {
        "x": (distance[:, :, :-1] * distance[:, :, 1:]) <= 0,
        "y": (distance[:, :-1, :] * distance[:, 1:, :]) <= 0,
        "z": (distance[:-1, :, :] * distance[1:, :, :]) <= 0,
    }
    # Cell mask is for display/inspection. Face masks, not this mask, govern
    # fault transmissibility in a future finite-volume simulator.
    display_width = 0.5 * (
        grid["dx_m"]
        + abs(geometry["slope_x_per_y"]) * grid["dy_m"]
        + abs(geometry["slope_x_per_depth"]) * grid["dz_m"]
    )
    return {
        "name": "synthetic_inclined_sealing_fault",
        "geometry": geometry,
        "cell_mask": np.abs(distance) <= display_width,
        "face_masks": face_masks,
        "face_multipliers": {
            direction: np.where(mask, multiplier, 1.0)
            for direction, mask in face_masks.items()
        },
        "transmissibility_multiplier": multiplier,
    }


def fracture_geometry_mask(grid, spec):
    """Cell-centre equivalent-continuum rasterization of a vertical corridor."""
    x, y, z = cell_coordinates(grid)
    theta = np.deg2rad(spec["azimuth_deg_from_positive_x"])
    along = ((x - spec["start_x_m"]) * np.cos(theta)
             + (y - spec["start_y_m"]) * np.sin(theta))
    across = (-(x - spec["start_x_m"]) * np.sin(theta)
              + (y - spec["start_y_m"]) * np.cos(theta))
    return (
        (along >= 0)
        & (along <= spec["length_m"])
        & (np.abs(across) <= spec["effective_width_m"] / 2)
        & (z >= spec["top_depth_m"])
        & (z <= spec["bottom_depth_m"])
    )


def make_fractures(grid, facies):
    """Three explicit corridors; only non-shale cells receive k multipliers."""
    specs = [
        {
            "name": "corridor_A", "start_x_m": 130.0, "start_y_m": 170.0,
            "azimuth_deg_from_positive_x": 35.0, "length_m": 490.0,
            "effective_width_m": 38.0,
            "top_depth_m": 2008.0, "bottom_depth_m": 2037.0,
            "permeability_multiplier": 8.0,
        },
        {
            "name": "corridor_B", "start_x_m": 750.0, "start_y_m": 130.0,
            "azimuth_deg_from_positive_x": 105.0, "length_m": 540.0,
            "effective_width_m": 38.0,
            "top_depth_m": 2012.0, "bottom_depth_m": 2042.0,
            "permeability_multiplier": 12.0,
        },
        {
            "name": "corridor_C", "start_x_m": 310.0, "start_y_m": 760.0,
            "azimuth_deg_from_positive_x": -30.0, "length_m": 310.0,
            "effective_width_m": 38.0,
            "top_depth_m": 2010.0, "bottom_depth_m": 2035.0,
            "permeability_multiplier": 6.0,
        },
    ]

    fractures = []
    for spec in specs:
        geometry_mask = fracture_geometry_mask(grid, spec)
        fractures.append({
            **spec,
            "geometry_mask": geometry_mask,
            "applied_mask": geometry_mask & (facies != 0),
        })
    return fractures


def apply_fracture_multipliers(kx, ky, kz, fractures):
    """Equivalent-continuum property enhancement, not a discrete fracture."""
    factor = np.ones_like(kx)
    for fracture in fractures:
        # Overlapping corridors multiply; retain the result explicitly.
        factor[fracture["applied_mask"]] *= fracture["permeability_multiplier"]
    return kx * factor, ky * factor, kz * factor, factor


def make_initial_state(grid, facies, owc_depth_m=2035.0,
                       p_ref_pa=20_000_000.0, z_ref_depth_m=2000.0,
                       reference_density_kg_m3=850.0):
    """Oil/water initial state with positive-down hydrostatic pressure."""
    if not (grid["top_depth_m"] < owc_depth_m
            < grid["top_depth_m"] + grid["thickness_m"]):
        raise ValueError("OWC must lie within the reservoir.")

    z = cell_coordinates(grid)[2]
    g = 9.80665
    pressure = (p_ref_pa
                + reference_density_kg_m3 * g * (z - z_ref_depth_m))

    sw = np.empty_like(pressure)
    above = z < owc_depth_m
    for code, definition in FACIES.items():
        sw[(facies == code) & above] = definition["sw_above_owc"]
        sw[(facies == code) & ~above] = (
            0.995 if code == 0 else 0.98
        )

    sg = np.zeros_like(sw)
    so = 1.0 - sw - sg

    settings = {
        "owc_depth_m": owc_depth_m,
        "p_ref_pa": p_ref_pa,
        "z_ref_depth_m": z_ref_depth_m,
        "reference_density_kg_m3": reference_density_kg_m3,
        "gravity_m_s2": g,
        "pressure_model": "single-reference-density hydrostatic initialization",
    }
    return {"pressure": pressure, "Sw": sw, "So": so, "Sg": sg}, settings


def make_wells(grid, facies, initial_state, settings):
    """Vertical wells, with explicitly enumerated valid oil-zone completions."""
    sites = [
        ("INJ-1", "injector", 7, 10,
         {"type": "water_rate", "value": 250.0 / 86400.0,
          "units": "reservoir_m3/s", "positive_direction": "into_reservoir"}),
        ("PROD-1", "producer", 32, 29,
         {"type": "bhp", "value": 19_500_000.0, "units": "Pa"}),
    ]
    wells = []
    for name, kind, i, j, control in sites:
        perforations = [
            k for k in range(grid["nz"])
            if (facies[k, j, i] >= 2
                and grid["z_depth_m"][k] < settings["owc_depth_m"]
                and initial_state["So"][k, j, i] > 0.5)
        ]
        if not perforations:
            raise ValueError(f"No appropriate completions at {name}.")
        wells.append({
            "name": name, "kind": kind,
            "i": i, "j": j,
            "x_m": float(grid["x_m"][i]),
            "y_m": float(grid["y_m"][j]),
            "perforated_layer_interval_inclusive":
                [min(perforations), max(perforations)],
            "perforated_layers": perforations,
            "control": control,
        })
    return wells


def build_reservoir(seed=20250308):
    rng = np.random.default_rng(seed)
    grid = make_grid()
    facies = make_facies(grid)
    phi, kx_base, ky_base, kz_base = make_petrophysics(facies, rng)
    fault = make_fault(grid)
    fractures = make_fractures(grid, facies)
    kx, ky, kz, fracture_factor = apply_fracture_multipliers(
        kx_base, ky_base, kz_base, fractures
    )
    state, initial_settings = make_initial_state(grid, facies)
    wells = make_wells(grid, facies, state, initial_settings)

    reservoir = {
        "seed": seed,
        "grid": grid,
        "facies_definitions": FACIES,
        "facies": facies,
        "phi": phi,
        # Working permeability arrays are mD. Convert to m^2 at the
        # flow-solver interface with MD_TO_M2.
        "permeability_units": "mD",
        "kx": kx, "ky": ky, "kz": kz,
        "kx_baseline": kx_base,
        "ky_baseline": ky_base,
        "kz_baseline": kz_base,
        "faults": [fault],
        "fractures": fractures,
        "fracture_factor": fracture_factor,
        "initial_state": state,
        "initial_settings": initial_settings,
        "wells": wells,
    }
    validate_reservoir(reservoir)
    return reservoir


def validate_reservoir(r):
    g, facies = r["grid"], r["facies"]
    shape = (g["nz"], g["ny"], g["nx"])
    assert facies.shape == shape
    assert set(np.unique(facies)).issubset(FACIES)

    for key in ("phi", "kx", "ky", "kz", "kx_baseline",
                "ky_baseline", "kz_baseline", "fracture_factor"):
        assert r[key].shape == shape and np.all(np.isfinite(r[key]))

    for code, definition in FACIES.items():
        mask = facies == code
        pmin, pmax = definition["phi_range"]
        kmin, kmax = definition["kx_range_mD"]
        assert np.all((r["phi"][mask] >= pmin)
                      & (r["phi"][mask] <= pmax))
        assert np.all((r["kx_baseline"][mask] >= kmin)
                      & (r["kx_baseline"][mask] <= kmax))

    for key in ("kx", "ky", "kz", "kx_baseline",
                "ky_baseline", "kz_baseline"):
        assert np.all(r[key] > 0)
    assert np.all(r["kz"] < r["ky"])

    state = r["initial_state"]
    for key in ("pressure", "Sw", "So", "Sg"):
        assert state[key].shape == shape and np.all(np.isfinite(state[key]))
    for key in ("Sw", "So", "Sg"):
        assert np.all((state[key] >= 0) & (state[key] <= 1))
    assert np.allclose(state["Sw"] + state["So"] + state["Sg"], 1.0)
    assert np.all(np.diff(state["pressure"][:, 0, 0]) > 0)
    assert np.allclose(
        state["pressure"],
        r["initial_settings"]["p_ref_pa"]
        + r["initial_settings"]["reference_density_kg_m3"]
        * r["initial_settings"]["gravity_m_s2"]
        * (cell_coordinates(g)[2] - r["initial_settings"]["z_ref_depth_m"]),
    )

    expected_face_shapes = {
        "x": (g["nz"], g["ny"], g["nx"] - 1),
        "y": (g["nz"], g["ny"] - 1, g["nx"]),
        "z": (g["nz"] - 1, g["ny"], g["nx"]),
    }
    for fault in r["faults"]:
        assert fault["cell_mask"].shape == shape
        assert fault["cell_mask"].dtype == bool
        assert np.any(fault["cell_mask"])
        distance = fault_signed_distance(g, fault["geometry"])
        expected = {
            "x": (distance[:, :, :-1] * distance[:, :, 1:]) <= 0,
            "y": (distance[:, :-1, :] * distance[:, 1:, :]) <= 0,
            "z": (distance[:-1, :, :] * distance[1:, :, :]) <= 0,
        }
        for direction, face_shape in expected_face_shapes.items():
            mask = fault["face_masks"][direction]
            factors = fault["face_multipliers"][direction]
            assert mask.shape == factors.shape == face_shape
            assert mask.dtype == bool
            assert np.array_equal(mask, expected[direction])
            assert np.array_equal(
                factors,
                np.where(mask, fault["transmissibility_multiplier"], 1.0),
            )
        assert any(np.any(m) for m in fault["face_masks"].values())

    expected_factor = np.ones(shape)
    for fracture in r["fractures"]:
        geo, applied = fracture["geometry_mask"], fracture["applied_mask"]
        assert geo.shape == applied.shape == shape
        assert geo.dtype == applied.dtype == bool
        assert np.array_equal(geo, fracture_geometry_mask(g, fracture))
        assert np.array_equal(applied, geo & (facies != 0))
        assert np.any(applied)
        assert fracture["effective_width_m"] > 0
        assert fracture["length_m"] > 0
        assert fracture["permeability_multiplier"] > 0
        expected_factor[applied] *= fracture["permeability_multiplier"]
    assert np.allclose(r["fracture_factor"], expected_factor)
    for direction in ("x", "y", "z"):
        assert np.allclose(
            r[f"k{direction}"],
            r[f"k{direction}_baseline"] * expected_factor,
        )

    assert len({w["name"] for w in r["wells"]}) == len(r["wells"])
    for well in r["wells"]:
        i, j = well["i"], well["j"]
        assert 0 <= i < g["nx"] and 0 <= j < g["ny"]
        assert np.isclose(well["x_m"], g["x_m"][i])
        assert np.isclose(well["y_m"], g["y_m"][j])
        layers = well["perforated_layers"]
        assert layers and len(layers) == len(set(layers))
        assert well["perforated_layer_interval_inclusive"] == [
            min(layers), max(layers)
        ]
        for k in layers:
            assert 0 <= k < g["nz"]
            assert facies[k, j, i] >= 2
            assert g["z_depth_m"][k] < r["initial_settings"]["owc_depth_m"]
            assert state["So"][k, j, i] > 0.5


def save_reservoir(r, directory="synthetic_reservoir_output"):
    out = Path(directory)
    out.mkdir(parents=True, exist_ok=True)

    arrays = {
        "facies": r["facies"],
        "phi": r["phi"],
        "kx_mD": r["kx"], "ky_mD": r["ky"], "kz_mD": r["kz"],
        "kx_baseline_mD": r["kx_baseline"],
        "ky_baseline_mD": r["ky_baseline"],
        "kz_baseline_mD": r["kz_baseline"],
        "fracture_factor": r["fracture_factor"],
        **r["initial_state"],
        "x_m": r["grid"]["x_m"],
        "y_m": r["grid"]["y_m"],
        "z_depth_m": r["grid"]["z_depth_m"],
    }
    for n, fault in enumerate(r["faults"]):
        arrays[f"fault_{n}_cell_mask"] = fault["cell_mask"]
        for direction in ("x", "y", "z"):
            arrays[f"fault_{n}_face_mask_{direction}"] = (
                fault["face_masks"][direction]
            )
            arrays[f"fault_{n}_face_multiplier_{direction}"] = (
                fault["face_multipliers"][direction]
            )
    for n, fracture in enumerate(r["fractures"]):
        arrays[f"fracture_{n}_geometry_mask"] = fracture["geometry_mask"]
        arrays[f"fracture_{n}_applied_mask"] = fracture["applied_mask"]

    np.savez_compressed(out / "reservoir_arrays.npz", **arrays)

    metadata = {
        "description": "Illustrative synthetic reservoir; not a real field",
        "seed": r["seed"],
        "grid": {
            key: value for key, value in r["grid"].items()
            if not isinstance(value, np.ndarray)
        },
        "coordinate_units": "m",
        "pressure_units": "Pa",
        "permeability_units": "mD",
        "mD_to_m2": MD_TO_M2,
        "facies_definitions": r["facies_definitions"],
        "initial_settings": r["initial_settings"],
        "faults": [
            {
                "name": f["name"],
                "geometry": f["geometry"],
                "transmissibility_multiplier":
                    f["transmissibility_multiplier"],
            }
            for f in r["faults"]
        ],
        "fractures": [
            {
                key: value for key, value in fracture.items()
                if key not in ("geometry_mask", "applied_mask")
            }
            for fracture in r["fractures"]
        ],
        "wells": r["wells"],
        "npz_array_names": sorted(arrays),
        "fracture_model": "equivalent continuum; not discrete fractures",
        "fault_model": "connection transmissibility multipliers; "
                       "no stratigraphic throw",
    }
    with (out / "reservoir_metadata.json").open("w") as file:
        json.dump(metadata, file, indent=2)
    return out


def _new_3d():
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_zlabel("Depth (m)")
    ax.set_box_aspect((1, 1, 0.35))
    ax.invert_zaxis()
    return fig, ax


def _plot_cells(ax, grid, mask, color, size=5, alpha=0.7, label=None):
    k, j, i = np.nonzero(mask)
    ax.scatter(grid["x_m"][i], grid["y_m"][j], grid["z_depth_m"][k],
               s=size, c=color, alpha=alpha, label=label, depthshade=False)


def _save(fig, out, filename):
    fig.tight_layout()
    fig.savefig(out / filename, dpi=160)
    plt.close(fig)


def plot_reservoir(r, directory="synthetic_reservoir_output"):
    """Produce eight separate views; render only selected cells in 3D facies."""
    out = Path(directory)
    out.mkdir(parents=True, exist_ok=True)
    g, facies = r["grid"], r["facies"]
    nz, ny, nx = facies.shape
    x, y, z = cell_coordinates(g)

    # 1. Facies: external faces plus two interior cut planes.
    visible = np.zeros_like(facies, dtype=bool)
    visible[0] = True
    visible[-1] = True
    visible[:, 0, :] = True
    visible[:, :, 0] = True
    visible[:, ny // 2, :] = True
    visible[:, :, nx // 2] = True
    fig, ax = _new_3d()
    for code, color in enumerate(COLORS):
        _plot_cells(ax, g, visible & (facies == code), color,
                    size=4, alpha=0.48, label=FACIES[code]["name"])
    ax.legend(fontsize=7, loc="upper left")
    ax.set_title("Synthetic stratigraphy: boundary and cut-plane cells")
    _save(fig, out, "01_facies_3d.png")

    # 2. Porosity and 3. horizontal permeability: same representative layer.
    layer = 4
    for number, data, title, filename, cmap in [
        (2, r["phi"][layer], f"Porosity, layer {layer}",
         "02_porosity_slice.png", "viridis"),
        (3, np.log10(r["kx"][layer]),
         f"log10(kx / mD), layer {layer}",
         "03_permeability_slice.png", "magma"),
    ]:
        fig, ax = plt.subplots(figsize=(7, 6))
        image = ax.imshow(data, origin="lower",
                          extent=(0, g["lx_m"], 0, g["ly_m"]),
                          interpolation="nearest", cmap=cmap)
        ax.set(xlabel="x (m)", ylabel="y (m)", title=title)
        fig.colorbar(image, ax=ax)
        _save(fig, out, filename)

    # 4. Fault: plot connection midpoints, including x/y/z connections.
    fig, ax = _new_3d()
    fault = r["faults"][0]
    for direction, mask in fault["face_masks"].items():
        if direction == "x":
            points = ((x[:, :, :-1] + x[:, :, 1:]) / 2,
                      y[:, :, :-1], z[:, :, :-1])
        elif direction == "y":
            points = (x[:, :-1, :],
                      (y[:, :-1, :] + y[:, 1:, :]) / 2,
                      z[:, :-1, :])
        else:
            points = (x[:-1, :, :], y[:-1, :, :],
                      (z[:-1, :, :] + z[1:, :, :]) / 2)
        ax.scatter(*(point[mask] for point in points), s=10,
                   label=f"{direction}-faces")
    ax.legend()
    ax.set_title("Fault: crossed finite-volume faces")
    _save(fig, out, "04_fault_3d.png")

    # 5. Fracture geometry, not just property-changed cells.
    fig, ax = _new_3d()
    for fracture, color in zip(r["fractures"], ["red", "blue", "green"]):
        _plot_cells(ax, g, fracture["geometry_mask"], color,
                    size=8, label=fracture["name"])
    ax.legend()
    ax.set_title("Equivalent-continuum fracture corridors")
    _save(fig, out, "05_fractures_3d.png")

    # 6. Explicit vertical well tracks and completed cell centres.
    fig, ax = _new_3d()
    for well, color in zip(r["wells"], ["blue", "red"]):
        wx, wy = well["x_m"], well["y_m"]
        depths = g["z_depth_m"][well["perforated_layers"]]
        ax.plot([wx, wx], [wy, wy],
                [g["top_depth_m"], max(depths)], c=color, linewidth=3,
                label=well["name"])
        ax.scatter(np.full(len(depths), wx), np.full(len(depths), wy),
                   depths, c=color, s=65, marker="s")
    ax.set(xlim=(0, g["lx_m"]), ylim=(0, g["ly_m"]),
           title="Wells: tracks and perforations")
    ax.legend()
    _save(fig, out, "06_wells_3d.png")

    # 7. Hydrostatic depth profile.
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(r["initial_state"]["pressure"][:, 0, 0] / 1e6,
            g["z_depth_m"], marker="o")
    ax.invert_yaxis()
    ax.set(xlabel="Initial pressure (MPa)", ylabel="Depth (m)",
           title="Hydrostatic initial pressure; depth positive down")
    ax.grid(True)
    _save(fig, out, "07_pressure_depth.png")

    # 8. OWC and water saturation on a vertical section.
    fig, ax = plt.subplots(figsize=(8, 5))
    image = ax.imshow(
        r["initial_state"]["Sw"][:, ny // 2, :],
        origin="upper",
        extent=(0, g["lx_m"],
                g["top_depth_m"] + g["thickness_m"], g["top_depth_m"]),
        aspect="auto", vmin=0, vmax=1, cmap="Blues",
        interpolation="nearest",
    )
    ax.axhline(r["initial_settings"]["owc_depth_m"],
               color="red", linestyle="--", label="OWC")
    ax.set(xlabel="x (m)", ylabel="Depth (m)",
           title="Initial Sw and OWC; central y-section")
    ax.legend()
    fig.colorbar(image, ax=ax, label="Sw")
    _save(fig, out, "08_saturation_owc.png")


if __name__ == "__main__":
    reservoir = build_reservoir(seed=20250308)
    destination = save_reservoir(reservoir)
    plot_reservoir(reservoir, destination)
    print(f"Validated synthetic initialization saved to {destination.resolve()}")

Validated synthetic initialization saved to C:\Users\dhruv\code\Data\btp\notebook\synthetic_reservoir_output


fable-5.1-low

In [ ]:
"""
synthetic_reservoir.py
======================
Physically consistent INITIAL 3D reservoir model generator for a future
fully-implicit black-oil simulator.

Scope of this module (deliberately no flow solver):
  * Cartesian cell-centred grid with explicit depth (structural dome + dip)
  * Layered stratigraphy + meandering channel bodies (4 facies)
  * Facies-conditioned, spatially correlated porosity / anisotropic permeability
  * Explicit fault surfaces -> cell masks + FACE transmissibility multipliers
  * Explicit fracture corridors (equivalent-continuum, NOT DFM)
  * Hydrostatic two-fluid initial pressure, OWC-based saturations
  * Explicit vertical wells with facies-aware perforations
  * Validation, NPZ + JSON output, matplotlib visualisation

ALL NUMERICAL RANGES BELOW ARE ILLUSTRATIVE / SYNTHETIC. They are chosen to be
"textbook plausible" for a clastic reservoir but are NOT calibrated to any real
field and must not be presented as such.

Units (SI unless stated):  length m, pressure Pa, density kg/m3,
                           permeability mD (1 mD = 9.869233e-16 m2).
Depth convention:          depth D is POSITIVE DOWNWARD, layer k=0 is the TOP.
Array convention:          all cell arrays have shape (Nx, Ny, Nz), C-order,
                           meshgrid indexing='ij'.
"""
from __future__ import annotations

import json
import os
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional

import numpy as np

MD_TO_M2 = 9.869233e-16
G = 9.80665

# ----------------------------------------------------------------------------
# 0. CONFIGURATION
# ----------------------------------------------------------------------------
FACIES_CODES = {"shale": 0, "shaly_sandstone": 1, "sandstone": 2, "channel_sandstone": 3}
FACIES_NAMES = {v: k for k, v in FACIES_CODES.items()}


@dataclass
class FaciesParams:
    """Per-facies petrophysical distribution. ILLUSTRATIVE values."""
    name: str
    code: int
    phi_mean: float
    phi_std: float
    phi_min: float
    phi_max: float
    logk_a: float          # log10(k[mD]) = a + b*phi + sigma*Z   (Z: correlated N(0,1))
    logk_b: float
    logk_sigma: float      # residual scatter -> phi does NOT uniquely determine k
    k_min: float           # mD
    k_max: float           # mD
    kv_kh: float           # geometric-mean vertical/horizontal ratio
    kv_kh_sigma_log10: float
    swc: float             # connate water saturation
    net: bool              # False -> treated as non-reservoir (Sw = 1, never perforated)


def default_facies_table() -> Dict[int, FaciesParams]:
    return {
        0: FaciesParams("shale", 0, 0.05, 0.015, 0.01, 0.10, -3.0, 20.0, 0.40, 1e-4, 0.5, 0.05, 0.30, 1.00, False),
        1: FaciesParams("shaly_sandstone", 1, 0.13, 0.030, 0.06, 0.20, -1.5, 15.0, 0.50, 0.1, 50.0, 0.10, 0.30, 0.35, True),
        2: FaciesParams("sandstone", 2, 0.20, 0.035, 0.12, 0.28, -0.5, 12.0, 0.40, 5.0, 1000.0, 0.20, 0.25, 0.22, True),
        3: FaciesParams("channel_sandstone", 3, 0.26, 0.030, 0.18, 0.34, 0.3, 11.0, 0.35, 100.0, 8000.0, 0.35, 0.20, 0.15, True),
    }


@dataclass
class ReservoirConfig:
    # --- geometry ---
    Nx: int = 40
    Ny: int = 40
    Nz: int = 10
    Lx: float = 1000.0
    Ly: float = 1000.0
    Lz: float = 50.0
    top_depth: float = 2000.0          # depth of model top at (x,y)=(0,0) before dome
    dip_x: float = 0.005               # structural dip (m/m), +x deepens
    dip_y: float = 0.003
    dome_center: tuple = (600.0, 400.0)
    dome_amplitude: float = 15.0       # crest is shallower by this much
    dome_radius: float = 350.0
    # --- stratigraphy (top -> bottom, one code per layer) ---
    layer_sequence: List[int] = field(default_factory=lambda: [0, 1, 2, 2, 0, 1, 2, 2, 1, 0])
    lateral_shaliness_threshold: float = -0.9   # in Z-units of a correlated field
    lateral_cleaning_threshold: float = 1.1
    channels: List[dict] = field(default_factory=lambda: [
        dict(name="CH1", layers=[2, 3], y0=300.0, amplitude=180.0, wavelength=800.0, phase=0.3, width=140.0),
        dict(name="CH2", layers=[6, 7], y0=650.0, amplitude=120.0, wavelength=650.0, phase=2.0, width=110.0),
    ])
    # --- correlated heterogeneity ---
    corr_len_xy: float = 180.0         # m, horizontal correlation length
    corr_len_z: float = 6.0            # m, vertical correlation length
    spectrum: str = "gaussian"         # "gaussian" | "fractal"  (fractal hook for later experiments)
    fractal_beta: float = 3.0
    ky_kx_ratio: float = 0.85          # mild horizontal anisotropy (depositional)
    # --- faults ---
    faults: List[dict] = field(default_factory=lambda: [
        dict(name="F1", x0=560.0, y0=0.0, z0=2000.0, strike_deg=75.0, dip_deg=80.0,
             curvature=2.0e-4, trans_mult=0.02, thickness=12.0),
    ])
    # --- fracture corridors (equivalent continuum) ---
    fractures: List[dict] = field(default_factory=lambda: [
        dict(name="FR1", xc=300.0, yc=700.0, strike_deg=30.0, length=350.0, width=25.0,
             layers=[1, 2, 3], aperture_mm=0.5, k_mult_along=40.0, k_mult_across=4.0, kz_mult=10.0),
        dict(name="FR2", xc=750.0, yc=250.0, strike_deg=120.0, length=280.0, width=25.0,
             layers=[5, 6, 7], aperture_mm=0.3, k_mult_along=25.0, k_mult_across=3.0, kz_mult=8.0),
        dict(name="FR3", xc=200.0, yc=250.0, strike_deg=60.0, length=200.0, width=25.0,
             layers=[6, 7, 8], aperture_mm=0.4, k_mult_along=30.0, k_mult_across=3.0, kz_mult=8.0),
    ])
    # --- fluids / initial state ---
    owc_depth: float = 2040.0
    transition_zone: float = 4.0       # m, linear Sw ramp above OWC (crude Pc surrogate)
    p_ref: float = 20.4e6              # Pa at depth_ref
    depth_ref: float = 2040.0          # = OWC (so Po = Pw there, Pc ignored)
    rho_w: float = 1020.0
    rho_o: float = 800.0
    # --- wells ---
    wells: List[dict] = field(default_factory=lambda: [
        dict(name="INJ1", type="injector", x=150.0, y=150.0, control="rate",
             value=300.0, units="m3/day", rw=0.1, skin=0.0, perforate="reservoir"),
        dict(name="PROD1", type="producer", x=640.0, y=430.0, control="bhp",
             value=17.0e6, units="Pa", rw=0.1, skin=0.0, perforate="oil_reservoir"),
    ])
    seed: int = 20240601


# ----------------------------------------------------------------------------
# 1. GRID
# ----------------------------------------------------------------------------
def build_grid(cfg: ReservoirConfig) -> dict:
    dx, dy, dz = cfg.Lx / cfg.Nx, cfg.Ly / cfg.Ny, cfg.Lz / cfg.Nz
    x = (np.arange(cfg.Nx) + 0.5) * dx
    y = (np.arange(cfg.Ny) + 0.5) * dy
    zrel = (np.arange(cfg.Nz) + 0.5) * dz          # depth below local top of model
    X, Y, Zrel = np.meshgrid(x, y, zrel, indexing="ij")

    xc, yc = cfg.dome_center
    dome = cfg.dome_amplitude * np.exp(-((X[:, :, 0] - xc) ** 2 + (Y[:, :, 0] - yc) ** 2) / (2 * cfg.dome_radius ** 2))
    top = cfg.top_depth + cfg.dip_x * X[:, :, 0] + cfg.dip_y * Y[:, :, 0] - dome   # (Nx,Ny)
    depth = top[:, :, None] + Zrel                                                # (Nx,Ny,Nz)

    return dict(
        Nx=cfg.Nx, Ny=cfg.Ny, Nz=cfg.Nz, Lx=cfg.Lx, Ly=cfg.Ly, Lz=cfg.Lz,
        dx=dx, dy=dy, dz=dz, x=x, y=y, X=X, Y=Y, depth=depth, top_depth_map=top,
        cell_volume=np.full((cfg.Nx, cfg.Ny, cfg.Nz), dx * dy * dz),
        depth_convention="positive downward; k=0 is the top layer",
    )


# ----------------------------------------------------------------------------
# 2. CORRELATED RANDOM FIELDS
# ----------------------------------------------------------------------------
def gaussian_random_field(shape, corr_len_cells, rng, spectrum="gaussian", beta=3.0) -> np.ndarray:
    """
    Zero-mean, unit-variance spatially correlated field via FFT spectral filtering.
      spectrum='gaussian': Gaussian covariance with per-axis correlation lengths.
      spectrum='fractal' : power-law S(k) ~ (|k|^2 + k0^2)^(-beta/2)  (fBm-like);
                           hook for the later fractal heterogeneity experiments.
    """
    freqs = [np.fft.fftfreq(n) for n in shape]
    K = np.meshgrid(*freqs, indexing="ij")
    if spectrum == "gaussian":
        arg = sum(((2 * np.pi * k * l) ** 2) for k, l in zip(K, corr_len_cells))
        S = np.exp(-0.5 * arg)
    elif spectrum == "fractal":
        k2 = sum(((k * l / max(corr_len_cells)) ** 2) for k, l in zip(K, corr_len_cells))
        k0 = 1.0 / max(corr_len_cells)
        S = (k2 + k0 ** 2) ** (-beta / 2.0)
    else:
        raise ValueError(spectrum)
    white = rng.standard_normal(shape)
    f = np.real(np.fft.ifftn(np.fft.fftn(white) * np.sqrt(S)))
    f = (f - f.mean()) / (f.std() + 1e-15)
    return f


# ----------------------------------------------------------------------------
# 3. FACIES ARCHITECTURE
# ----------------------------------------------------------------------------
def build_facies(grid: dict, cfg: ReservoirConfig, rng: np.random.Generator) -> dict:
    Nx, Ny, Nz = grid["Nx"], grid["Ny"], grid["Nz"]
    assert len(cfg.layer_sequence) == Nz, "layer_sequence must have Nz entries"

    # (a) layer-cake background
    facies = np.broadcast_to(np.array(cfg.layer_sequence)[None, None, :], (Nx, Ny, Nz)).copy()

    # (b) lateral facies variability inside sand/shaly-sand layers (correlated, not white noise)
    lc = (cfg.corr_len_xy / grid["dx"], cfg.corr_len_xy / grid["dy"], cfg.corr_len_z / grid["dz"])
    Zf = gaussian_random_field((Nx, Ny, Nz), lc, rng, cfg.spectrum, cfg.fractal_beta)
    facies[(facies == 2) & (Zf < cfg.lateral_shaliness_threshold)] = 1     # shaly patches in sand
    facies[(facies == 1) & (Zf > cfg.lateral_cleaning_threshold)] = 2      # clean lenses in shaly sand

    # (c) meandering channel bodies (sinusoidal centreline, erode whatever is there)
    X, Y = grid["X"], grid["Y"]
    channel_mask = np.zeros((Nx, Ny, Nz), dtype=np.int16)
    for ci, ch in enumerate(cfg.channels):
        yc = ch["y0"] + ch["amplitude"] * np.sin(2 * np.pi * X[:, :, 0] / ch["wavelength"] + ch["phase"])
        inside_xy = np.abs(Y[:, :, 0] - yc) <= ch["width"] / 2
        for k in ch["layers"]:
            facies[:, :, k][inside_xy] = 3
            channel_mask[:, :, k][inside_xy] = ci + 1

    return dict(facies=facies.astype(np.int8), channel_mask=channel_mask, facies_field_Z=Zf)


# ----------------------------------------------------------------------------
# 4. POROSITY & PERMEABILITY
# ----------------------------------------------------------------------------
def build_porosity(facies, table, grid, cfg, rng) -> np.ndarray:
    lc = (cfg.corr_len_xy / grid["dx"], cfg.corr_len_xy / grid["dy"], cfg.corr_len_z / grid["dz"])
    Z = gaussian_random_field(facies.shape, lc, rng, cfg.spectrum, cfg.fractal_beta)
    phi = np.zeros_like(Z)
    for code, fp in table.items():
        m = facies == code
        phi[m] = np.clip(fp.phi_mean + fp.phi_std * Z[m], fp.phi_min, fp.phi_max)
    return phi


def build_permeability(facies, phi, table, grid, cfg, rng) -> dict:
    lc = (cfg.corr_len_xy / grid["dx"], cfg.corr_len_xy / grid["dy"], cfg.corr_len_z / grid["dz"])
    Zk = gaussian_random_field(facies.shape, lc, rng, cfg.spectrum, cfg.fractal_beta)   # independent of phi field
    Zv = gaussian_random_field(facies.shape, (lc[0] / 2, lc[1] / 2, lc[2]), rng, cfg.spectrum, cfg.fractal_beta)
    kx = np.zeros_like(phi)
    kz = np.zeros_like(phi)
    for code, fp in table.items():
        m = facies == code
        log10k = fp.logk_a + fp.logk_b * phi[m] + fp.logk_sigma * Zk[m]
        kx[m] = np.clip(10.0 ** log10k, fp.k_min, fp.k_max)
        ratio = 10.0 ** (np.log10(fp.kv_kh) + fp.kv_kh_sigma_log10 * Zv[m])
        kz[m] = kx[m] * np.clip(ratio, 1e-3, 1.0)
    ky = kx * cfg.ky_kx_ratio
    return dict(kx=kx, ky=ky, kz=kz)


# ----------------------------------------------------------------------------
# 5. FAULTS  (explicit surfaces -> cell mask + face multipliers)
# ----------------------------------------------------------------------------
def _fault_frame(fd: dict):
    """Unit vectors: s (along strike), n (unit normal), e (down-dip within plane)."""
    st, dp = np.deg2rad(fd["strike_deg"]), np.deg2rad(fd["dip_deg"])
    s = np.array([np.cos(st), np.sin(st), 0.0])
    n = np.array([-np.sin(st) * np.sin(dp), np.cos(st) * np.sin(dp), -np.cos(dp)])
    e = np.cross(n, s)
    p0 = np.array([fd["x0"], fd["y0"], fd["z0"]])
    return s, n, e, p0


def fault_signed_distance(fd: dict, X, Y, D) -> np.ndarray:
    """d = n.(r-p0) + c*t^2 ; t = along-strike coordinate. c>0 -> gently curved fault."""
    s, n, _, p0 = _fault_frame(fd)
    rx, ry, rz = X - p0[0], Y - p0[1], D - p0[2]
    t = s[0] * rx + s[1] * ry
    return n[0] * rx + n[1] * ry + n[2] * rz + fd.get("curvature", 0.0) * t ** 2


def fault_surface_points(fd: dict, grid: dict, nt=40, nu=12):
    """Exact points on the d=0 surface for plotting: r = p0 + t s + u e - c t^2 n."""
    s, n, e, p0 = _fault_frame(fd)
    L = np.hypot(grid["Lx"], grid["Ly"])
    T, U = np.meshgrid(np.linspace(-L, L, nt), np.linspace(-5, grid["Lz"] + 40, nu), indexing="ij")
    c = fd.get("curvature", 0.0)
    P = p0[None, None, :] + T[..., None] * s + U[..., None] * e - (c * T ** 2)[..., None] * n
    inside = (P[..., 0] >= 0) & (P[..., 0] <= grid["Lx"]) & (P[..., 1] >= 0) & (P[..., 1] <= grid["Ly"])
    P[~inside] = np.nan
    return P[..., 0], P[..., 1], P[..., 2]


def build_faults(grid: dict, cfg: ReservoirConfig) -> dict:
    Nx, Ny, Nz = grid["Nx"], grid["Ny"], grid["Nz"]
    X, Y, D = grid["X"], grid["Y"], grid["depth"]
    # face-multiplier arrays are aligned with INTERNAL faces
    mult_x = np.ones((Nx - 1, Ny, Nz)); mult_y = np.ones((Nx, Ny - 1, Nz)); mult_z = np.ones((Nx, Ny, Nz - 1))
    face_x = np.zeros_like(mult_x, dtype=np.int8); face_y = np.zeros_like(mult_y, dtype=np.int8); face_z = np.zeros_like(mult_z, dtype=np.int8)
    cell_mask = np.zeros((Nx, Ny, Nz), dtype=np.int8)
    fault_defs = []
    for fi, fd in enumerate(cfg.faults):
        d = fault_signed_distance(fd, X, Y, D)
        cell_mask[np.abs(d) <= fd.get("thickness", grid["dx"]) / 2] = fi + 1
        cx = d[:-1] * d[1:] < 0; cy = d[:, :-1] * d[:, 1:] < 0; cz = d[:, :, :-1] * d[:, :, 1:] < 0
        for cm, fm, mm in ((cx, face_x, mult_x), (cy, face_y, mult_y), (cz, face_z, mult_z)):
            fm[cm] = fi + 1
            mm[cm] = np.minimum(mm[cm], fd["trans_mult"])
        s, n, e, p0 = _fault_frame(fd)
        fault_defs.append(dict(fd, id=fi + 1, strike_vec=s.tolist(), normal_vec=n.tolist(), point=p0.tolist(),
                               n_cells=int((cell_mask == fi + 1).sum()),
                               n_faces=int((face_x == fi + 1).sum() + (face_y == fi + 1).sum() + (face_z == fi + 1).sum())))
    return dict(cell_mask=cell_mask, face_mask_x=face_x, face_mask_y=face_y, face_mask_z=face_z,
                trans_mult_x=mult_x, trans_mult_y=mult_y, trans_mult_z=mult_z, definitions=fault_defs,
                note="Face arrays index internal faces: x-face m sits between cells i=m and i=m+1. "
                     "Throw/juxtaposition is NOT modelled (numerical simplification).")


# ----------------------------------------------------------------------------
# 6. FRACTURE CORRIDORS (equivalent continuum)
# ----------------------------------------------------------------------------
def build_fractures(grid: dict, cfg: ReservoirConfig, kx, ky, kz) -> dict:
    """
    Equivalent-continuum representation: cells inside a corridor get their MATRIX
    permeability multiplied. This is NOT a discrete fracture model (no separate
    fracture DOF, no explicit aperture flow law). Aperture is stored as metadata only.
    Anisotropic multiplier projected on the grid axes (TPFA cannot carry full tensors).
    """
    X, Y = grid["X"][:, :, 0], grid["Y"][:, :, 0]
    mask = np.zeros(kx.shape, dtype=np.int8)
    kx_f, ky_f, kz_f = kx.copy(), ky.copy(), kz.copy()
    defs = []
    for fi, fr in enumerate(cfg.fractures):
        th = np.deg2rad(fr["strike_deg"])
        s = np.array([np.cos(th), np.sin(th)]); nrm = np.array([-np.sin(th), np.cos(th)])
        rx, ry = X - fr["xc"], Y - fr["yc"]
        t = s[0] * rx + s[1] * ry; u = nrm[0] * rx + nrm[1] * ry
        in_xy = (np.abs(t) <= fr["length"] / 2) & (np.abs(u) <= fr["width"] / 2)
        mx = 1 + (fr["k_mult_along"] - 1) * np.cos(th) ** 2 + (fr["k_mult_across"] - 1) * np.sin(th) ** 2
        my = 1 + (fr["k_mult_along"] - 1) * np.sin(th) ** 2 + (fr["k_mult_across"] - 1) * np.cos(th) ** 2
        for k in fr["layers"]:
            sel = in_xy & (mask[:, :, k] == 0)
            kx_f[:, :, k][sel] *= mx; ky_f[:, :, k][sel] *= my; kz_f[:, :, k][sel] *= fr["kz_mult"]
            mask[:, :, k][sel] = fi + 1
        # corners of the corridor rectangle for plotting / metadata
        corners = [fr["xc"] + a * s[0] * fr["length"] / 2 + b * nrm[0] * fr["width"] / 2 for a, b in ((1, 1), (1, -1), (-1, -1), (-1, 1))]
        cornersy = [fr["yc"] + a * s[1] * fr["length"] / 2 + b * nrm[1] * fr["width"] / 2 for a, b in ((1, 1), (1, -1), (-1, -1), (-1, 1))]
        defs.append(dict(fr, id=fi + 1, corners_x=corners, corners_y=cornersy, n_cells=int((mask == fi + 1).sum()),
                         representation="equivalent_continuum"))
    return dict(mask=mask, kx=kx_f, ky=ky_f, kz=kz_f, definitions=defs)


# ----------------------------------------------------------------------------
# 7 & 8. INITIAL PRESSURE AND SATURATION
# ----------------------------------------------------------------------------
def build_initial_state(grid: dict, facies, table, cfg: ReservoirConfig) -> dict:
    """
    Pressure: depth D positive downward; P increases with D.
        below OWC : P = P_ref + rho_w g (D - D_ref)
        above OWC : P = P_owc + rho_o g (D - D_owc)   (continuous at OWC, Pc = 0 assumed)
    Saturation: Sw = swc(facies) above OWC (+ linear transition zone), 1 in water leg;
                shale is non-net -> Sw = 1. Sg = 0 (undersaturated oil start).
    """
    D = grid["depth"]
    P_owc = cfg.p_ref + cfg.rho_w * G * (cfg.owc_depth - cfg.depth_ref)
    P = np.where(D >= cfg.owc_depth,
                 P_owc + cfg.rho_w * G * (D - cfg.owc_depth),
                 P_owc + cfg.rho_o * G * (D - cfg.owc_depth))

    swc = np.zeros_like(D)
    for code, fp in table.items():
        swc[facies == code] = fp.swc if fp.net else 1.0
    h = cfg.owc_depth - D                                    # height above OWC (>0 in oil leg)
    tz = max(cfg.transition_zone, 1e-9)
    frac = np.clip(h / tz, 0.0, 1.0)                         # 0 at/below OWC, 1 above transition
    Sw = 1.0 - frac * (1.0 - swc)
    Sw = np.clip(Sw, 0.0, 1.0)
    So = 1.0 - Sw
    Sg = np.zeros_like(Sw)
    return dict(pressure=P, Sw=Sw, So=So, Sg=Sg, swc=swc, P_owc=P_owc,
                pressure_units="Pa", convention="depth positive downward, P increases with depth")


# ----------------------------------------------------------------------------
# 9. WELLS
# ----------------------------------------------------------------------------
def build_wells(grid: dict, facies, So, table, cfg: ReservoirConfig) -> List[dict]:
    wells = []
    for w in cfg.wells:
        i = int(np.clip(w["x"] // grid["dx"], 0, grid["Nx"] - 1))
        j = int(np.clip(w["y"] // grid["dy"], 0, grid["Ny"] - 1))
        col_f = facies[i, j, :]; col_so = So[i, j, :]
        net = np.array([table[int(c)].net for c in col_f])
        if w["perforate"] == "reservoir":
            perf = np.where(net)[0]
        elif w["perforate"] == "oil_reservoir":
            perf = np.where(net & (col_so > 0.05))[0]
        elif w["perforate"] == "all":
            perf = np.arange(grid["Nz"])
        else:                                # explicit list of layers
            perf = np.array(w["perforate"], dtype=int)
        wells.append(dict(
            name=w["name"], type=w["type"], x=w["x"], y=w["y"], i=i, j=j,
            cell_x=float(grid["x"][i]), cell_y=float(grid["y"][j]),
            perforated_layers=[int(k) for k in perf],
            perforation_depths=[float(grid["depth"][i, j, k]) for k in perf],
            perforated_facies=[FACIES_NAMES[int(col_f[k])] for k in perf],
            control=w["control"], value=w["value"], units=w["units"],
            rw=w["rw"], skin=w["skin"], geometry="vertical", perforation_rule=w["perforate"],
        ))
    return wells


# ----------------------------------------------------------------------------
# 10. ASSEMBLY
# ----------------------------------------------------------------------------
def build_reservoir(cfg: Optional[ReservoirConfig] = None) -> dict:
    cfg = cfg or ReservoirConfig()
    rng = np.random.default_rng(cfg.seed)
    table = default_facies_table()

    grid = build_grid(cfg)
    fac = build_facies(grid, cfg, rng)
    phi = build_porosity(fac["facies"], table, grid, cfg, rng)
    perm = build_permeability(fac["facies"], phi, table, grid, cfg, rng)
    faults = build_faults(grid, cfg)
    frac = build_fractures(grid, cfg, perm["kx"], perm["ky"], perm["kz"])
    state = build_initial_state(grid, fac["facies"], table, cfg)
    wells = build_wells(grid, fac["facies"], state["So"], table, cfg)

    return {
        "config": cfg,
        "facies_table": table,
        "grid": grid,
        "facies": fac["facies"],
        "channel_mask": fac["channel_mask"],
        "phi": phi,
        "kx_matrix": perm["kx"], "ky_matrix": perm["ky"], "kz_matrix": perm["kz"],   # before fracture enhancement
        "kx": frac["kx"], "ky": frac["ky"], "kz": frac["kz"],                          # effective (matrix + corridors)
        "faults": faults,
        "fractures": dict(mask=frac["mask"], definitions=frac["definitions"]),
        "initial_state": dict(pressure=state["pressure"], Sw=state["Sw"], So=state["So"], Sg=state["Sg"]),
        "swc": state["swc"],
        "wells": wells,
        "units": dict(length="m", pressure="Pa", permeability="mD", density="kg/m3", md_to_m2=MD_TO_M2),
    }


# ----------------------------------------------------------------------------
# 11. VALIDATION
# ----------------------------------------------------------------------------
def validate_reservoir(res: dict, raise_on_fail=True) -> List[str]:
    cfg, table, g = res["config"], res["facies_table"], res["grid"]
    fac, phi = res["facies"], res["phi"]
    st = res["initial_state"]
    problems = []

    def chk(cond, msg):
        if not cond:
            problems.append(msg)

    for code, fp in table.items():
        m = fac == code
        if m.any():
            chk(phi[m].min() >= fp.phi_min - 1e-12 and phi[m].max() <= fp.phi_max + 1e-12, f"phi out of range in {fp.name}")
            km = res["kx_matrix"][m]
            chk(km.min() >= fp.k_min - 1e-12 and km.max() <= fp.k_max + 1e-12, f"kx_matrix out of range in {fp.name}")
    for k in ("kx", "ky", "kz", "kx_matrix", "ky_matrix", "kz_matrix"):
        chk(np.all(np.isfinite(res[k])) and np.all(res[k] > 0), f"{k} must be finite and > 0")
    chk(np.all(res["kz"] <= res["kx"] * 1.0001 + 1e-12) or True, "")  # kz<=kx expected in matrix; corridors may break it
    chk(np.all(res["kz_matrix"] <= res["kx_matrix"] + 1e-12), "kz_matrix > kx_matrix somewhere")

    S = st["Sw"] + st["So"] + st["Sg"]
    chk(np.allclose(S, 1.0, atol=1e-10), "Sw+So+Sg != 1")
    for k in ("Sw", "So", "Sg"):
        chk(np.all((st[k] >= 0) & (st[k] <= 1)), f"{k} outside [0,1]")
    D = g["depth"]
    chk(np.all(st["Sw"][D >= cfg.owc_depth] == 1.0), "oil found below OWC")
    net = np.isin(fac, [c for c, fp in table.items() if fp.net])
    chk(np.all(st["Sw"][~net] == 1.0), "non-net (shale) cells must have Sw = 1")

    # pressure monotonic with depth in every column
    dP = np.diff(st["pressure"], axis=2)
    chk(np.all(dP > 0), "pressure does not increase downward in every column")
    chk(np.all(np.diff(D, axis=2) > 0), "depth not increasing with k")
    # gradient check against expected water gradient in the water leg
    wl = (D[:, :, :-1] >= cfg.owc_depth)
    if wl.any():
        grad = dP[wl] / np.diff(D, axis=2)[wl]
        chk(np.allclose(grad, cfg.rho_w * G, rtol=1e-6), "water-leg gradient != rho_w g")

    for w in res["wells"]:
        chk(0 <= w["i"] < g["Nx"] and 0 <= w["j"] < g["Ny"], f"{w['name']} outside grid")
        chk(0 <= w["x"] <= g["Lx"] and 0 <= w["y"] <= g["Ly"], f"{w['name']} physical location outside reservoir")
        chk(len(w["perforated_layers"]) > 0, f"{w['name']} has no perforations")
        for k, fn in zip(w["perforated_layers"], w["perforated_facies"]):
            chk(0 <= k < g["Nz"], f"{w['name']} perforation layer {k} out of range")
            if w["perforation_rule"] in ("reservoir", "oil_reservoir"):
                chk(table[FACIES_CODES[fn]].net, f"{w['name']} perforates non-net facies at k={k}")
            if w["perforation_rule"] == "oil_reservoir":
                chk(st["So"][w["i"], w["j"], k] > 0.05, f"{w['name']} perforates water-bearing layer k={k}")

    fl = res["faults"]
    for fd in fl["definitions"]:
        chk(fd["n_faces"] > 0, f"fault {fd['name']} crosses no cell faces (invalid geometry)")
        chk(0 < fd["trans_mult"], f"fault {fd['name']} multiplier must be > 0")
    for arr in (fl["trans_mult_x"], fl["trans_mult_y"], fl["trans_mult_z"]):
        chk(np.all((arr > 0) & (arr <= 1.0)), "fault multipliers must be in (0,1] (increase later if desired)")
    chk(fl["trans_mult_x"].shape == (g["Nx"] - 1, g["Ny"], g["Nz"]), "trans_mult_x wrong shape")
    for fr in res["fractures"]["definitions"]:
        chk(fr["n_cells"] > 0, f"fracture {fr['name']} contains no cells")
        chk(all(0 <= k < g["Nz"] for k in fr["layers"]), f"fracture {fr['name']} layer out of range")
    chk(np.all(res["fractures"]["mask"][res["kx"] > res["kx_matrix"]] > 0), "perm enhanced outside fracture mask")

    if problems and raise_on_fail:
        raise AssertionError("Validation failed:\n  - " + "\n  - ".join(problems))
    return problems


# ----------------------------------------------------------------------------
# 12. OUTPUT
# ----------------------------------------------------------------------------
def _jsonable(o):
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, dict):
        return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_jsonable(v) for v in o]
    return o


def save_reservoir(res: dict, outdir: str, name="synthetic_reservoir") -> None:
    os.makedirs(outdir, exist_ok=True)
    g, fl = res["grid"], res["faults"]
    np.savez_compressed(
        os.path.join(outdir, f"{name}.npz"),
        x=g["x"], y=g["y"], depth=g["depth"], top_depth_map=g["top_depth_map"], cell_volume=g["cell_volume"],
        facies=res["facies"], channel_mask=res["channel_mask"], phi=res["phi"],
        kx=res["kx"], ky=res["ky"], kz=res["kz"],
        kx_matrix=res["kx_matrix"], ky_matrix=res["ky_matrix"], kz_matrix=res["kz_matrix"],
        fault_cell_mask=fl["cell_mask"], fault_face_mask_x=fl["face_mask_x"], fault_face_mask_y=fl["face_mask_y"],
        fault_face_mask_z=fl["face_mask_z"], fault_trans_mult_x=fl["trans_mult_x"],
        fault_trans_mult_y=fl["trans_mult_y"], fault_trans_mult_z=fl["trans_mult_z"],
        fracture_mask=res["fractures"]["mask"], swc=res["swc"],
        pressure=res["initial_state"]["pressure"], Sw=res["initial_state"]["Sw"],
        So=res["initial_state"]["So"], Sg=res["initial_state"]["Sg"],
    )
    cfg = res["config"]
    meta = dict(
        disclaimer="SYNTHETIC model. All ranges are illustrative; not calibrated to any real field.",
        grid=dict(Nx=g["Nx"], Ny=g["Ny"], Nz=g["Nz"], Lx=g["Lx"], Ly=g["Ly"], Lz=g["Lz"],
                  dx=g["dx"], dy=g["dy"], dz=g["dz"], depth_convention=g["depth_convention"],
                  array_shape="(Nx,Ny,Nz) C-order, indexing='ij'"),
        structure=dict(top_depth=cfg.top_depth, dip_x=cfg.dip_x, dip_y=cfg.dip_y,
                       dome_center=cfg.dome_center, dome_amplitude=cfg.dome_amplitude, dome_radius=cfg.dome_radius),
        seed=cfg.seed,
        facies_codes=FACIES_CODES,
        facies_definitions={fp.name: asdict(fp) for fp in res["facies_table"].values()},
        layer_sequence=[FACIES_NAMES[c] for c in cfg.layer_sequence],
        channels=cfg.channels,
        heterogeneity=dict(corr_len_xy=cfg.corr_len_xy, corr_len_z=cfg.corr_len_z, spectrum=cfg.spectrum,
                           fractal_beta=cfg.fractal_beta, ky_kx_ratio=cfg.ky_kx_ratio),
        property_ranges=dict(phi=[float(res["phi"].min()), float(res["phi"].max())],
                             kx=[float(res["kx"].min()), float(res["kx"].max())],
                             kz=[float(res["kz"].min()), float(res["kz"].max())]),
        faults=fl["definitions"], fault_note=fl["note"],
        fractures=res["fractures"]["definitions"],
        fracture_note="Equivalent-continuum corridors (matrix perm multipliers). Not a DFM.",
        fluids=dict(owc_depth=cfg.owc_depth, transition_zone=cfg.transition_zone, p_ref=cfg.p_ref,
                    depth_ref=cfg.depth_ref, rho_w=cfg.rho_w, rho_o=cfg.rho_o,
                    pressure_convention="P = P_ref + rho g (D - D_ref); D positive downward"),
        wells=res["wells"],
        units=res["units"],
    )
    with open(os.path.join(outdir, f"{name}.json"), "w") as f:
        json.dump(_jsonable(meta), f, indent=2)


def load_reservoir(outdir: str, name="synthetic_reservoir"):
    arrays = dict(np.load(os.path.join(outdir, f"{name}.npz")))
    with open(os.path.join(outdir, f"{name}.json")) as f:
        meta = json.load(f)
    return arrays, meta


# ----------------------------------------------------------------------------
# 13. VISUALISATION
# ----------------------------------------------------------------------------
def _mpl():
    import matplotlib
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
    return matplotlib, plt


FACIES_COLORS = {0: "#6b6b6b", 1: "#c9b37e", 2: "#f2d16b", 3: "#e07b39"}


def _draw_wells(ax, res, zkey="depth"):
    g = res["grid"]
    for w in res["wells"]:
        col = "blue" if w["type"] == "injector" else "red"
        ztop, zbot = g["depth"][w["i"], w["j"], 0] - g["dz"], g["depth"][w["i"], w["j"], -1] + g["dz"]
        ax.plot([w["cell_x"]] * 2, [w["cell_y"]] * 2, [ztop, zbot], color=col, lw=2.5)
        ax.scatter([w["cell_x"]] * len(w["perforation_depths"]), [w["cell_y"]] * len(w["perforation_depths"]),
                   w["perforation_depths"], color=col, s=35, marker="s", edgecolor="k", zorder=10)
        ax.text(w["cell_x"], w["cell_y"], ztop - 3, w["name"], color=col, fontsize=9, weight="bold")


def _style3d(ax, res, title):
    g = res["grid"]
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_zlabel("depth [m]")
    ax.set_xlim(0, g["Lx"]); ax.set_ylim(0, g["Ly"])
    ax.set_zlim(np.nanmax(g["depth"]) + 5, np.nanmin(g["depth"]) - 5)   # depth increases downward
    ax.set_box_aspect((1, 1, 0.35)); ax.set_title(title)


def plot_facies_3d(res, fname=None):
    _, plt = _mpl()
    g, fac = res["grid"], res["facies"]
    fig = plt.figure(figsize=(10, 7)); ax = fig.add_subplot(111, projection="3d")
    X, Y, D = g["X"], g["Y"], g["depth"]
    # plot outer faces + every layer as point cloud, skipping shale to keep channels visible
    for code, colr in FACIES_COLORS.items():
        m = (fac == code) & (fac != 0)
        if m.any():
            ax.scatter(X[m], Y[m], D[m], c=colr, s=6, alpha=0.5, label=FACIES_NAMES[code], depthshade=False)
    _draw_wells(ax, res); _style3d(ax, res, "3D facies (shale hidden for clarity)")
    ax.legend(loc="upper left", markerscale=3)
    if fname: fig.savefig(fname, dpi=140, bbox_inches="tight")
    return fig


def plot_property_slice(res, prop="phi", k=2, j=None, log=False, fname=None):
    _, plt = _mpl()
    g = res["grid"]; arr = res[prop]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    data = np.log10(arr[:, :, k]) if log else arr[:, :, k]
    im = axes[0].pcolormesh(g["x"], g["y"], data.T, shading="auto", cmap="viridis")
    fig.colorbar(im, ax=axes[0], label=("log10 " if log else "") + prop)
    axes[0].contour(g["x"], g["y"], res["faults"]["cell_mask"][:, :, k].T, levels=[0.5], colors="k", linewidths=2)
    axes[0].contour(g["x"], g["y"], res["fractures"]["mask"][:, :, k].T, levels=[0.5], colors="magenta", linewidths=1.5)
    for w in res["wells"]:
        axes[0].plot(w["cell_x"], w["cell_y"], "o", ms=10, mfc="blue" if w["type"] == "injector" else "red", mec="k")
        axes[0].annotate(w["name"], (w["cell_x"], w["cell_y"]), xytext=(6, 6), textcoords="offset points")
    axes[0].set_title(f"{prop} map, layer k={k} (black=fault cells, magenta=fracture corridors)")
    axes[0].set_xlabel("x [m]"); axes[0].set_ylabel("y [m]"); axes[0].set_aspect("equal")
    j = res["wells"][1]["j"] if j is None else j
    data = np.log10(arr[:, j, :]) if log else arr[:, j, :]
    Xs, Ds = g["X"][:, j, :], g["depth"][:, j, :]
    im = axes[1].pcolormesh(Xs, Ds, data, shading="auto", cmap="viridis")
    fig.colorbar(im, ax=axes[1], label=("log10 " if log else "") + prop)
    axes[1].contour(Xs, Ds, res["faults"]["cell_mask"][:, j, :], levels=[0.5], colors="k", linewidths=2)
    axes[1].axhline(res["config"].owc_depth, color="cyan", ls="--", label="OWC")
    axes[1].invert_yaxis(); axes[1].set_title(f"{prop} x-z section, j={j}"); axes[1].set_xlabel("x [m]"); axes[1].set_ylabel("depth [m]")
    axes[1].legend()
    if fname: fig.savefig(fname, dpi=140, bbox_inches="tight")
    return fig


def plot_faults_3d(res, fname=None):
    _, plt = _mpl()
    fig = plt.figure(figsize=(10, 7)); ax = fig.add_subplot(111, projection="3d")
    g = res["grid"]
    cm = res["faults"]["cell_mask"]
    for fd in res["faults"]["definitions"]:
        Xs, Ys, Zs = fault_surface_points(fd, g)
        ax.plot_surface(Xs, Ys, Zs, color="k", alpha=0.35, linewidth=0)
        m = cm == fd["id"]
        ax.scatter(g["X"][m], g["Y"][m], g["depth"][m], c="crimson", s=8, label=f"{fd['name']} cells (mult={fd['trans_mult']})")
    # translucent reservoir top surface for context
    ax.plot_surface(g["X"][:, :, 0], g["Y"][:, :, 0], g["top_depth_map"], cmap="terrain", alpha=0.35)
    _draw_wells(ax, res); _style3d(ax, res, "Fault surface(s) and flagged cells (top surface shown for structure)")
    ax.legend(loc="upper left")
    if fname: fig.savefig(fname, dpi=140, bbox_inches="tight")
    return fig


def plot_fractures_3d(res, fname=None):
    _, plt = _mpl()
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection
    fig = plt.figure(figsize=(10, 7)); ax = fig.add_subplot(111, projection="3d")
    g = res["grid"]; mk = res["fractures"]["mask"]
    cmap = plt.get_cmap("cool")
    for fr in res["fractures"]["definitions"]:
        m = mk == fr["id"]
        ztop, zbot = g["depth"][m].min() - g["dz"] / 2, g["depth"][m].max() + g["dz"] / 2
        th = np.deg2rad(fr["strike_deg"]); L = fr["length"] / 2
        p1 = (fr["xc"] - L * np.cos(th), fr["yc"] - L * np.sin(th)); p2 = (fr["xc"] + L * np.cos(th), fr["yc"] + L * np.sin(th))
        verts = [[(p1[0], p1[1], ztop), (p2[0], p2[1], ztop), (p2[0], p2[1], zbot), (p1[0], p1[1], zbot)]]
        c = cmap(fr["id"] / max(1, len(res["fractures"]["definitions"])))
        ax.add_collection3d(Poly3DCollection(verts, facecolor=c, alpha=0.5, edgecolor="k"))
        ax.scatter(g["X"][m], g["Y"][m], g["depth"][m], color=c, s=10, label=f"{fr['name']} (x{fr['k_mult_along']})")
    _draw_wells(ax, res); _style3d(ax, res, "Fracture corridors (equivalent continuum cells + nominal planes)")
    ax.legend(loc="upper left")
    if fname: fig.savefig(fname, dpi=140, bbox_inches="tight")
    return fig


def plot_wells_3d(res, fname=None):
    _, plt = _mpl()
    fig = plt.figure(figsize=(10, 7)); ax = fig.add_subplot(111, projection="3d")
    g = res["grid"]
    net = res["facies"] > 0
    oil = res["initial_state"]["So"] > 0.05
    ax.scatter(g["X"][net & oil], g["Y"][net & oil], g["depth"][net & oil], c="palegreen", s=4, alpha=0.25, label="oil-bearing net cells")
    ax.plot_surface(g["X"][:, :, 0], g["Y"][:, :, 0], np.full_like(g["top_depth_map"], res["config"].owc_depth), color="cyan", alpha=0.2)
    _draw_wells(ax, res); _style3d(ax, res, "Wells (blue=injector, red=producer, squares=perforations), cyan plane=OWC")
    ax.legend(loc="upper left")
    if fname: fig.savefig(fname, dpi=140, bbox_inches="tight")
    return fig


def plot_pressure_depth(res, fname=None):
    _, plt = _mpl()
    g, P = res["grid"], res["initial_state"]["pressure"]
    fig, ax = plt.subplots(figsize=(6, 7))
    ax.scatter(P.ravel() / 1e6, g["depth"].ravel(), s=3, c=res["initial_state"]["Sw"].ravel(), cmap="coolwarm_r")
    ax.axhline(res["config"].owc_depth, color="k", ls="--", label="OWC")
    ax.invert_yaxis(); ax.set_xlabel("pressure [MPa]"); ax.set_ylabel("depth [m]")
    ax.set_title("Initial hydrostatic pressure (colour: Sw)\noil gradient above OWC, water gradient below")
    ax.legend(); ax.grid(alpha=0.3)
    if fname: fig.savefig(fname, dpi=140, bbox_inches="tight")
    return fig


def plot_saturation(res, j=None, fname=None):
    _, plt = _mpl()
    g = res["grid"]; Sw = res["initial_state"]["Sw"]
    j = res["wells"][1]["j"] if j is None else j
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    Xs, Ds = g["X"][:, j, :], g["depth"][:, j, :]
    im = axes[0].pcolormesh(Xs, Ds, Sw[:, j, :], shading="auto", cmap="coolwarm_r", vmin=0, vmax=1)
    fig.colorbar(im, ax=axes[0], label="Sw"); axes[0].axhline(res["config"].owc_depth, color="k", ls="--", label="OWC")
    axes[0].invert_yaxis(); axes[0].set_title(f"Initial Sw, x-z section j={j}"); axes[0].legend()
    axes[0].set_xlabel("x [m]"); axes[0].set_ylabel("depth [m]")
    So_col = res["initial_state"]["So"].sum(axis=2) * g["dz"]
    im = axes[1].pcolormesh(g["x"], g["y"], So_col.T, shading="auto", cmap="YlOrRd")
    fig.colorbar(im, ax=axes[1], label="sum(So*dz) [m]")
    axes[1].contour(g["x"], g["y"], res["faults"]["cell_mask"].max(axis=2).T, levels=[0.5], colors="k")
    for w in res["wells"]:
        axes[1].plot(w["cell_x"], w["cell_y"], "o", ms=10, mfc="blue" if w["type"] == "injector" else "red", mec="k")
    axes[1].set_title("Oil column thickness (structure + OWC control)"); axes[1].set_aspect("equal")
    if fname: fig.savefig(fname, dpi=140, bbox_inches="tight")
    return fig




def make_all_figures(res, outdir):
    os.makedirs(outdir, exist_ok=True)
    _, plt = _mpl()
    plot_facies_3d(res, os.path.join(outdir, "01_facies_3d.png"))
    plot_property_slice(res, "phi", k=2, fname=os.path.join(outdir, "02_porosity_slice.png"))
    plot_property_slice(res, "kx", k=2, log=True, fname=os.path.join(outdir, "03_permeability_slice.png"))
    plot_faults_3d(res, os.path.join(outdir, "04_faults_3d.png"))
    plot_fractures_3d(res, os.path.join(outdir, "05_fractures_3d.png"))
    plot_wells_3d(res, os.path.join(outdir, "06_wells_3d.png"))
    plot_pressure_depth(res, os.path.join(outdir, "07_pressure_depth.png"))
    plot_saturation(res, fname=os.path.join(outdir, "08_saturation_owc.png"))
    plt.close("all")


# ----------------------------------------------------------------------------
# 14. SUMMARY / MAIN
# ----------------------------------------------------------------------------
def summarize(res):
    g, fac = res["grid"], res["facies"]
    print(f"Grid {g['Nx']}x{g['Ny']}x{g['Nz']}  cell {g['dx']:.1f}x{g['dy']:.1f}x{g['dz']:.1f} m  "
          f"depth {g['depth'].min():.1f}-{g['depth'].max():.1f} m")
    for code, name in FACIES_NAMES.items():
        m = fac == code
        if m.any():
            print(f"  {name:18s} {100*m.mean():5.1f}%  phi {res['phi'][m].mean():.3f}  "
                  f"kx(geomean) {10**np.log10(res['kx'][m]).mean():8.2f} mD  kz/kx {np.median(res['kz'][m]/res['kx'][m]):.3f}")
    print(f"Faults: {[ (f['name'], f['n_cells'], f['n_faces']) for f in res['faults']['definitions']]}")
    print(f"Fracture corridors: {[ (f['name'], f['n_cells']) for f in res['fractures']['definitions']]}")
    P = res["initial_state"]["pressure"]
    print(f"Pressure {P.min()/1e6:.2f}-{P.max()/1e6:.2f} MPa; OWC {res['config'].owc_depth} m; "
          f"oil-in-place (pore volume) {(res['phi']*res['initial_state']['So']*g['cell_volume']).sum():.3e} m3")
    for w in res["wells"]:
        print(f"  {w['name']:6s} {w['type']:9s} cell ({w['i']},{w['j']}) perfs k={w['perforated_layers']} "
              f"{w['perforated_facies']} control={w['control']}:{w['value']} {w['units']}")


if __name__ == "__main__":
    res = build_reservoir(ReservoirConfig())
    validate_reservoir(res)
    summarize(res)
    save_reservoir(res, "output")
    make_all_figures(res, "output/figures")
    print("Saved to ./output")

Grid 40x40x10  cell 25.0x25.0x5.0 m  depth 1991.6-2053.4 m
  shale               30.0%  phi 0.051  kx(geomean)     0.01 mD  kz/kx 0.053
  shaly_sandstone     32.5%  phi 0.130  kx(geomean)     3.19 mD  kz/kx 0.090
  sandstone           32.4%  phi 0.201  kx(geomean)   104.84 mD  kz/kx 0.208
  channel_sandstone    5.1%  phi 0.245  kx(geomean)   877.64 mD  kz/kx 0.329
Faults: [('F1', 198, 543)]
Fracture corridors: [('FR1', 42), ('FR2', 36), ('FR3', 24)]
Pressure 20.02-20.53 MPa; OWC 2040.0 m; oil-in-place (pore volume) 4.031e+06 m3
  INJ1   injector  cell (6,6) perfs k=[1, 2, 3, 5, 6, 7, 8] ['shaly_sandstone', 'shaly_sandstone', 'shaly_sandstone', 'shaly_sandstone', 'sandstone', 'sandstone', 'sandstone'] control=rate:300.0 m3/day
  PROD1  producer  cell (25,17) perfs k=[1, 2, 3, 5, 6, 7, 8] ['shaly_sandstone', 'sandstone', 'sandstone', 'shaly_sandstone', 'shaly_sandstone', 'sandstone', 'shaly_sandstone'] control=bhp:17000000.0 Pa


C:\Users\dhruv\AppData\Local\Temp\ipykernel_24200\2870993126.py:640: UserWarning: The input coordinates to pcolormesh are interpreted as cell centers, but are not monotonically increasing or decreasing. This may lead to incorrectly calculated cell edges, in which case, please supply explicit cell edges to pcolormesh.
  im = axes[1].pcolormesh(Xs, Ds, data, shading="auto", cmap="viridis")
C:\Users\dhruv\AppData\Local\Temp\ipykernel_24200\2870993126.py:722: UserWarning: The input coordinates to pcolormesh are interpreted as cell centers, but are not monotonically increasing or decreasing. This may lead to incorrectly calculated cell edges, in which case, please supply explicit cell edges to pcolormesh.
  im = axes[0].pcolormesh(Xs, Ds, Sw[:, j, :], shading="auto", cmap="coolwarm_r", vmin=0, vmax=1)


Saved to ./output


In [4]:
"""
synthetic_reservoir.py
======================
Complete synthetic 3D petroleum reservoir model generator with interactive visualization.

Generates:
- 3D Cartesian grid with structural dip and dome
- Layered stratigraphy with meandering channel bodies (4 facies)
- Spatially correlated porosity and anisotropic permeability
- Explicit fault surfaces with face transmissibility multipliers
- Fracture corridors (equivalent continuum)
- Hydrostatic initial pressure and OWC-based saturations
- Vertical wells with facies-aware perforations
- Validation, NPZ+JSON output, and interactive Plotly 3D visualization

ALL NUMERICAL RANGES ARE ILLUSTRATIVE/SYNTHETIC. Not calibrated to any real field.

Units: SI (m, Pa, kg/m3, mD where 1 mD = 9.869233e-16 m2)
Depth convention: positive downward, k=0 is top layer.
"""

from __future__ import annotations
import json
import os
import numpy as np
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional, Tuple

# Constants
MD_TO_M2 = 9.869233e-16
G = 9.80665

# Facies codes
FACIES_CODES = {"shale": 0, "shaly_sandstone": 1, "sandstone": 2, "channel_sandstone": 3}
FACIES_NAMES = {v: k for k, v in FACIES_CODES.items()}
FACIES_COLORS = ["#6b6b6b", "#c9b37e", "#f2d16b", "#e07b39"]  # gray, tan, yellow, orange


@dataclass
class FaciesParams:
    """Petrophysical distribution per facies. ILLUSTRATIVE values only."""
    name: str
    code: int
    phi_mean: float
    phi_std: float
    phi_min: float
    phi_max: float
    logk_a: float          # log10(k[mD]) = a + b*phi + sigma*Z
    logk_b: float
    logk_sigma: float
    k_min: float
    k_max: float
    kv_kh: float
    kv_kh_sigma_log10: float
    swc: float
    net: bool


def default_facies_table() -> Dict[int, FaciesParams]:
    return {
        0: FaciesParams("shale", 0, 0.05, 0.015, 0.01, 0.10, -3.0, 20.0, 0.40, 1e-4, 0.5, 0.05, 0.30, 1.00, False),
        1: FaciesParams("shaly_sandstone", 1, 0.13, 0.030, 0.06, 0.20, -1.5, 15.0, 0.50, 0.1, 50.0, 0.10, 0.30, 0.35, True),
        2: FaciesParams("sandstone", 2, 0.20, 0.035, 0.12, 0.28, -0.5, 12.0, 0.40, 5.0, 1000.0, 0.20, 0.25, 0.22, True),
        3: FaciesParams("channel_sandstone", 3, 0.26, 0.030, 0.18, 0.34, 0.3, 11.0, 0.35, 100.0, 8000.0, 0.35, 0.20, 0.15, True),
    }


@dataclass
class ReservoirConfig:
    # Geometry
    Nx: int = 40
    Ny: int = 40
    Nz: int = 10
    Lx: float = 1000.0
    Ly: float = 1000.0
    Lz: float = 50.0
    top_depth: float = 2000.0
    dip_x: float = 0.005
    dip_y: float = 0.003
    dome_center: Tuple[float, float] = (600.0, 400.0)
    dome_amplitude: float = 15.0
    dome_radius: float = 350.0
    
    # Stratigraphy (top to bottom)
    layer_sequence: List[int] = field(default_factory=lambda: [0, 1, 2, 2, 0, 1, 2, 2, 1, 0])
    lateral_shaliness_threshold: float = -0.9
    lateral_cleaning_threshold: float = 1.1
    channels: List[dict] = field(default_factory=lambda: [
        dict(name="CH1", layers=[2, 3], y0=300.0, amplitude=180.0, wavelength=800.0, phase=0.3, width=140.0),
        dict(name="CH2", layers=[6, 7], y0=650.0, amplitude=120.0, wavelength=650.0, phase=2.0, width=110.0),
    ])
    
    # Heterogeneity
    corr_len_xy: float = 180.0
    corr_len_z: float = 6.0
    spectrum: str = "gaussian"
    fractal_beta: float = 3.0
    ky_kx_ratio: float = 0.85
    
    # Faults
    faults: List[dict] = field(default_factory=lambda: [
        dict(name="F1", x0=560.0, y0=0.0, z0=2000.0, strike_deg=75.0, dip_deg=80.0,
             curvature=2.0e-4, trans_mult=0.02, thickness=12.0),
    ])
    
    # Fracture corridors (equivalent continuum)
    fractures: List[dict] = field(default_factory=lambda: [
        dict(name="FR1", xc=300.0, yc=700.0, strike_deg=30.0, length=350.0, width=25.0,
             layers=[1, 2, 3], aperture_mm=0.5, k_mult_along=40.0, k_mult_across=4.0, kz_mult=10.0),
        dict(name="FR2", xc=750.0, yc=250.0, strike_deg=120.0, length=280.0, width=25.0,
             layers=[5, 6, 7], aperture_mm=0.3, k_mult_along=25.0, k_mult_across=3.0, kz_mult=8.0),
        dict(name="FR3", xc=200.0, yc=250.0, strike_deg=60.0, length=200.0, width=25.0,
             layers=[6, 7, 8], aperture_mm=0.4, k_mult_along=30.0, k_mult_across=3.0, kz_mult=8.0),
    ])
    
    # Fluids / Initial state
    owc_depth: float = 2040.0
    transition_zone: float = 4.0
    p_ref: float = 20.4e6
    depth_ref: float = 2040.0
    rho_w: float = 1020.0
    rho_o: float = 800.0
    
    # Wells
    wells: List[dict] = field(default_factory=lambda: [
        dict(name="INJ1", type="injector", x=150.0, y=150.0, control="rate",
             value=300.0, units="m3/day", rw=0.1, skin=0.0, perforate="reservoir"),
        dict(name="PROD1", type="producer", x=640.0, y=430.0, control="bhp",
             value=17.0e6, units="Pa", rw=0.1, skin=0.0, perforate="oil_reservoir"),
    ])
    
    seed: int = 20240601


# ============================================================================
# 1. GRID
# ============================================================================
def build_grid(cfg: ReservoirConfig) -> dict:
    dx, dy, dz = cfg.Lx / cfg.Nx, cfg.Ly / cfg.Ny, cfg.Lz / cfg.Nz
    x = (np.arange(cfg.Nx) + 0.5) * dx
    y = (np.arange(cfg.Ny) + 0.5) * dy
    zrel = (np.arange(cfg.Nz) + 0.5) * dz
    X, Y, Zrel = np.meshgrid(x, y, zrel, indexing="ij")
    
    xc, yc = cfg.dome_center
    dome = cfg.dome_amplitude * np.exp(-((X[:, :, 0] - xc) ** 2 + (Y[:, :, 0] - yc) ** 2) / (2 * cfg.dome_radius ** 2))
    top = cfg.top_depth + cfg.dip_x * X[:, :, 0] + cfg.dip_y * Y[:, :, 0] - dome
    depth = top[:, :, None] + Zrel
    
    return dict(
        Nx=cfg.Nx, Ny=cfg.Ny, Nz=cfg.Nz, Lx=cfg.Lx, Ly=cfg.Ly, Lz=cfg.Lz,
        dx=dx, dy=dy, dz=dz, x=x, y=y, X=X, Y=Y, depth=depth, top_depth_map=top,
        cell_volume=np.full((cfg.Nx, cfg.Ny, cfg.Nz), dx * dy * dz),
        depth_convention="positive downward; k=0 is the top layer",
    )


# ============================================================================
# 2. CORRELATED RANDOM FIELDS
# ============================================================================
def gaussian_random_field(shape: Tuple[int, ...], corr_len_cells: Tuple[float, ...], 
                          rng: np.random.Generator, spectrum: str = "gaussian", 
                          beta: float = 3.0) -> np.ndarray:
    """Zero-mean, unit-variance spatially correlated field via FFT spectral filtering."""
    freqs = [np.fft.fftfreq(n) for n in shape]
    K = np.meshgrid(*freqs, indexing="ij")
    
    if spectrum == "gaussian":
        arg = sum(((2 * np.pi * k * l) ** 2) for k, l in zip(K, corr_len_cells))
        S = np.exp(-0.5 * arg)
    elif spectrum == "fractal":
        k2 = sum(((k * l / max(corr_len_cells)) ** 2) for k, l in zip(K, corr_len_cells))
        k0 = 1.0 / max(corr_len_cells)
        S = (k2 + k0 ** 2) ** (-beta / 2.0)
    else:
        raise ValueError(f"Unknown spectrum: {spectrum}")
    
    white = rng.standard_normal(shape)
    f = np.real(np.fft.ifftn(np.fft.fftn(white) * np.sqrt(S)))
    f = (f - f.mean()) / (f.std() + 1e-15)
    return f


# ============================================================================
# 3. FACIES ARCHITECTURE
# ============================================================================
def build_facies(grid: dict, cfg: ReservoirConfig, rng: np.random.Generator) -> dict:
    Nx, Ny, Nz = grid["Nx"], grid["Ny"], grid["Nz"]
    assert len(cfg.layer_sequence) == Nz, "layer_sequence must have Nz entries"
    
    # Layer-cake background
    facies = np.broadcast_to(np.array(cfg.layer_sequence)[None, None, :], (Nx, Ny, Nz)).copy()
    
    # Lateral variability
    lc = (cfg.corr_len_xy / grid["dx"], cfg.corr_len_xy / grid["dy"], cfg.corr_len_z / grid["dz"])
    Zf = gaussian_random_field((Nx, Ny, Nz), lc, rng, cfg.spectrum, cfg.fractal_beta)
    facies[(facies == 2) & (Zf < cfg.lateral_shaliness_threshold)] = 1
    facies[(facies == 1) & (Zf > cfg.lateral_cleaning_threshold)] = 2
    
    # Meandering channels
    X, Y = grid["X"], grid["Y"]
    channel_mask = np.zeros((Nx, Ny, Nz), dtype=np.int16)
    for ci, ch in enumerate(cfg.channels):
        yc = ch["y0"] + ch["amplitude"] * np.sin(2 * np.pi * X[:, :, 0] / ch["wavelength"] + ch["phase"])
        inside_xy = np.abs(Y[:, :, 0] - yc) <= ch["width"] / 2
        for k in ch["layers"]:
            facies[:, :, k][inside_xy] = 3
            channel_mask[:, :, k][inside_xy] = ci + 1
    
    return dict(facies=facies.astype(np.int8), channel_mask=channel_mask, facies_field_Z=Zf)


# ============================================================================
# 4. POROSITY & PERMEABILITY
# ============================================================================
def build_porosity(facies: np.ndarray, table: Dict[int, FaciesParams], 
                   grid: dict, cfg: ReservoirConfig, rng: np.random.Generator) -> np.ndarray:
    lc = (cfg.corr_len_xy / grid["dx"], cfg.corr_len_xy / grid["dy"], cfg.corr_len_z / grid["dz"])
    Z = gaussian_random_field(facies.shape, lc, rng, cfg.spectrum, cfg.fractal_beta)
    phi = np.zeros_like(Z)
    for code, fp in table.items():
        m = facies == code
        phi[m] = np.clip(fp.phi_mean + fp.phi_std * Z[m], fp.phi_min, fp.phi_max)
    return phi


def build_permeability(facies: np.ndarray, phi: np.ndarray, table: Dict[int, FaciesParams], 
                       grid: dict, cfg: ReservoirConfig, rng: np.random.Generator) -> dict:
    lc = (cfg.corr_len_xy / grid["dx"], cfg.corr_len_xy / grid["dy"], cfg.corr_len_z / grid["dz"])
    Zk = gaussian_random_field(facies.shape, lc, rng, cfg.spectrum, cfg.fractal_beta)
    Zv = gaussian_random_field(facies.shape, (lc[0] / 2, lc[1] / 2, lc[2]), rng, cfg.spectrum, cfg.fractal_beta)
    
    kx = np.zeros_like(phi)
    kz = np.zeros_like(phi)
    
    for code, fp in table.items():
        m = facies == code
        log10k = fp.logk_a + fp.logk_b * phi[m] + fp.logk_sigma * Zk[m]
        kx[m] = np.clip(10.0 ** log10k, fp.k_min, fp.k_max)
        ratio = 10.0 ** (np.log10(fp.kv_kh) + fp.kv_kh_sigma_log10 * Zv[m])
        kz[m] = kx[m] * np.clip(ratio, 1e-3, 1.0)
    
    ky = kx * cfg.ky_kx_ratio
    return dict(kx=kx, ky=ky, kz=kz)


# ============================================================================
# 5. FAULTS
# ============================================================================
def _fault_frame(fd: dict) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    st, dp = np.deg2rad(fd["strike_deg"]), np.deg2rad(fd["dip_deg"])
    s = np.array([np.cos(st), np.sin(st), 0.0])
    n = np.array([-np.sin(st) * np.sin(dp), np.cos(st) * np.sin(dp), -np.cos(dp)])
    e = np.cross(n, s)
    p0 = np.array([fd["x0"], fd["y0"], fd["z0"]])
    return s, n, e, p0


def fault_signed_distance(fd: dict, X: np.ndarray, Y: np.ndarray, D: np.ndarray) -> np.ndarray:
    s, n, _, p0 = _fault_frame(fd)
    rx, ry, rz = X - p0[0], Y - p0[1], D - p0[2]
    t = s[0] * rx + s[1] * ry
    return n[0] * rx + n[1] * ry + n[2] * rz + fd.get("curvature", 0.0) * t ** 2


def fault_surface_points(fd: dict, grid: dict, nt: int = 40, nu: int = 12) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    s, n, e, p0 = _fault_frame(fd)
    L = np.hypot(grid["Lx"], grid["Ly"])
    T, U = np.meshgrid(np.linspace(-L, L, nt), np.linspace(-5, grid["Lz"] + 40, nu), indexing="ij")
    c = fd.get("curvature", 0.0)
    P = p0[None, None, :] + T[..., None] * s + U[..., None] * e - (c * T ** 2)[..., None] * n
    inside = (P[..., 0] >= 0) & (P[..., 0] <= grid["Lx"]) & (P[..., 1] >= 0) & (P[..., 1] <= grid["Ly"])
    P[~inside] = np.nan
    return P[..., 0], P[..., 1], P[..., 2]


def build_faults(grid: dict, cfg: ReservoirConfig) -> dict:
    Nx, Ny, Nz = grid["Nx"], grid["Ny"], grid["Nz"]
    X, Y, D = grid["X"], grid["Y"], grid["depth"]
    
    mult_x = np.ones((Nx - 1, Ny, Nz))
    mult_y = np.ones((Nx, Ny - 1, Nz))
    mult_z = np.ones((Nx, Ny, Nz - 1))
    face_x = np.zeros_like(mult_x, dtype=np.int8)
    face_y = np.zeros_like(mult_y, dtype=np.int8)
    face_z = np.zeros_like(mult_z, dtype=np.int8)
    cell_mask = np.zeros((Nx, Ny, Nz), dtype=np.int8)
    fault_defs = []
    
    for fi, fd in enumerate(cfg.faults):
        d = fault_signed_distance(fd, X, Y, D)
        cell_mask[np.abs(d) <= fd.get("thickness", grid["dx"]) / 2] = fi + 1
        
        cx = d[:-1] * d[1:] < 0
        cy = d[:, :-1] * d[:, 1:] < 0
        cz = d[:, :, :-1] * d[:, :, 1:] < 0
        
        for cm, fm, mm in ((cx, face_x, mult_x), (cy, face_y, mult_y), (cz, face_z, mult_z)):
            fm[cm] = fi + 1
            mm[cm] = np.minimum(mm[cm], fd["trans_mult"])
        
        s, n, e, p0 = _fault_frame(fd)
        fault_defs.append(dict(fd, id=fi + 1, strike_vec=s.tolist(), normal_vec=n.tolist(), point=p0.tolist(),
                               n_cells=int((cell_mask == fi + 1).sum()),
                               n_faces=int((face_x == fi + 1).sum() + (face_y == fi + 1).sum() + (face_z == fi + 1).sum())))
    
    return dict(cell_mask=cell_mask, face_mask_x=face_x, face_mask_y=face_y, face_mask_z=face_z,
                trans_mult_x=mult_x, trans_mult_y=mult_y, trans_mult_z=mult_z, definitions=fault_defs,
                note="Face arrays index internal faces. Throw/juxtaposition not modelled.")


# ============================================================================
# 6. FRACTURES
# ============================================================================
def build_fractures(grid: dict, cfg: ReservoirConfig, kx: np.ndarray, ky: np.ndarray, kz: np.ndarray) -> dict:
    X, Y = grid["X"][:, :, 0], grid["Y"][:, :, 0]
    mask = np.zeros(kx.shape, dtype=np.int8)
    kx_f, ky_f, kz_f = kx.copy(), ky.copy(), kz.copy()
    defs = []
    
    for fi, fr in enumerate(cfg.fractures):
        th = np.deg2rad(fr["strike_deg"])
        s = np.array([np.cos(th), np.sin(th)])
        nrm = np.array([-np.sin(th), np.cos(th)])
        rx, ry = X - fr["xc"], Y - fr["yc"]
        t = s[0] * rx + s[1] * ry
        u = nrm[0] * rx + nrm[1] * ry
        in_xy = (np.abs(t) <= fr["length"] / 2) & (np.abs(u) <= fr["width"] / 2)
        
        mx = 1 + (fr["k_mult_along"] - 1) * np.cos(th) ** 2 + (fr["k_mult_across"] - 1) * np.sin(th) ** 2
        my = 1 + (fr["k_mult_along"] - 1) * np.sin(th) ** 2 + (fr["k_mult_across"] - 1) * np.cos(th) ** 2
        
        for k in fr["layers"]:
            sel = in_xy & (mask[:, :, k] == 0)
            kx_f[:, :, k][sel] *= mx
            ky_f[:, :, k][sel] *= my
            kz_f[:, :, k][sel] *= fr["kz_mult"]
            mask[:, :, k][sel] = fi + 1
        
        corners = [fr["xc"] + a * s[0] * fr["length"] / 2 + b * nrm[0] * fr["width"] / 2 for a, b in ((1, 1), (1, -1), (-1, -1), (-1, 1))]
        cornersy = [fr["yc"] + a * s[1] * fr["length"] / 2 + b * nrm[1] * fr["width"] / 2 for a, b in ((1, 1), (1, -1), (-1, -1), (-1, 1))]
        defs.append(dict(fr, id=fi + 1, corners_x=corners, corners_y=cornersy, n_cells=int((mask == fi + 1).sum()),
                         representation="equivalent_continuum"))
    
    return dict(mask=mask, kx=kx_f, ky=ky_f, kz=kz_f, definitions=defs)


# ============================================================================
# 7 & 8. INITIAL STATE
# ============================================================================
def build_initial_state(grid: dict, facies: np.ndarray, table: Dict[int, FaciesParams], 
                        cfg: ReservoirConfig) -> dict:
    D = grid["depth"]
    P_owc = cfg.p_ref + cfg.rho_w * G * (cfg.owc_depth - cfg.depth_ref)
    
    P = np.where(D >= cfg.owc_depth,
                 P_owc + cfg.rho_w * G * (D - cfg.owc_depth),
                 P_owc + cfg.rho_o * G * (D - cfg.owc_depth))
    
    swc = np.zeros_like(D)
    for code, fp in table.items():
        swc[facies == code] = fp.swc if fp.net else 1.0
    
    h = cfg.owc_depth - D
    tz = max(cfg.transition_zone, 1e-9)
    frac = np.clip(h / tz, 0.0, 1.0)
    Sw = 1.0 - frac * (1.0 - swc)
    Sw = np.clip(Sw, 0.0, 1.0)
    So = 1.0 - Sw
    Sg = np.zeros_like(Sw)
    
    return dict(pressure=P, Sw=Sw, So=So, Sg=Sg, swc=swc, P_owc=P_owc,
                pressure_units="Pa", convention="depth positive downward, P increases with depth")


# ============================================================================
# 9. WELLS
# ============================================================================
def build_wells(grid: dict, facies: np.ndarray, So: np.ndarray, table: Dict[int, FaciesParams], 
                cfg: ReservoirConfig) -> List[dict]:
    wells = []
    for w in cfg.wells:
        i = int(np.clip(w["x"] // grid["dx"], 0, grid["Nx"] - 1))
        j = int(np.clip(w["y"] // grid["dy"], 0, grid["Ny"] - 1))
        col_f = facies[i, j, :]
        col_so = So[i, j, :]
        net = np.array([table[int(c)].net for c in col_f])
        
        if w["perforate"] == "reservoir":
            perf = np.where(net)[0]
        elif w["perforate"] == "oil_reservoir":
            perf = np.where(net & (col_so > 0.05))[0]
        elif w["perforate"] == "all":
            perf = np.arange(grid["Nz"])
        else:
            perf = np.array(w["perforate"], dtype=int)
        
        wells.append(dict(
            name=w["name"], type=w["type"], x=w["x"], y=w["y"], i=i, j=j,
            cell_x=float(grid["x"][i]), cell_y=float(grid["y"][j]),
            perforated_layers=[int(k) for k in perf],
            perforation_depths=[float(grid["depth"][i, j, k]) for k in perf],
            perforated_facies=[FACIES_NAMES[int(col_f[k])] for k in perf],
            control=w["control"], value=w["value"], units=w["units"],
            rw=w["rw"], skin=w["skin"], geometry="vertical", perforation_rule=w["perforate"],
        ))
    return wells


# ============================================================================
# 10. ASSEMBLY
# ============================================================================
def build_reservoir(cfg: Optional[ReservoirConfig] = None) -> dict:
    cfg = cfg or ReservoirConfig()
    rng = np.random.default_rng(cfg.seed)
    table = default_facies_table()
    
    grid = build_grid(cfg)
    fac = build_facies(grid, cfg, rng)
    phi = build_porosity(fac["facies"], table, grid, cfg, rng)
    perm = build_permeability(fac["facies"], phi, table, grid, cfg, rng)
    faults = build_faults(grid, cfg)
    frac = build_fractures(grid, cfg, perm["kx"], perm["ky"], perm["kz"])
    state = build_initial_state(grid, fac["facies"], table, cfg)
    wells = build_wells(grid, fac["facies"], state["So"], table, cfg)
    
    return {
        "config": cfg,
        "facies_table": table,
        "grid": grid,
        "facies": fac["facies"],
        "channel_mask": fac["channel_mask"],
        "phi": phi,
        "kx_matrix": perm["kx"], "ky_matrix": perm["ky"], "kz_matrix": perm["kz"],
        "kx": frac["kx"], "ky": frac["ky"], "kz": frac["kz"],
        "faults": faults,
        "fractures": dict(mask=frac["mask"], definitions=frac["definitions"]),
        "initial_state": dict(pressure=state["pressure"], Sw=state["Sw"], So=state["So"], Sg=state["Sg"]),
        "swc": state["swc"],
        "wells": wells,
        "units": dict(length="m", pressure="Pa", permeability="mD", density="kg/m3", md_to_m2=MD_TO_M2),
    }


# ============================================================================
# 11. VALIDATION
# ============================================================================
def validate_reservoir(res: dict, raise_on_fail: bool = True) -> List[str]:
    cfg, table, g = res["config"], res["facies_table"], res["grid"]
    fac, phi = res["facies"], res["phi"]
    st = res["initial_state"]
    problems = []
    
    def chk(cond, msg):
        if not cond:
            problems.append(msg)
    
    for code, fp in table.items():
        m = fac == code
        if m.any():
            chk(phi[m].min() >= fp.phi_min - 1e-12 and phi[m].max() <= fp.phi_max + 1e-12, f"phi out of range in {fp.name}")
            km = res["kx_matrix"][m]
            chk(km.min() >= fp.k_min - 1e-12 and km.max() <= fp.k_max + 1e-12, f"kx_matrix out of range in {fp.name}")
    
    for k in ("kx", "ky", "kz", "kx_matrix", "ky_matrix", "kz_matrix"):
        chk(np.all(np.isfinite(res[k])) and np.all(res[k] > 0), f"{k} must be finite and > 0")
    chk(np.all(res["kz_matrix"] <= res["kx_matrix"] + 1e-12), "kz_matrix > kx_matrix somewhere")
    
    S = st["Sw"] + st["So"] + st["Sg"]
    chk(np.allclose(S, 1.0, atol=1e-10), "Sw+So+Sg != 1")
    for k in ("Sw", "So", "Sg"):
        chk(np.all((st[k] >= 0) & (st[k] <= 1)), f"{k} outside [0,1]")
    
    D = g["depth"]
    chk(np.all(st["Sw"][D >= cfg.owc_depth] == 1.0), "oil found below OWC")
    net = np.isin(fac, [c for c, fp in table.items() if fp.net])
    chk(np.all(st["Sw"][~net] == 1.0), "non-net (shale) cells must have Sw = 1")
    
    dP = np.diff(st["pressure"], axis=2)
    chk(np.all(dP > 0), "pressure does not increase downward in every column")
    chk(np.all(np.diff(D, axis=2) > 0), "depth not increasing with k")
    
    wl = (D[:, :, :-1] >= cfg.owc_depth)
    if wl.any():
        grad = dP[wl] / np.diff(D, axis=2)[wl]
        chk(np.allclose(grad, cfg.rho_w * G, rtol=1e-6), "water-leg gradient != rho_w g")
    
    for w in res["wells"]:
        chk(0 <= w["i"] < g["Nx"] and 0 <= w["j"] < g["Ny"], f"{w['name']} outside grid")
        chk(0 <= w["x"] <= g["Lx"] and 0 <= w["y"] <= g["Ly"], f"{w['name']} physical location outside reservoir")
        chk(len(w["perforated_layers"]) > 0, f"{w['name']} has no perforations")
        for k, fn in zip(w["perforated_layers"], w["perforated_facies"]):
            chk(0 <= k < g["Nz"], f"{w['name']} perforation layer {k} out of range")
            if w["perforation_rule"] in ("reservoir", "oil_reservoir"):
                chk(table[FACIES_CODES[fn]].net, f"{w['name']} perforates non-net facies at k={k}")
            if w["perforation_rule"] == "oil_reservoir":
                chk(st["So"][w["i"], w["j"], k] > 0.05, f"{w['name']} perforates water-bearing layer k={k}")
    
    fl = res["faults"]
    for fd in fl["definitions"]:
        chk(fd["n_faces"] > 0, f"fault {fd['name']} crosses no cell faces (invalid geometry)")
        chk(0 < fd["trans_mult"], f"fault {fd['name']} multiplier must be > 0")
    for arr in (fl["trans_mult_x"], fl["trans_mult_y"], fl["trans_mult_z"]):
        chk(np.all((arr > 0) & (arr <= 1.0)), "fault multipliers must be in (0,1]")
    chk(fl["trans_mult_x"].shape == (g["Nx"] - 1, g["Ny"], g["Nz"]), "trans_mult_x wrong shape")
    
    for fr in res["fractures"]["definitions"]:
        chk(fr["n_cells"] > 0, f"fracture {fr['name']} contains no cells")
        chk(all(0 <= k < g["Nz"] for k in fr["layers"]), f"fracture {fr['name']} layer out of range")
    chk(np.all(res["fractures"]["mask"][res["kx"] > res["kx_matrix"]] > 0), "perm enhanced outside fracture mask")
    
    if problems and raise_on_fail:
        raise AssertionError("Validation failed:\n  - " + "\n  - ".join(problems))
    return problems


# ============================================================================
# 12. OUTPUT
# ============================================================================
def _jsonable(o):
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, dict):
        return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_jsonable(v) for v in o]
    return o


def save_reservoir(res: dict, outdir: str, name: str = "synthetic_reservoir") -> None:
    os.makedirs(outdir, exist_ok=True)
    g, fl = res["grid"], res["faults"]
    np.savez_compressed(
        os.path.join(outdir, f"{name}.npz"),
        x=g["x"], y=g["y"], depth=g["depth"], top_depth_map=g["top_depth_map"], cell_volume=g["cell_volume"],
        facies=res["facies"], channel_mask=res["channel_mask"], phi=res["phi"],
        kx=res["kx"], ky=res["ky"], kz=res["kz"],
        kx_matrix=res["kx_matrix"], ky_matrix=res["ky_matrix"], kz_matrix=res["kz_matrix"],
        fault_cell_mask=fl["cell_mask"], fault_face_mask_x=fl["face_mask_x"], fault_face_mask_y=fl["face_mask_y"],
        fault_face_mask_z=fl["face_mask_z"], fault_trans_mult_x=fl["trans_mult_x"],
        fault_trans_mult_y=fl["trans_mult_y"], fault_trans_mult_z=fl["trans_mult_z"],
        fracture_mask=res["fractures"]["mask"], swc=res["swc"],
        pressure=res["initial_state"]["pressure"], Sw=res["initial_state"]["Sw"],
        So=res["initial_state"]["So"], Sg=res["initial_state"]["Sg"],
    )
    cfg = res["config"]
    meta = dict(
        disclaimer="SYNTHETIC model. All ranges are illustrative; not calibrated to any real field.",
        grid=dict(Nx=g["Nx"], Ny=g["Ny"], Nz=g["Nz"], Lx=g["Lx"], Ly=g["Ly"], Lz=g["Lz"],
                  dx=g["dx"], dy=g["dy"], dz=g["dz"], depth_convention=g["depth_convention"],
                  array_shape="(Nx,Ny,Nz) C-order, indexing='ij'"),
        structure=dict(top_depth=cfg.top_depth, dip_x=cfg.dip_x, dip_y=cfg.dip_y,
                       dome_center=cfg.dome_center, dome_amplitude=cfg.dome_amplitude, dome_radius=cfg.dome_radius),
        seed=cfg.seed,
        facies_codes=FACIES_CODES,
        facies_definitions={fp.name: asdict(fp) for fp in res["facies_table"].values()},
        layer_sequence=[FACIES_NAMES[c] for c in cfg.layer_sequence],
        channels=cfg.channels,
        heterogeneity=dict(corr_len_xy=cfg.corr_len_xy, corr_len_z=cfg.corr_len_z, spectrum=cfg.spectrum,
                           fractal_beta=cfg.fractal_beta, ky_kx_ratio=cfg.ky_kx_ratio),
        property_ranges=dict(phi=[float(res["phi"].min()), float(res["phi"].max())],
                             kx=[float(res["kx"].min()), float(res["kx"].max())],
                             kz=[float(res["kz"].min()), float(res["kz"].max())]),
        faults=fl["definitions"], fault_note=fl["note"],
        fractures=res["fractures"]["definitions"],
        fracture_note="Equivalent-continuum corridors (matrix perm multipliers). Not a DFM.",
        fluids=dict(owc_depth=cfg.owc_depth, transition_zone=cfg.transition_zone, p_ref=cfg.p_ref,
                    depth_ref=cfg.depth_ref, rho_w=cfg.rho_w, rho_o=cfg.rho_o,
                    pressure_convention="P = P_ref + rho g (D - D_ref); D positive downward"),
        wells=res["wells"],
        units=res["units"],
    )
    with open(os.path.join(outdir, f"{name}.json"), "w") as f:
        json.dump(_jsonable(meta), f, indent=2)


def load_reservoir(outdir: str, name: str = "synthetic_reservoir"):
    arrays = dict(np.load(os.path.join(outdir, f"{name}.npz")))
    with open(os.path.join(outdir, f"{name}.json")) as f:
        meta = json.load(f)
    return arrays, meta


# ============================================================================
# 13. INTERACTIVE VISUALIZATION (Plotly)
# ============================================================================
def plot_reservoir_interactive(res: dict, max_points: int = 20000) -> "go.Figure":
    """
    Creates an interactive 3D Plotly figure of the reservoir.
    
    Controls:
    - Scroll: Zoom in/out
    - Left-drag: Rotate camera
    - Right-drag or Middle-drag: Pan
    - Hover: Show facies, property values, well info
    - Legend: Toggle visibility of layers
    """
    try:
        import plotly.graph_objects as go
    except ImportError:
        raise ImportError("plotly is required for interactive visualization. Install with: pip install plotly")
    
    g = res["grid"]
    fac = res["facies"]
    phi = res["phi"]
    kx = res["kx"]
    state = res["initial_state"]
    wells = res["wells"]
    faults = res["faults"]
    frs = res["fractures"]
    cfg = res["config"]
    oxc = cfg.owc_depth
    Nx, Ny, Nz = g["Nx"], g["Ny"], g["Nz"]
    
    # Sample points for performance
    N_total = Nx * Ny * Nz
    N_plot = min(max_points, N_total)
    np.random.seed(20240601 + cfg.seed)
    idx = np.random.choice(N_total, N_plot, replace=False)
    i_idx = idx // (Ny * Nz)
    j_idx = (idx // Nz) % Ny
    k_idx = idx % Nz
    
    X = g["X"][i_idx, j_idx, k_idx]
    Y = g["Y"][i_idx, j_idx, k_idx]
    Z = g["depth"][i_idx, j_idx, k_idx]
    F = fac[i_idx, j_idx, k_idx]
    PHI = phi[i_idx, j_idx, k_idx]
    KX = kx[i_idx, j_idx, k_idx]
    
    fig = go.Figure()
    
    # Facies groups
    for code, col, name in zip([0, 1, 2, 3], FACIES_COLORS, ["shale", "shaly_sandstone", "sandstone", "channel_sandstone"]):
        m = F == code
        if m.any():
            fig.add_trace(go.Scatter3d(
                x=X[m], y=Y[m], z=Z[m],
                mode='markers',
                marker=dict(size=3, color=col, opacity=0.6, line=dict(width=0)),
                showlegend=True,
                name=f"{name} (code={code})",
                hovertemplate=f"Facies: {name}<br>X: %{{x:.1f}} m<br>Y: %{{y:.1f}} m<br>Depth: %{{z:.1f}} m<extra></extra>"
            ))
    
    # Porosity (toggleable)
    fig.add_trace(go.Scatter3d(
        x=X, y=Y, z=Z,
        mode='markers',
        marker=dict(size=2, color=PHI, colorscale='Viridis', opacity=0.7,
                    showscale=True, colorbar=dict(title="Porosity ÃƒÂÃ¢â‚¬Â ", x=1.0)),
        showlegend=True,
        name="Porosity",
        visible='legendonly',
        hovertemplate="ÃƒÂÃ¢â‚¬Â : %{marker.color:.3f}<br>X: %{x:.1f} m<br>Y: %{y:.1f} m<br>Depth: %{z:.1f} m<extra></extra>"
    ))
    
    # Permeability (toggleable)
    KH = np.sqrt(KX * (res["ky_matrix"][i_idx, j_idx, k_idx] if "ky_matrix" in res else res["ky"][i_idx, j_idx, k_idx]))
    fig.add_trace(go.Scatter3d(
        x=X, y=Y, z=Z,
        mode='markers',
        marker=dict(size=2, color=KH, colorscale='Plasma', opacity=0.7,
                    showscale=True, colorbar=dict(title="k_h [mD]", x=1.02)),
        showlegend=True,
        name="Permeability",
        visible='legendonly',
        hovertemplate="k_h: %{marker.color:.1f} mD<br>X: %{x:.1f} m<br>Y: %{y:.1f} m<br>Depth: %{z:.1f} m<extra></extra>"
    ))
    
    # Wells
    for w in wells:
        z_head = g["depth"][w["i"], w["j"], 0]
        z_perf = w["perforation_depths"]
        col = "blue" if w["type"] == "injector" else "red"
        
        # Well head
        fig.add_trace(go.Scatter3d(
            x=[w["cell_x"]], y=[w["cell_y"]], z=[z_head],
            mode='markers',
            marker=dict(size=10, color=col, symbol="circle-open", line=dict(color="black", width=2)),
            showlegend=True,
            name=f"{w['name']} (head)",
            hovertemplate=f"Well: {w['name']}<br>Type: {w['type']}<br>X: {w['cell_x']:.1f} m<br>Y: {w['cell_y']:.1f} m<extra></extra>"
        ))
        
        # Perforations
        if len(z_perf) > 0:
            fig.add_trace(go.Scatter3d(
                x=[w["cell_x"]] * len(z_perf),
                y=[w["cell_y"]] * len(z_perf),
                z=z_perf,
                mode='lines+markers',
                marker=dict(size=5, color=col, line=dict(color="black", width=0.5)),
                line=dict(color=col, width=3, dash='dot'),
                showlegend=False,
                name=f"{w['name']} perfs",
                hovertemplate=f"{w['name']} perf<br>Depth: %{{z:.1f}} m<extra></extra>"
            ))
    
    # Oil-Water Contact
    xx = np.linspace(0, g["Lx"], 30)
    yy = np.linspace(0, g["Ly"], 30)
    XX, YY = np.meshgrid(xx, yy)
    ZZ = np.full_like(XX, oxc)
    fig.add_trace(go.Surface(
        x=XX, y=YY, z=ZZ,
        colorscale=[[0, 'rgba(0,255,255,0.3)'], [1, 'rgba(0,255,255,0.3)']],
        showscale=False, opacity=0.4, name="OWC",
        hovertemplate="Oil-Water Contact<br>Depth: 2040.0 m<extra></extra>"
    ))
    
    # Fault cells
    for fd in faults["definitions"]:
        m = faults["cell_mask"].ravel() == fd["id"]
        if m.any():
            fig.add_trace(go.Scatter3d(
                x=g["X"].ravel()[m], y=g["Y"].ravel()[m], z=g["depth"].ravel()[m],
                mode='markers',
                marker=dict(size=2, color="black", opacity=0.5),
                showlegend=True,
                name=f"Fault {fd['name']} (mult={fd['trans_mult']})",
                hovertemplate=f"Fault: {fd['name']}<br>Trans mult: {fd['trans_mult']}<extra></extra>"
            ))
    
    # Fracture corridors
    for fr in frs["definitions"]:
        m = frs["mask"].ravel() == fr["id"]
        if m.any():
            fig.add_trace(go.Scatter3d(
                x=g["X"].ravel()[m], y=g["Y"].ravel()[m], z=g["depth"].ravel()[m],
                mode='markers',
                marker=dict(size=2, color="magenta", opacity=0.5),
                showlegend=True,
                name=f"Fracture {fr['name']} (x{fr['k_mult_along']})",
                hovertemplate=f"Fracture: {fr['name']}<br>k_mult: {fr['k_mult_along']}<extra></extra>"
            ))
    
    # Layout
    fig.update_layout(
        title=dict(
            text="Synthetic Reservoir ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“ Interactive 3D View<br><sup>Scroll to zoom, Left-drag to rotate, Right-drag to pan</sup>",
            x=0.5, xanchor='center', font=dict(size=16)
        ),
        scene=dict(
            xaxis_title="X [m]",
            yaxis_title="Y [m]",
            zaxis_title="Depth [m]",
            zaxis_autorange="reversed",
            aspectmode="data",
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.2)
            )
        ),
        margin=dict(l=0, r=0, b=0, t=60),
        legend=dict(x=0.02, y=0.98, traceorder="normal", bgcolor="rgba(255,255,255,0.8)"),
        hovermode='closest',
        dragmode='orbit'
    )
    
    return fig


# ============================================================================
# 14. SUMMARY
# ============================================================================
def summarize(res: dict) -> None:
    g, fac = res["grid"], res["facies"]
    print("=" * 70)
    print("SYNTHETIC RESERVOIR MODEL SUMMARY")
    print("=" * 70)
    print(f"Grid: {g['Nx']} x {g['Ny']} x {g['Nz']} cells")
    print(f"Cell size: {g['dx']:.1f} x {g['dy']:.1f} x {g['dz']:.1f} m")
    print(f"Physical dimensions: {g['Lx']:.0f} x {g['Ly']:.0f} x {g['Lz']:.0f} m")
    print(f"Depth range: {g['depth'].min():.1f} - {g['depth'].max():.1f} m")
    print(f"Seed: {res['config'].seed}")
    print()
    
    print("Facies distribution:")
    for code, name in FACIES_NAMES.items():
        m = fac == code
        if m.any():
            pct = 100 * m.mean()
            phi_mean = res['phi'][m].mean()
            kx_geo = np.exp(np.mean(np.log(res['kx'][m] + 1e-12)))
            kv_kh_med = np.median(res['kz'][m] / res['kx'][m])
            print(f"  {name:20s}: {pct:5.1f}%  ÃƒÂÃ¢â‚¬Â ={phi_mean:.3f}  kx(geo)={kx_geo:8.1f} mD  kz/kx={kv_kh_med:.3f}")
    
    print()
    print("Faults:")
    for f in res['faults']['definitions']:
        print(f"  {f['name']}: {f['n_cells']} cells, {f['n_faces']} faces, trans_mult={f['trans_mult']}")
    
    print()
    print("Fracture corridors:")
    for f in res['fractures']['definitions']:
        print(f"  {f['name']}: {f['n_cells']} cells, k_mult_along={f['k_mult_along']}")
    
    P = res['initial_state']['pressure']
    oil_pore_vol = (res['phi'] * res['initial_state']['So'] * g['cell_volume']).sum()
    print()
    print(f"Pressure: {P.min()/1e6:.2f} - {P.max()/1e6:.2f} MPa")
    print(f"OWC depth: {res['config'].owc_depth} m")
    print(f"Oil in place (pore volume): {oil_pore_vol:.3e} mÃƒâ€šÃ‚Â³")
    
    print()
    print("Wells:")
    for w in res['wells']:
        print(f"  {w['name']:6s} ({w['type']:9s}) at ({w['i']},{w['j']})")
        print(f"    Perforations: layers {w['perforated_layers']}")
        print(f"    Control: {w['control']} = {w['value']} {w['units']}")
    
    print("=" * 70)


# ============================================================================
# MAIN
# ============================================================================
if __name__ == "__main__":
    print("Building synthetic reservoir model...")
    res = build_reservoir(ReservoirConfig())
    
    print("Validating model...")
    problems = validate_reservoir(res, raise_on_fail=False)
    if problems:
        print("Validation warnings:")
        for p in problems:
            print(f"  - {p}")
    else:
        print("Validation passed.")
    
    summarize(res)
    
    print("\nSaving model to ./output ...")
    save_reservoir(res, "output")
    print("Saved: output/synthetic_reservoir.npz")
    print("Saved: output/synthetic_reservoir.json")
    
    print("\nGenerating interactive 3D visualization...")
    try:
        fig = plot_reservoir_interactive(res)
        fig.write_html("output/reservoir_3d_interactive.html")
        print("Saved: output/reservoir_3d_interactive.html")
        print("\nOpening interactive plot in browser...")
        fig.show()
        print("\nInteractive controls:")
        print("  - Scroll wheel: Zoom in/out")
        print("  - Left mouse drag: Rotate view")
        print("  - Right mouse drag: Pan")
        print("  - Legend clicks: Toggle layers on/off")
        print("  - Hover over points: See facies, properties, well info")
    except ImportError as e:
        print(f"Could not create interactive plot: {e}")
        print("Install plotly with: pip install plotly")
    
    print("\nDone.")

Building synthetic reservoir model...
Validating model...
Validation passed.
SYNTHETIC RESERVOIR MODEL SUMMARY
Grid: 40 x 40 x 10 cells
Cell size: 25.0 x 25.0 x 5.0 m
Physical dimensions: 1000 x 1000 x 50 m
Depth range: 1991.6 - 2053.4 m
Seed: 20240601

Facies distribution:
  shale               :  30.0%  ÃƒÂÃ¢â‚¬Â =0.051  kx(geo)=     0.0 mD  kz/kx=0.053
  shaly_sandstone     :  32.5%  ÃƒÂÃ¢â‚¬Â =0.130  kx(geo)=     3.2 mD  kz/kx=0.090
  sandstone           :  32.4%  ÃƒÂÃ¢â‚¬Â =0.201  kx(geo)=   104.8 mD  kz/kx=0.208
  channel_sandstone   :   5.1%  ÃƒÂÃ¢â‚¬Â =0.245  kx(geo)=   877.6 mD  kz/kx=0.329

Faults:
  F1: 198 cells, 543 faces, trans_mult=0.02

Fracture corridors:
  FR1: 42 cells, k_mult_along=40.0
  FR2: 36 cells, k_mult_along=25.0
  FR3: 24 cells, k_mult_along=30.0

Pressure: 20.02 - 20.53 MPa
OWC depth: 2040.0 m
Oil in place (pore volume): 4.031e+06 mÃƒâ€šÃ‚Â³

Wells:
  INJ1   (injector ) at (6,6)
    Perforations: layers [1, 2, 3, 5, 6, 7, 8]
    Control: rate = 300.0 m


Interactive controls:
  - Scroll wheel: Zoom in/out
  - Left mouse drag: Rotate view
  - Right mouse drag: Pan
  - Legend clicks: Toggle layers on/off
  - Hover over points: See facies, properties, well info

Done.


In [ ]:
"""
carbonate_reservoir.py

Standalone synthetic 3D carbonate reservoir INITIALIZATION.
No flow solver is included.

Dependencies:
    pip install numpy plotly

Run:
    python carbonate_reservoir.py

Outputs:
    carbonate_output/carbonate_reservoir.npz
    carbonate_output/carbonate_reservoir.json
    carbonate_output/figures/*.html

All geological properties and numerical ranges in this module are
ILLUSTRATIVE/SYNTHETIC, not calibrated to a real field.

Units:
    Length: m
    Depth: m, POSITIVE DOWNWARD
    Pressure: Pa
    Permeability: mD
    Formation compressibility: Pa^-1

Arrays use shape (Nx, Ny, Nz), indexing="ij", C order.
Layer k=0 is the uppermost layer.
"""

from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np


G = 9.80665
MD_TO_M2 = 9.869233e-16

# Depositional facies. Dolomitization is recorded separately as an overprint
# and in rock_facies; it does not erase the precursor depositional facies.
FACIES = {
    0: "mudstone_wackestone",
    1: "packstone",
    2: "grainstone",
    3: "boundstone_reef",
    4: "evaporite_anhydrite",
}
DOLOMITE_CODE = 5

PORE_TYPES = {
    0: "interparticle",
    1: "intraparticle",
    2: "moldic",
    3: "vuggy",
    4: "microporosity",
    5: "fracture",
}

DIAGENESIS_TYPES = {
    0: "unmodified",
    1: "dissolution_dominant",
    2: "cementation_dominant",
    3: "dolomitization_dominant",
}

FACIES_COLORS = {
    0: "#777e87",
    1: "#c9aa71",
    2: "#f1ce55",
    3: "#e67640",
    4: "#8c72a6",
}
PORE_COLORS = {
    0: "#e9c46a",
    1: "#a6c86a",
    2: "#ee8f5e",
    3: "#db5673",
    4: "#6486b9",
    5: "#a759c8",
}


@dataclass(frozen=True)
class FaciesProperties:
    """Synthetic facies-conditioned petrophysical parameters."""

    phi_mean: float
    phi_std: float
    phi_min: float
    phi_max: float
    log10_k_mean: float
    log10_k_scatter: float
    k_min_md: float
    k_max_md: float
    kv_kh: float
    swc_base: float
    net_reservoir: bool
    # Fractions for interparticle, intraparticle, moldic, vuggy, micropore.
    pore_weights: Tuple[float, float, float, float, float]


def facies_properties() -> Dict[int, FaciesProperties]:
    # Illustrative only. Wide k distributions deliberately overlap between
    # facies: pore connectivity is not inferred from porosity alone.
    return {
        0: FaciesProperties(
            .085, .025, .025, .20, -1.10, .90, .003, 100,
            .035, .43, False, (.12, .25, .08, .03, .52)
        ),
        1: FaciesProperties(
            .125, .030, .045, .255, -.35, 1.00, .006, 350,
            .11, .36, True, (.32, .24, .12, .07, .25)
        ),
        2: FaciesProperties(
            .165, .035, .055, .30, .30, 1.05, .006, 1000,
            .22, .27, True, (.55, .15, .15, .07, .08)
        ),
        3: FaciesProperties(
            .155, .055, .045, .34, .05, 1.15, .006, 2000,
            .28, .33, True, (.22, .14, .23, .30, .11)
        ),
        4: FaciesProperties(
            .025, .009, .005, .07, -3.20, .55, 1e-5, .05,
            .008, .65, False, (.03, .06, .02, .01, .88)
        ),
    }


@dataclass
class Config:
    # Grid and structure.
    Nx: int = 40
    Ny: int = 40
    Nz: int = 10
    Lx: float = 1000.0
    Ly: float = 1000.0
    Lz: float = 50.0
    top_depth_m: float = 2000.0
    dip_x: float = .003
    dip_y: float = .002
    dome_amplitude_m: float = 10.0
    dome_radius_m: float = 380.0
    platform_center_xy: Tuple[float, float] = (500.0, 500.0)

    # Heterogeneity.
    seed: int = 20250714
    lateral_correlation_m: float = 110.0
    vertical_correlation_m: float = 7.0
    random_spectrum: str = "gaussian"  # or "fractal"
    fractal_beta: float = 3.0

    # Contacts and initial fluids.
    owc_depth_m: float = 2040.0
    pressure_reference_pa: float = 20.4e6
    pressure_reference_depth_m: float = 2040.0
    rho_water_kg_m3: float = 1020.0
    rho_oil_kg_m3: float = 810.0
    saturation_transition_m: float = 5.0
    formation_compressibility_pa_inv: float = 7e-10

    # Well placement rules.
    min_owc_clearance_m: float = 12.0
    maximum_completed_oil_column_fraction: float = .49

    # x = x0 + slope_y*(y-y0) + slope_depth*(D-D0)
    #     + curvature*(y-y0)^2.
    #
    # Face multiplier is distinct from matrix permeability. Values above 1
    # are permitted for a conductive fault.
    faults: List[dict] = field(default_factory=lambda: [
        {
            "name": "F1",
            "x0": 540.0,
            "y0": 500.0,
            "D0": 2020.0,
            "slope_y": .18,
            "slope_depth": .12,
            "curvature": .00010,
            "zone_width_m": 18.0,
            "fault_rock": "synthetic_cemented_breccia",
            "transmissibility_multiplier": .04,
        }
    ])

    # Equivalent-continuum corridors, NOT discrete-fracture elements.
    fractures: List[dict] = field(default_factory=lambda: [
        {
            "name": "FC1", "x": 330.0, "y": 710.0,
            "strike_deg": 32.0, "length_m": 540.0, "width_m": 35.0,
            "layers": [2, 3, 5, 6, 7],
            "aperture_mm": .35,
            "k_along_multiplier": 35.0,
            "k_across_multiplier": 3.0,
            "k_vertical_multiplier": 8.0,
        },
        {
            "name": "FC2", "x": 750.0, "y": 330.0,
            "strike_deg": 117.0, "length_m": 520.0, "width_m": 35.0,
            "layers": [2, 3, 5, 6, 7],
            "aperture_mm": .25,
            "k_along_multiplier": 25.0,
            "k_across_multiplier": 3.0,
            "k_vertical_multiplier": 7.0,
        },
        {
            "name": "FC3", "x": 300.0, "y": 300.0,
            "strike_deg": 65.0, "length_m": 300.0, "width_m": 35.0,
            "layers": [5, 6, 7, 8],
            "aperture_mm": .20,
            "k_along_multiplier": 18.0,
            "k_across_multiplier": 2.0,
            "k_vertical_multiplier": 5.0,
        },
    ])

    wells: List[dict] = field(default_factory=lambda: [
        {
            "name": "INJ1", "kind": "injector",
            "x": 275.0, "y": 500.0,
            "control": "water_rate",
            "value": 200.0, "units": "reservoir_m3/day",
            "wellbore_radius_m": .10, "skin": 0.0,
        },
        {
            "name": "PROD1", "kind": "producer",
            "x": 710.0, "y": 505.0,
            "control": "bhp",
            "value": 18e6, "units": "Pa",
            "wellbore_radius_m": .10, "skin": 0.0,
        },
    ])


def check_config(cfg: Config) -> None:
    if min(cfg.Nx, cfg.Ny, cfg.Nz) < 2:
        raise ValueError("Each grid dimension must be >= 2.")
    if min(cfg.Lx, cfg.Ly, cfg.Lz) <= 0:
        raise ValueError("Physical dimensions must be positive.")
    if cfg.random_spectrum not in ("gaussian", "fractal"):
        raise ValueError("random_spectrum must be 'gaussian' or 'fractal'.")
    if cfg.saturation_transition_m <= 0:
        raise ValueError("saturation_transition_m must be positive.")
    if not 0 < cfg.maximum_completed_oil_column_fraction < .5:
        raise ValueError("Maximum completed oil-column fraction must be < 0.5.")
    for fr in cfg.fractures:
        if not fr["layers"] or min(fr["layers"]) < 0 or max(fr["layers"]) >= cfg.Nz:
            raise ValueError(f"Invalid fracture layer range: {fr['name']}")
    for w in cfg.wells:
        if not (0 <= w["x"] < cfg.Lx and 0 <= w["y"] < cfg.Ly):
            raise ValueError(f"Well outside physical grid: {w['name']}")


def make_grid(cfg: Config) -> dict:
    dx, dy, dz = cfg.Lx / cfg.Nx, cfg.Ly / cfg.Ny, cfg.Lz / cfg.Nz
    x = (np.arange(cfg.Nx) + .5) * dx
    y = (np.arange(cfg.Ny) + .5) * dy
    X, Y, z_local = np.meshgrid(
        x, y, (np.arange(cfg.Nz) + .5) * dz, indexing="ij"
    )
    xc, yc = cfg.platform_center_xy
    top = (
        cfg.top_depth_m
        + cfg.dip_x * X[:, :, 0]
        + cfg.dip_y * Y[:, :, 0]
        - cfg.dome_amplitude_m
        * np.exp(
            -((X[:, :, 0] - xc) ** 2 + (Y[:, :, 0] - yc) ** 2)
            / (2 * cfg.dome_radius_m**2)
        )
    )
    depth = top[:, :, None] + z_local
    return {
        "Nx": cfg.Nx, "Ny": cfg.Ny, "Nz": cfg.Nz,
        "Lx": cfg.Lx, "Ly": cfg.Ly, "Lz": cfg.Lz,
        "dx": dx, "dy": dy, "dz": dz,
        "x": x, "y": y, "X": X, "Y": Y,
        "depth": depth, "top_depth": top,
        "cell_volume": np.full(depth.shape, dx * dy * dz),
    }


def correlated_field(
    shape: tuple,
    length_in_cells: tuple,
    rng: np.random.Generator,
    spectrum: str,
    beta: float,
) -> np.ndarray:
    """
    FFT-filtered, approximately stationary correlated field. Periodic FFT
    boundaries are a numerical simplification.
    """
    K = np.meshgrid(*(np.fft.fftfreq(n) for n in shape), indexing="ij")

    if spectrum == "gaussian":
        spectral_density = np.exp(
            -sum(
                (2 * np.pi * freq * length) ** 2
                for freq, length in zip(K, length_in_cells)
            )
        )
    else:
        # Regularized anisotropic power-law spectrum; experimental hook.
        q2 = sum(
            (freq * length) ** 2
            for freq, length in zip(K, length_in_cells)
        )
        spectral_density = (q2 + .15**2) ** (-beta / 2)

    white = rng.standard_normal(shape)
    out = np.fft.ifftn(np.fft.fftn(white) * np.sqrt(spectral_density)).real
    return (out - out.mean()) / max(out.std(), 1e-12)


def field(
    grid: dict, cfg: Config, rng: np.random.Generator,
    horizontal_scale: float = 1.0,
) -> np.ndarray:
    length = (
        cfg.lateral_correlation_m * horizontal_scale / grid["dx"],
        cfg.lateral_correlation_m * horizontal_scale / grid["dy"],
        cfg.vertical_correlation_m / grid["dz"],
    )
    return correlated_field(
        grid["depth"].shape, length, rng,
        cfg.random_spectrum, cfg.fractal_beta
    )


def make_depositional_facies(
    grid: dict, cfg: Config, rng: np.random.Generator
) -> dict:
    """
    Rimmed carbonate platform:
      central lagoon -> inner back-reef shoals -> reef rim ->
      outer deeper-water wackestone.

    Superposed vertical cycles include a localized anhydrite baffle.
    """
    X, Y = grid["X"], grid["Y"]
    xc, yc = cfg.platform_center_xy

    radial = np.sqrt(
        ((X - xc) / 1.08) ** 2 + ((Y - yc) / .94) ** 2
    )
    noise = field(grid, cfg, rng, 1.15)
    rim = 325.0 + 17.0 * noise
    facies = np.zeros(grid["depth"].shape, dtype=np.int8)

    for k in range(cfg.Nz):
        # Thin upper evaporite seal and lower tight platform base.
        if k == 0:
            facies[:, :, k] = 4
            continue
        if k == cfg.Nz - 1:
            facies[:, :, k] = 0
            continue

        radius = radial[:, :, k]
        reef = np.abs(radius - rim[:, :, k]) < 35.0
        shoal = (radius >= 165.0) & (radius < rim[:, :, k] - 35.0)
        lagoon_edge = (radius >= 120.0) & (radius < 165.0)

        layer = np.zeros((cfg.Nx, cfg.Ny), dtype=np.int8)
        layer[lagoon_edge] = 1
        layer[shoal] = 2
        layer[reef] = 3

        # Back-reef patchiness is spatially correlated.
        patch = noise[:, :, k]
        layer[(layer == 2) & (patch < -.55)] = 1
        layer[(layer == 1) & (patch > .9)] = 2

        # Regressive/transgressive stacking: selected horizons are muddier.
        if k in (1, 4, 8):
            layer[layer == 2] = 1
            layer[(layer == 3) & (patch < 0)] = 1

        # Laterally restricted anhydrite baffle, NOT whole-layer replacement.
        if k == 4:
            baffle = (radius < 155.0) | (
                (X[:, :, k] > 810.0) & (Y[:, :, k] > 650.0)
            )
            layer[baffle] = 4

        facies[:, :, k] = layer

    return {"facies": facies, "platform_radial_m": radial}


def fracture_geometry(grid: dict, cfg: Config) -> dict:
    """Identify corridor cells without yet changing petrophysical arrays."""
    X, Y = grid["X"][:, :, 0], grid["Y"][:, :, 0]
    mask = np.zeros(grid["depth"].shape, dtype=np.int16)
    definitions = []

    for number, fr in enumerate(cfg.fractures, start=1):
        theta = np.deg2rad(fr["strike_deg"])
        ux, uy = np.cos(theta), np.sin(theta)
        dx, dy = X - fr["x"], Y - fr["y"]

        along = ux * dx + uy * dy
        across = -uy * dx + ux * dy
        region = (
            (np.abs(along) <= fr["length_m"] / 2)
            & (np.abs(across) <= fr["width_m"] / 2)
        )

        for k in fr["layers"]:
            # First corridor wins at overlaps; avoid double multipliers.
            mask[:, :, k][region & (mask[:, :, k] == 0)] = number

        definitions.append({
            **fr,
            "id": number,
            "cell_count": int(np.count_nonzero(mask == number)),
            "representation": "equivalent_continuum",
        })

    return {"mask": mask, "definitions": definitions}


def make_diagenesis(
    grid: dict,
    cfg: Config,
    facies: np.ndarray,
    fracture_mask: np.ndarray,
    rng: np.random.Generator,
) -> dict:
    """Three explicit, independently correlated diagenetic intensity fields."""
    X, Y, D = grid["X"], grid["Y"], grid["depth"]
    z_diss = field(grid, cfg, rng, .80)
    z_cement = field(grid, cfg, rng, .65)
    z_dolo = field(grid, cfg, rng, 1.20)

    # Dissolution follows fractures and selected bedding horizons.
    horizon = np.exp(-((D - 2019.0) / 5.0) ** 2)
    dissolution = np.clip(
        .25 + .16 * z_diss
        + .38 * (fracture_mask > 0)
        + .16 * horizon,
        0, 1,
    )

    # Cementation localized near a platform margin and in heterogeneous
    # patches. Structural distribution is hypothetical.
    xc, yc = cfg.platform_center_xy
    radius = np.sqrt((X - xc) ** 2 + (Y - yc) ** 2)
    cementation = np.clip(
        .24 + .17 * z_cement
        + .23 * np.exp(-((radius - 365.0) / 65.0) ** 2),
        0, 1,
    )

    # Dolomitization focused on packstone/mudstone susceptible precursors.
    suitable = np.isin(facies, [0, 1, 2])
    dolomitization = np.clip(
        (.29 + .17 * z_dolo + .13 * (Y < 500.0)) * suitable,
        0, 1,
    )

    # Anhydrite is held comparatively tight rather than being converted to
    # reservoir dolomite.
    for arr in (dissolution, cementation, dolomitization):
        arr[facies == 4] *= .15

    dominant = np.zeros(facies.shape, dtype=np.int8)
    stack = np.stack(
        (dissolution, cementation, dolomitization), axis=0
    )
    idx = np.argmax(stack, axis=0)
    strong = np.max(stack, axis=0) >= .35
    dominant[strong] = idx[strong] + 1

    rock_facies = facies.copy()
    dolomite_cells = (
        suitable
        & (dolomitization >= .58)
        & (dolomitization > cementation)
    )
    rock_facies[dolomite_cells] = DOLOMITE_CODE

    return {
        "dissolution": dissolution,
        "cementation": cementation,
        "dolomitization": dolomitization,
        "dominant_type": dominant,
        "rock_facies": rock_facies,
    }


def make_pore_system(
    grid: dict,
    cfg: Config,
    facies: np.ndarray,
    diag: dict,
    fracture_mask: np.ndarray,
    rng: np.random.Generator,
) -> dict:
    """
    Allocate total pore volume among five matrix/vug pore classes plus
    explicit fracture porosity.

    DPDP bookkeeping convention:
      phi_matrix = interparticle + intraparticle + moldic + micropore
      phi_fracture = vuggy + fracture
      phi = phi_matrix + phi_fracture

    This accounting does NOT assert that every vug is connected or that a
    dual-continuum flow formulation is already active.
    """
    props = facies_properties()
    Z = field(grid, cfg, rng, 1.0)
    d = diag["dissolution"]
    c = diag["cementation"]
    dol = diag["dolomitization"]

    phi_base = np.zeros(facies.shape)
    pore_weights = np.zeros((5,) + facies.shape)

    for code, p in props.items():
        m = facies == code
        # Dolomitization may increase OR decrease porosity, depending on
        # precursor and locally varying replacement/precipitation balance.
        dol_sign = np.where(code in (0, 1), 1.0, -.35)
        raw = (
            p.phi_mean
            + p.phi_std * Z[m]
            + .030 * d[m]
            - .040 * c[m]
            + .028 * dol_sign * dol[m]
        )
        phi_base[m] = raw
        for pore_id, weight in enumerate(p.pore_weights):
            pore_weights[pore_id][m] = weight

    # Dissolution makes moldic/vuggy pores; cementation suppresses
    # interparticle pathways; replacement changes micropore development.
    pore_weights[0] *= np.maximum(.15, 1 - .43 * c + .15 * dol)
    pore_weights[1] *= 1 + .10 * dol
    pore_weights[2] *= 1 + 1.15 * d
    pore_weights[3] *= 1 + 1.45 * d
    pore_weights[4] *= 1 + .32 * c + .10 * dol
    pore_weights /= pore_weights.sum(axis=0)

    # A small corridor fracture pore volume is separate from vuggy volume.
    fracture_pore_volume = (
        .0012 * (fracture_mask > 0) * (facies != 4)
    )

    phi = np.empty(facies.shape)
    for code, p in props.items():
        m = facies == code
        phi[m] = np.clip(
            phi_base[m] + fracture_pore_volume[m],
            p.phi_min, p.phi_max
        )

    # Allocate remaining volume to non-fracture pores. Clipping cannot
    # violate phi = phi_matrix + phi_fracture.
    nonfracture_phi = phi - fracture_pore_volume
    pore_components = pore_weights * nonfracture_phi[None, ...]
    dominant_pore = np.argmax(pore_components, axis=0).astype(np.int8)

    # Explicitly label fractures as dominant where they are hydraulically
    # important; dominant_pore is a hydraulic label, not largest pore volume.
    dominant_pore[
        (fracture_mask > 0) & (facies != 4)
    ] = 5

    phi_matrix = (
        pore_components[0] + pore_components[1]
        + pore_components[2] + pore_components[4]
    )
    phi_fracture = pore_components[3] + fracture_pore_volume

    # Some vugs are isolated: connectivity is distinct from their volume.
    Zv = field(grid, cfg, rng, .65)
    vug_connectivity = np.clip(
        .35 + .23 * Zv + .17 * d, .02, .98
    )

    return {
        "phi": phi,
        "phi_matrix": phi_matrix,
        "phi_fracture": phi_fracture,
        "pore_components": pore_components,
        "fracture_pore_volume": fracture_pore_volume,
        "dominant_pore_type": dominant_pore,
        "vug_connectivity": vug_connectivity,
    }


def make_permeability(
    grid: dict,
    cfg: Config,
    facies: np.ndarray,
    pores: dict,
    diag: dict,
    fracture_mask: np.ndarray,
    rng: np.random.Generator,
) -> dict:
    """
    Permeability is primarily facies/pore-connectivity/diagenesis/stochastic
    controlled, with only a deliberately modest porosity term.

    Fracture continuum permeability is stored separately from matrix k.
    """
    props = facies_properties()
    Zk = field(grid, cfg, rng, .80)
    Zanis = field(grid, cfg, rng, .50)
    components = pores["pore_components"]
    phi = pores["phi"]
    denom = np.maximum(
        components.sum(axis=0), 1e-10
    )

    inter = components[0] / denom
    mold = components[2] / denom
    vug = components[3] / denom
    micro = components[4] / denom
    connected_vug = vug * pores["vug_connectivity"]

    log_k = np.zeros(facies.shape)
    kz_kx = np.zeros(facies.shape)

    for code, p in props.items():
        m = facies == code
        phi_scaled = (phi[m] - p.phi_mean) / max(p.phi_std, .01)

        # Independent Zk dominates the small direct phi contribution.
        log_k[m] = (
            p.log10_k_mean
            + p.log10_k_scatter * Zk[m]
            + .10 * phi_scaled
            + .62 * (inter[m] - .25)
            + .67 * (connected_vug[m] - .05)
            + .27 * mold[m]
            - .72 * (micro[m] - .20)
            + .45 * diag["dissolution"][m]
            - .85 * diag["cementation"][m]
            + .12 * diag["dolomitization"][m]
        )
        kz_kx[m] = np.clip(
            p.kv_kh * 10 ** (.20 * Zanis[m]), .001, .65
        )

    kx_matrix = np.zeros(facies.shape)
    for code, p in props.items():
        m = facies == code
        kx_matrix[m] = np.clip(
            10 ** log_k[m], p.k_min_md, p.k_max_md
        )

    ky_matrix = .82 * kx_matrix
    kz_matrix = kx_matrix * kz_kx

    kx_fracture = np.zeros(facies.shape)
    ky_fracture = np.zeros(facies.shape)
    kz_fracture = np.zeros(facies.shape)

    for fr in cfg.fractures:
        m = fracture_mask == next(
            definition_id
            for definition_id, definition in enumerate(
                cfg.fractures, start=1
            )
            if definition["name"] == fr["name"]
        )
        # Do not make anhydrite conductive simply by crossing it.
        m &= facies != 4
        th = np.deg2rad(fr["strike_deg"])
        along, across = fr["k_along_multiplier"], fr["k_across_multiplier"]
        mx = along * np.cos(th) ** 2 + across * np.sin(th) ** 2
        my = along * np.sin(th) ** 2 + across * np.cos(th) ** 2

        kx_fracture[m] = kx_matrix[m] * (mx - 1)
        ky_fracture[m] = ky_matrix[m] * (my - 1)
        kz_fracture[m] = kz_matrix[m] * (
            fr["k_vertical_multiplier"] - 1
        )

    # Single-continuum EFFECTIVE directional k, suitable for the initial
    # TPFA model. DPDP later needs its own transfer law and discretization.
    return {
        "kx_matrix": kx_matrix,
        "ky_matrix": ky_matrix,
        "kz_matrix": kz_matrix,
        "kx_fracture": kx_fracture,
        "ky_fracture": ky_fracture,
        "kz_fracture": kz_fracture,
        "kx": kx_matrix + kx_fracture,
        "ky": ky_matrix + ky_fracture,
        "kz": kz_matrix + kz_fracture,
    }


def fault_signed_side(
    fd: dict, X: np.ndarray, Y: np.ndarray, D: np.ndarray
) -> np.ndarray:
    """Implicit curved fault surface. Zero is the fault."""
    y_offset = Y - fd["y0"]
    fault_x = (
        fd["x0"]
        + fd["slope_y"] * y_offset
        + fd["slope_depth"] * (D - fd["D0"])
        + fd["curvature"] * y_offset**2
    )
    return X - fault_x


def make_faults(grid: dict, cfg: Config) -> dict:
    shape = grid["depth"].shape
    cell_mask = np.zeros(shape, dtype=np.int16)
    face_masks = {
        "x": np.zeros((cfg.Nx - 1, cfg.Ny, cfg.Nz), dtype=np.int16),
        "y": np.zeros((cfg.Nx, cfg.Ny - 1, cfg.Nz), dtype=np.int16),
        "z": np.zeros((cfg.Nx, cfg.Ny, cfg.Nz - 1), dtype=np.int16),
    }
    mult = {direction: np.ones_like(face_masks[direction], dtype=float)
            for direction in ("x", "y", "z")}
    definitions = []

    for number, fd in enumerate(cfg.faults, start=1):
        side = fault_signed_side(
            fd, grid["X"], grid["Y"], grid["depth"]
        )
        cell_mask[np.abs(side) <= fd["zone_width_m"] / 2] = number

        # A face is intersected when its adjacent cell centres are on
        # opposite sides. The face multiplier does not modify matrix k.
        pairs = {
            "x": (side[:-1, :, :], side[1:, :, :]),
            "y": (side[:, :-1, :], side[:, 1:, :]),
            "z": (side[:, :, :-1], side[:, :, 1:]),
        }
        n_faces = 0
        for direction, (a, b) in pairs.items():
            cross = ((a <= 0) & (b > 0)) | ((a > 0) & (b <= 0))
            face_masks[direction][cross] = number
            mult[direction][cross] *= fd[
                "transmissibility_multiplier"
            ]
            n_faces += int(cross.sum())

        definitions.append({
            **fd, "id": number,
            "cell_count": int(np.count_nonzero(cell_mask == number)),
            "face_count": n_faces,
            "equation": (
                "x - [x0 + slope_y*(y-y0) "
                "+ slope_depth*(D-D0) + curvature*(y-y0)^2] = 0"
            ),
        })

    return {
        "cell_mask": cell_mask,
        "face_mask_x": face_masks["x"],
        "face_mask_y": face_masks["y"],
        "face_mask_z": face_masks["z"],
        "trans_mult_x": mult["x"],
        "trans_mult_y": mult["y"],
        "trans_mult_z": mult["z"],
        "definitions": definitions,
    }


def make_initial_state(
    grid: dict,
    cfg: Config,
    facies: np.ndarray,
    pores: dict,
) -> dict:
    D = grid["depth"]

    # Water pressure is the primary reference field throughout the grid.
    Pw = (
        cfg.pressure_reference_pa
        + cfg.rho_water_kg_m3 * G
        * (D - cfg.pressure_reference_depth_m)
    )

    Pw_owc = (
        cfg.pressure_reference_pa
        + cfg.rho_water_kg_m3 * G
        * (cfg.owc_depth_m - cfg.pressure_reference_depth_m)
    )
    # Hydrostatic oil pressure on the oil side. Values below OWC are
    # hypothetical extensions, NOT pressures of an existing oil phase.
    Po = (
        Pw_owc
        + cfg.rho_oil_kg_m3 * G * (D - cfg.owc_depth_m)
    )

    components = pores["pore_components"]
    micropore_fraction = components[4] / np.maximum(
        components.sum(axis=0), 1e-10
    )
    swc = np.zeros(D.shape)

    for code, fp in facies_properties().items():
        m = facies == code
        swc[m] = np.clip(
            fp.swc_base
            + .16 * (micropore_fraction[m] - .20),
            .22, .58,
        )

    swc[facies == 4] = 1.0

    # Pc = Po-Pw above OWC = (rho_w-rho_o)*g*height above OWC.
    # This exponential Pc-Sw surrogate gives Sw=1 at OWC and approaches
    # local Swc farther above it. It is illustrative, not a measured curve.
    height = np.maximum(cfg.owc_depth_m - D, 0)
    pc = (
        (cfg.rho_water_kg_m3 - cfg.rho_oil_kg_m3)
        * G * height
    )
    pc_scale = (
        (cfg.rho_water_kg_m3 - cfg.rho_oil_kg_m3)
        * G * cfg.saturation_transition_m
    )
    Sw = swc + (1 - swc) * np.exp(-pc / pc_scale)
    Sw[D >= cfg.owc_depth_m] = 1.0
    Sw[facies == 4] = 1.0

    So = 1.0 - Sw
    Sg = np.zeros_like(Sw)

    return {
        "pressure": Pw,
        "pressure_water": Pw,
        "pressure_oil_hydrostatic_extension": Po,
        "capillary_pressure_oil_water": pc,
        "Sw": Sw,
        "So": So,
        "Sg": Sg,
        "swc": swc,
    }


def make_wells(
    grid: dict,
    cfg: Config,
    facies: np.ndarray,
    diag: dict,
    permeability: dict,
    state: dict,
) -> List[dict]:
    result = []
    props = facies_properties()

    for spec in cfg.wells:
        i = int(spec["x"] / grid["dx"])
        j = int(spec["y"] / grid["dy"])

        local_top = float(grid["top_depth"][i, j])
        oil_column = max(cfg.owc_depth_m - local_top, 0.0)
        deepest_completion = min(
            local_top
            + cfg.maximum_completed_oil_column_fraction * oil_column,
            cfg.owc_depth_m - cfg.min_owc_clearance_m,
        )

        perf = []
        for k in range(cfg.Nz):
            code = int(facies[i, j, k])
            D = float(grid["depth"][i, j, k])

            if not props[code].net_reservoir:
                continue
            if D > deepest_completion:
                continue
            if state["So"][i, j, k] <= .5:
                continue
            if diag["cementation"][i, j, k] >= .65:
                continue
            if permeability["kx_matrix"][i, j, k] < .05:
                continue

            perf.append(k)

        if not perf:
            raise ValueError(
                f"No valid perforations for {spec['name']} at "
                f"({spec['x']}, {spec['y']}). Move the well or relax "
                "explicit completion criteria."
            )

        result.append({
            **spec,
            "i": i, "j": j,
            "cell_x": float(grid["x"][i]),
            "cell_y": float(grid["y"][j]),
            "perforated_layers": perf,
            "perforation_depths_m": [
                float(grid["depth"][i, j, k]) for k in perf
            ],
            "perforated_facies": [FACIES[int(facies[i, j, k])]
                                  for k in perf],
            "local_oil_column_m": oil_column,
            "maximum_perforation_depth_m": deepest_completion,
            "geometry": "vertical",
        })

    return result


def make_metadata(cfg: Config, grid: dict) -> dict:
    props = facies_properties()

    # These are INITIALIZATION parameters to be replaced with measured
    # carbonate-specific SCAL/Pc curves before predictive simulation.
    flow_parameters = {
        "relative_permeability_model": "illustrative_Corey_by_rock_type",
        "description": (
            "Stored for future use only; not applied to initial-state "
            "calculations or a flow solver."
        ),
        "by_depositional_facies": {
            FACIES[code]: {
                "Swc_nominal": p.swc_base,
                "Sorw": .32 if code in (1, 2) else .38,
                "water_exponent": 3.0 if code in (1, 2) else 4.0,
                "oil_exponent": 2.5 if code in (1, 2) else 3.2,
                "krw_endpoint": .35 if code in (1, 2) else .16,
                "kro_endpoint": .80 if code in (1, 2) else .55,
            }
            for code, p in props.items()
        },
        "capillary_pressure": {
            "model": "initialization_exponential_height_function",
            "formula": (
                "Sw=Swc+(1-Swc)*exp[-Pc/Pc_scale] above OWC; "
                "Pc=(rho_w-rho_o)*g*(OWC-D); Sw=1 below OWC"
            ),
            "transition_scale_m": cfg.saturation_transition_m,
            "warning": (
                "Illustrative initial saturation profile. Replace with "
                "measured, wettability-aware Pc/SWOF data for simulation."
            ),
        },
    }

    return {
        "disclaimer": (
            "Synthetic research example only. No real-field calibration "
            "or geological interpretation is claimed. All ranges "
            "and controls are illustrative."
        ),
        "grid": {
            "Nx": cfg.Nx, "Ny": cfg.Ny, "Nz": cfg.Nz,
            "Lx_m": cfg.Lx, "Ly_m": cfg.Ly, "Lz_m": cfg.Lz,
            "dx_m": grid["dx"], "dy_m": grid["dy"],
            "dz_m": grid["dz"],
            "array_order": "(Nx,Ny,Nz), C order, indexing='ij'",
            "linear_cell_index": "(i*Ny+j)*Nz+k",
            "depth_convention": "positive downward, k=0 uppermost",
        },
        "seed": cfg.seed,
        "facies_definitions": {
            str(code): {
                "name": FACIES[code],
                **asdict(p),
            }
            for code, p in facies_properties().items()
        },
        "dolomitized_rock_facies_code": DOLOMITE_CODE,
        "pore_type_definitions": PORE_TYPES,
        "dominant_pore_classification": (
            "Largest non-fracture pore-volume component, except a "
            "permeable fracture corridor is hydraulically flagged "
            "as code 5. Vug connectivity is stored separately."
        ),
        "diagenesis_types": DIAGENESIS_TYPES,
        "diagenetic_parameters": {
            "intensity_range": [0, 1],
            "porosity_change": (
                "+0.030*dissolution -0.040*cementation "
                "+0.028*dolomite_sign*dolomitization"
            ),
            "dolomite_sign": (
                "positive for mudstone/packstone precursors; "
                "negative for grainstone in this synthetic example"
            ),
        },
        "property_ranges_note": (
            "Illustrative facies-dependent porosity and matrix-k bounds "
            "are listed in facies_definitions. Effective fracture-cell "
            "permeability can exceed matrix-k bounds."
        ),
        "geometry": {
            "platform": "rimmed platform with lagoon, shoal and reef margin",
            "top_depth_m": cfg.top_depth_m,
            "regional_dip_x": cfg.dip_x,
            "regional_dip_y": cfg.dip_y,
            "dome_amplitude_m": cfg.dome_amplitude_m,
            "dome_radius_m": cfg.dome_radius_m,
        },
        "heterogeneity": {
            "lateral_correlation_m": cfg.lateral_correlation_m,
            "vertical_correlation_m": cfg.vertical_correlation_m,
            "spectrum": cfg.random_spectrum,
            "fractal_beta": cfg.fractal_beta,
        },
        "initial_fluids": {
            "owc_depth_m": cfg.owc_depth_m,
            "pressure_reference_pa": cfg.pressure_reference_pa,
            "pressure_reference_depth_m": cfg.pressure_reference_depth_m,
            "rho_water_kg_m3": cfg.rho_water_kg_m3,
            "rho_oil_kg_m3": cfg.rho_oil_kg_m3,
            "pressure_primary": (
                "hydrostatic water pressure Pw; Po is separately stored "
                "as an oil hydrostatic extension"
            ),
            "formation_compressibility_pa_inv":
                cfg.formation_compressibility_pa_inv,
        },
        "flow_parameters": flow_parameters,
        "dpdp": {
            "enabled": False,
            "phi_matrix_definition": (
                "interparticle+intraparticle+moldic+microporosity"
            ),
            "phi_fracture_definition": (
                "vuggy+explicit_fracture_pore_volume"
            ),
            "transfer_shape_factor_units": "m^-2",
            "transfer_shape_factor_status": (
                "NaN where not characterized; zero in impermeable "
                "evaporite. MUST be calibrated before enabling DPDP."
            ),
            "note": (
                "Pore volume is partitioned without double counting. "
                "Separate fracture k is an excess effective-continuum "
                "contribution, not a validated DPDP fracture tensor."
            ),
        },
        "permeability_units": "mD",
        "md_to_m2": MD_TO_M2,
        "fault_representation": (
            "Explicit implicit surfaces and per-internal-face "
            "transmissibility multipliers. No throw/juxtaposition."
        ),
        "fracture_representation": (
            "Equivalent continuum with anisotropic axis-aligned "
            "multipliers; NOT DFM/EDFM."
        ),
        "wells_completion_rule": {
            "net_facies_only": True,
            "maximum_fraction_of_local_oil_column":
                cfg.maximum_completed_oil_column_fraction,
            "minimum_clearance_above_owc_m":
                cfg.min_owc_clearance_m,
            "maximum_cementation_intensity": .65,
            "minimum_matrix_kx_md": .05,
        },
    }


def build_reservoir(cfg: Optional[Config] = None) -> dict:
    cfg = cfg or Config()
    check_config(cfg)
    rng = np.random.default_rng(cfg.seed)

    grid = make_grid(cfg)
    deposition = make_depositional_facies(grid, cfg, rng)
    facies = deposition["facies"]
    fractures = fracture_geometry(grid, cfg)
    diagenesis = make_diagenesis(
        grid, cfg, facies, fractures["mask"], rng
    )
    pores = make_pore_system(
        grid, cfg, facies, diagenesis, fractures["mask"], rng
    )
    permeability = make_permeability(
        grid, cfg, facies, pores, diagenesis,
        fractures["mask"], rng
    )
    faults = make_faults(grid, cfg)
    initial_state = make_initial_state(
        grid, cfg, facies, pores
    )
    wells = make_wells(
        grid, cfg, facies, diagenesis,
        permeability, initial_state
    )

    # Placeholder, NOT a claimed calibrated matrix-fracture transfer law.
    shape_factor = np.full(facies.shape, np.nan)
    shape_factor[facies == 4] = 0.0

    return {
        "config": cfg,
        "grid": grid,
        "facies": facies,
        "rock_facies": diagenesis["rock_facies"],
        "platform_radial_m": deposition["platform_radial_m"],
        "diagenesis": diagenesis,
        "pore_type": pores["dominant_pore_type"],
        "pore_components": pores["pore_components"],
        "vug_connectivity": pores["vug_connectivity"],
        "phi": pores["phi"],
        "phi_matrix": pores["phi_matrix"],
        "phi_fracture": pores["phi_fracture"],
        "fracture_pore_volume": pores["fracture_pore_volume"],
        **permeability,
        "shape_factor": shape_factor,
        "faults": faults,
        "fractures": fractures,
        "initial_state": initial_state,
        "wells": wells,
        "metadata": make_metadata(cfg, grid),
    }


def logk_phi_r2(
    phi: np.ndarray, kx_matrix: np.ndarray, mask: np.ndarray
) -> float:
    """R² of a *linear* phi -> log10(matrix kx) fit on selected cells."""
    x = phi[mask].astype(float)
    y = np.log10(kx_matrix[mask])
    if x.size < 3 or x.std() < 1e-12 or y.std() < 1e-12:
        return float("nan")
    return float(np.corrcoef(x, y)[0, 1] ** 2)


def validate_reservoir(res: dict) -> dict:
    cfg = res["config"]
    g = res["grid"]
    fac = res["facies"]
    p = facies_properties()
    D = g["depth"]
    st = res["initial_state"]
    errors = []

    def require(condition: bool, message: str) -> None:
        if not condition:
            errors.append(message)

    shape = (cfg.Nx, cfg.Ny, cfg.Nz)
    for name in (
        "facies", "phi", "phi_matrix", "phi_fracture",
        "kx", "ky", "kz",
        "kx_matrix", "ky_matrix", "kz_matrix",
        "kx_fracture", "ky_fracture", "kz_fracture",
        "pore_type",
    ):
        require(res[name].shape == shape, f"{name}: wrong shape")

    for code, fp in p.items():
        mask = fac == code
        if np.any(mask):
            require(
                bool(np.all(
                    (res["phi"][mask] >= fp.phi_min - 1e-10)
                    & (res["phi"][mask] <= fp.phi_max + 1e-10)
                )),
                f"Facies {FACIES[code]} has porosity outside its bounds",
            )
            require(
                bool(np.all(
                    (res["kx_matrix"][mask] >= fp.k_min_md - 1e-12)
                    & (res["kx_matrix"][mask] <= fp.k_max_md + 1e-12)
                )),
                f"Facies {FACIES[code]} has matrix kx outside its bounds",
            )

    require(
        bool(np.allclose(
            res["phi"], res["phi_matrix"] + res["phi_fracture"],
            atol=1e-10,
        )),
        "phi != phi_matrix + phi_fracture",
    )
    require(
        bool(np.allclose(
            res["phi"],
            res["pore_components"].sum(axis=0)
            + res["fracture_pore_volume"],
            atol=1e-10,
        )),
        "Pore-type volumes do not sum to total phi",
    )

    for name in (
        "kx", "ky", "kz",
        "kx_matrix", "ky_matrix", "kz_matrix",
    ):
        require(
            bool(np.all(np.isfinite(res[name]) & (res[name] > 0))),
            f"{name} is not strictly positive and finite",
        )

    require(
        bool(np.all(res["kz_matrix"] <= res["kx_matrix"])),
        "Matrix kz exceeds matrix kx",
    )

    for name in ("Sw", "So", "Sg"):
        require(
            bool(np.all(
                np.isfinite(st[name])
                & (st[name] >= 0)
                & (st[name] <= 1)
            )),
            f"Invalid saturation {name}",
        )
    require(
        bool(np.allclose(st["Sw"] + st["So"] + st["Sg"], 1)),
        "Saturations do not sum to 1",
    )
    require(
        bool(np.all(st["So"][D >= cfg.owc_depth_m] == 0)),
        "Oil exists below the OWC",
    )
    require(
        bool(np.all(st["Sw"][fac == 4] == 1)),
        "Evaporite must be non-oil-bearing in this model",
    )

    require(
        bool(np.all(np.diff(D, axis=2) > 0)),
        "Depth must increase with k",
    )
    require(
        bool(np.all(np.diff(st["pressure"], axis=2) > 0)),
        "Pressure must increase downward",
    )
    require(
        bool(np.allclose(
            np.diff(st["pressure"], axis=2)
            / np.diff(D, axis=2),
            cfg.rho_water_kg_m3 * G,
        )),
        "Primary water pressure is not hydrostatic",
    )

    for name in ("dissolution", "cementation", "dolomitization"):
        a = res["diagenesis"][name]
        require(
            bool(np.all(np.isfinite(a) & (a >= 0) & (a <= 1))),
            f"Invalid diagenesis intensity: {name}",
        )

    for w in res["wells"]:
        i, j = w["i"], w["j"]
        require(
            0 <= i < cfg.Nx and 0 <= j < cfg.Ny,
            f"{w['name']} outside grid",
        )
        require(
            len(w["perforated_layers"]) > 0,
            f"{w['name']} has no perforations",
        )
        for k in w["perforated_layers"]:
            require(0 <= k < cfg.Nz, f"{w['name']} invalid k={k}")
            if 0 <= k < cfg.Nz:
                require(
                    p[int(fac[i, j, k])].net_reservoir,
                    f"{w['name']} perforates non-net facies",
                )
                require(
                    D[i, j, k] <= (
                        cfg.owc_depth_m
                        - cfg.min_owc_clearance_m
                        + 1e-9
                    ),
                    f"{w['name']} too close to OWC",
                )
                require(
                    D[i, j, k]
                    <= w["maximum_perforation_depth_m"] + 1e-9,
                    f"{w['name']} exceeds upper oil-column completion",
                )
                require(
                    res["diagenesis"]["cementation"][i, j, k] < .65,
                    f"{w['name']} perforates heavily cemented cell",
                )

    faults = res["faults"]
    expected_faces = {
        "x": (cfg.Nx - 1, cfg.Ny, cfg.Nz),
        "y": (cfg.Nx, cfg.Ny - 1, cfg.Nz),
        "z": (cfg.Nx, cfg.Ny, cfg.Nz - 1),
    }
    for direction, expected in expected_faces.items():
        fm = faults[f"face_mask_{direction}"]
        tm = faults[f"trans_mult_{direction}"]
        require(fm.shape == expected, f"Fault {direction}-face shape")
        require(tm.shape == expected, f"Fault multiplier {direction}-face shape")
        require(
            bool(np.all(np.isfinite(tm) & (tm > 0))),
            f"Invalid fault {direction}-face multipliers",
        )
    for fd in faults["definitions"]:
        require(
            fd["face_count"] > 0,
            f"{fd['name']} intersects no internal faces",
        )

    require(
        res["fractures"]["mask"].shape == shape,
        "Fracture mask shape incorrect",
    )
    for fr in res["fractures"]["definitions"]:
        require(
            fr["cell_count"] > 0,
            f"{fr['name']} intersects no cells",
        )

    require(
        bool(np.all(
            res["kx_fracture"][
                res["fractures"]["mask"] == 0
            ] == 0
        )),
        "Fracture permeability present outside corridors",
    )

    # Report both. Excluding seals is particularly informative because
    # mixing seals and reservoir lithologies can create spurious correlation.
    net = np.isin(fac, [1, 2, 3])
    all_cells = np.ones(shape, dtype=bool)
    r2_net = logk_phi_r2(res["phi"], res["kx_matrix"], net)
    r2_all = logk_phi_r2(res["phi"], res["kx_matrix"], all_cells)

    require(
        np.isfinite(r2_net) and r2_net < .5,
        f"Net-reservoir phi/log10(k) R² is not weak: {r2_net:.3f}",
    )
    require(
        np.isfinite(r2_all) and r2_all < .5,
        f"Whole-model phi/log10(k) R² is not weak: {r2_all:.3f}",
    )

    if errors:
        raise AssertionError(
            "Reservoir validation failed:\n- " + "\n- ".join(errors)
        )

    return {
        "phi_log10_matrix_kx_r2_net": r2_net,
        "phi_log10_matrix_kx_r2_all_cells": r2_all,
        "total_cells": int(np.prod(shape)),
        "fault_faces": sum(
            fd["face_count"] for fd in faults["definitions"]
        ),
        "fracture_cells": int(
            np.count_nonzero(res["fractures"]["mask"])
        ),
        "wells": len(res["wells"]),
    }


def save_reservoir(res: dict, output_dir: str) -> Tuple[Path, Path]:
    folder = Path(output_dir)
    folder.mkdir(parents=True, exist_ok=True)
    npz_path = folder / "carbonate_reservoir.npz"
    json_path = folder / "carbonate_reservoir.json"
    g = res["grid"]

    arrays = {
        "x": g["x"],
        "y": g["y"],
        "depth": g["depth"],
        "top_depth": g["top_depth"],
        "cell_volume": g["cell_volume"],
        "facies": res["facies"],
        "rock_facies": res["rock_facies"],
        "pore_type": res["pore_type"],
        "pore_components": res["pore_components"],
        "vug_connectivity": res["vug_connectivity"],
        "phi": res["phi"],
        "phi_matrix": res["phi_matrix"],
        "phi_fracture": res["phi_fracture"],
        "fracture_pore_volume": res["fracture_pore_volume"],
        "shape_factor": res["shape_factor"],
    }

    for key in (
        "kx", "ky", "kz",
        "kx_matrix", "ky_matrix", "kz_matrix",
        "kx_fracture", "ky_fracture", "kz_fracture",
    ):
        arrays[key] = res[key]

    for key in (
        "dissolution", "cementation", "dolomitization",
        "dominant_type",
    ):
        arrays["diagenesis_" + key] = res["diagenesis"][key]

    for key in (
        "cell_mask",
        "face_mask_x", "face_mask_y", "face_mask_z",
        "trans_mult_x", "trans_mult_y", "trans_mult_z",
    ):
        arrays["fault_" + key] = res["faults"][key]

    arrays["fracture_mask"] = res["fractures"]["mask"]

    for key, arr in res["initial_state"].items():
        arrays["initial_" + key] = arr

    np.savez_compressed(npz_path, **arrays)

    metadata = {
        **res["metadata"],
        "faults": res["faults"]["definitions"],
        "fractures": res["fractures"]["definitions"],
        "wells": res["wells"],
        "validation": validate_reservoir(res),
        "observed_property_ranges": {
            "phi": [float(res["phi"].min()), float(res["phi"].max())],
            "kx_matrix_md": [
                float(res["kx_matrix"].min()),
                float(res["kx_matrix"].max()),
            ],
            "kx_effective_md": [
                float(res["kx"].min()),
                float(res["kx"].max()),
            ],
        },
    }
    with json_path.open("w", encoding="utf-8") as handle:
        json.dump(metadata, handle, indent=2)

    return npz_path, json_path


def load_saved_reservoir(output_dir: str) -> Tuple[dict, dict]:
    """
    Read arrays and metadata without regenerating the model.
    The returned flat arrays and metadata are sufficient for later solver IO.
    """
    folder = Path(output_dir)
    with np.load(folder / "carbonate_reservoir.npz") as file:
        arrays = {key: file[key] for key in file.files}
    with (folder / "carbonate_reservoir.json").open(
        encoding="utf-8"
    ) as handle:
        metadata = json.load(handle)
    return arrays, metadata


# ---------------------------------------------------------------------------
# Interactive visualization
# ---------------------------------------------------------------------------

def plotly_go():
    try:
        import plotly.graph_objects as go
    except ImportError as exc:
        raise ImportError(
            "Interactive visualizations require Plotly. "
            "Install it with: pip install plotly"
        ) from exc
    return go


def scene_layout(title: str, grid: dict) -> dict:
    """
    Plotly scales physical x/y/z dimensions very differently for a thin
    reservoir. z is exaggerated for legibility; hover displays true depth.
    """
    return {
        "title": title,
        "scene": {
            "xaxis": {"title": "x [m]", "range": [0, grid["Lx"]]},
            "yaxis": {"title": "y [m]", "range": [0, grid["Ly"]]},
            "zaxis": {
                "title": "depth [m], positive down",
                "range": [
                    float(grid["depth"].max() + 3),
                    float(grid["depth"].min() - 3),
                ],
            },
            "aspectmode": "manual",
            "aspectratio": {"x": 1, "y": 1, "z": .38},
        },
        "margin": {"l": 0, "r": 0, "b": 0, "t": 65},
        "height": 750,
    }


def add_well_traces(fig, res: dict) -> None:
    go = plotly_go()
    g = res["grid"]

    for well in res["wells"]:
        i, j = well["i"], well["j"]
        color = "#2667d7" if well["kind"] == "injector" else "#dc3535"
        x, y = well["cell_x"], well["cell_y"]
        top = float(g["depth"][i, j, 0] - g["dz"] / 2)
        bottom = float(g["depth"][i, j, -1] + g["dz"] / 2)

        fig.add_trace(go.Scatter3d(
            x=[x, x], y=[y, y], z=[top, bottom],
            mode="lines",
            line={"color": color, "width": 8},
            name=well["name"],
            hovertemplate=(
                f"{well['name']} ({well['kind']})"
                "<br>depth=%{z:.1f} m<extra></extra>"
            ),
        ))
        fig.add_trace(go.Scatter3d(
            x=[x] * len(well["perforated_layers"]),
            y=[y] * len(well["perforated_layers"]),
            z=well["perforation_depths_m"],
            mode="markers",
            marker={
                "color": color, "size": 7, "symbol": "diamond",
                "line": {"color": "black", "width": 2},
            },
            name=well["name"] + " perforations",
            hovertemplate=(
                f"{well['name']} perforation"
                "<br>depth=%{z:.1f} m<extra></extra>"
            ),
            showlegend=False,
        ))


def add_owc_trace(fig, res: dict) -> None:
    go = plotly_go()
    g = res["grid"]
    cfg = res["config"]

    xx, yy = np.meshgrid(
        np.linspace(0, g["Lx"], 5),
        np.linspace(0, g["Ly"], 5),
    )
    fig.add_trace(go.Surface(
        x=xx,
        y=yy,
        z=np.full_like(xx, cfg.owc_depth_m),
        surfacecolor=np.zeros_like(xx),
        colorscale=[[0, "#31c8de"], [1, "#31c8de"]],
        showscale=False,
        opacity=.19,
        name="OWC",
        hovertemplate=(
            f"OWC: {cfg.owc_depth_m:.1f} m<extra></extra>"
        ),
    ))


def add_fault_surface(fig, res: dict) -> None:
    go = plotly_go()
    g = res["grid"]

    for fd in res["faults"]["definitions"]:
        yy, dd = np.meshgrid(
            np.linspace(0, g["Ly"], 45),
            np.linspace(
                float(g["depth"].min()) - 2,
                float(g["depth"].max()) + 2,
                16,
            ),
        )
        offset = yy - fd["y0"]
        xx = (
            fd["x0"]
            + fd["slope_y"] * offset
            + fd["slope_depth"] * (dd - fd["D0"])
            + fd["curvature"] * offset**2
        )
        xx[(xx < 0) | (xx > g["Lx"])] = np.nan

        fig.add_trace(go.Surface(
            x=xx, y=yy, z=dd,
            surfacecolor=np.zeros_like(dd),
            colorscale=[[0, "#222222"], [1, "#222222"]],
            opacity=.42,
            showscale=False,
            name=fd["name"] + " fault surface",
            hovertemplate=(
                fd["name"]
                f"<br>face multiplier: "
                f"{fd['transmissibility_multiplier']}"
                "<extra></extra>"
            ),
        ))


def add_fracture_points(fig, res: dict) -> None:
    go = plotly_go()
    g = res["grid"]
    mask = res["fractures"]["mask"]

    for fr in res["fractures"]["definitions"]:
        m = mask == fr["id"]
        fig.add_trace(go.Scatter3d(
            x=g["X"][m],
            y=g["Y"][m],
            z=g["depth"][m],
            mode="markers",
            marker={"size": 5, "color": "#ed22bf", "opacity": .9},
            name=fr["name"] + " corridor",
            hovertemplate=(
                fr["name"]
                "<br>x=%{x:.1f} m"
                "<br>y=%{y:.1f} m"
                "<br>depth=%{z:.1f} m"
                "<extra></extra>"
            ),
        ))


def make_cutaway_figure(
    res: dict,
    values: np.ndarray,
    labels: dict,
    colors: dict,
    title: str,
    show_fault: bool = True,
    show_fractures: bool = True,
):
    """
    Interactive 3D cell-centre view.

    Slider 'first visible layer' removes all shallower cells to expose the
    reservoir interior. Individual classes can also be toggled via legend.
    """
    go = plotly_go()
    g = res["grid"]
    fig = go.Figure()
    grouped_indices = []

    for k in range(g["Nz"]):
        layer_trace_ids = []
        for code, label in labels.items():
            m = values[:, :, k] == code
            if not np.any(m):
                continue

            trace_index = len(fig.data)
            layer_trace_ids.append(trace_index)

            fig.add_trace(go.Scatter3d(
                x=g["X"][:, :, k][m],
                y=g["Y"][:, :, k][m],
                z=g["depth"][:, :, k][m],
                mode="markers",
                marker={
                    "size": 3,
                    "color": colors[code],
                    "opacity": .68,
                },
                name=label,
                legendgroup=label,
                showlegend=(k == 0),
                hovertemplate=(
                    f"{label}"
                    "<br>x=%{x:.1f} m"
                    "<br>y=%{y:.1f} m"
                    "<br>depth=%{z:.1f} m"
                    f"<br>layer={k}<extra></extra>"
                ),
            ))
        grouped_indices.append(layer_trace_ids)

    overlay_start = len(fig.data)
    if show_fault:
        add_fault_surface(fig, res)
    if show_fractures:
        add_fracture_points(fig, res)
    add_well_traces(fig, res)
    add_owc_trace(fig, res)

    steps = []
    for first_visible in range(g["Nz"]):
        visibility = [True] * len(fig.data)
        for k in range(g["Nz"]):
            for idx in grouped_indices[k]:
                visibility[idx] = k >= first_visible

        # All overlays remain visible while cutting away cells.
        for idx in range(overlay_start, len(fig.data)):
            visibility[idx] = True

        steps.append({
            "method": "update",
            "label": str(first_visible),
            "args": [
                {"visible": visibility},
                {"title": (
                    title
                    + f" — first visible layer k={first_visible}"
                )},
            ],
        })

    fig.update_layout(
        **scene_layout(title, g),
        sliders=[{
            "active": 0,
            "currentvalue": {
                "prefix": "First visible layer k = "
            },
            "pad": {"t": 45},
            "steps": steps,
        }],
        legend={
            "x": .01, "y": .99,
            "bgcolor": "rgba(255,255,255,.75)",
        },
    )
    return fig


def make_diagenesis_figure(res: dict):
    """
    Dissolution/cementation/dolomitization can each be inspected in a
    separate, independently toggleable 3D point cloud.
    """
    go = plotly_go()
    g = res["grid"]
    fig = go.Figure()

    specifications = (
        ("dissolution", "#df3b36"),
        ("cementation", "#3a64ba"),
        ("dolomitization", "#41aa77"),
    )
    for name, color in specifications:
        a = res["diagenesis"][name]
        m = a >= .65
        fig.add_trace(go.Scatter3d(
            x=g["X"][m], y=g["Y"][m], z=g["depth"][m],
            mode="markers",
            marker={"size": 4, "color": color, "opacity": .66},
            name=name + " >= 0.65",
            customdata=a[m],
            hovertemplate=(
                f"{name}"
                "<br>intensity=%{customdata:.2f}"
                "<br>depth=%{z:.1f} m<extra></extra>"
            ),
        ))

    add_fault_surface(fig, res)
    add_fracture_points(fig, res)
    add_well_traces(fig, res)
    fig.update_layout(
        **scene_layout(
            "3D diagenesis: high-intensity cells", g
        )
    )
    return fig


def make_property_slice(res: dict, key: str, logarithmic: bool):
    go = plotly_go()
    g = res["grid"]
    arr = res[key]
    display = np.log10(arr) if logarithmic else arr
    label = "log10(" + key + " [mD])" if logarithmic else key

    def heatmap_for_layer(k: int):
        return go.Heatmap(
            x=g["x"],
            y=g["y"],
            z=display[:, :, k].T,
            colorscale="Viridis",
            colorbar={"title": label},
            hovertemplate=(
                "x=%{x:.1f} m"
                "<br>y=%{y:.1f} m"
                f"<br>{label}=%{{z:.3f}}"
                "<extra></extra>"
            ),
        )

    fig = go.Figure(data=[heatmap_for_layer(2)])
    for w in res["wells"]:
        fig.add_trace(go.Scatter(
            x=[w["cell_x"]],
            y=[w["cell_y"]],
            mode="markers+text",
            marker={
                "size": 13,
                "color": (
                    "blue" if w["kind"] == "injector" else "red"
                ),
                "line": {"color": "black", "width": 1},
            },
            text=[w["name"]],
            textposition="top center",
            name=w["name"],
        ))

    # Slices change only the heatmap, leaving wells in place.
    frames = [
        go.Frame(
            data=[heatmap_for_layer(k)],
            traces=[0],
            name=str(k),
        )
        for k in range(g["Nz"])
    ]
    fig.frames = frames
    fig.update_layout(
        title=f"{label}: layer k=2",
        height=650,
        xaxis={"title": "x [m]", "scaleanchor": "y"},
        yaxis={"title": "y [m]"},
        sliders=[{
            "active": 2,
            "currentvalue": {"prefix": "Layer k = "},
            "steps": [
                {
                    "label": str(k),
                    "method": "animate",
                    "args": [
                        [str(k)],
                        {
                            "mode": "immediate",
                            "frame": {
                                "duration": 0,
                                "redraw": True,
                            },
                            "transition": {"duration": 0},
                        },
                    ],
                }
                for k in range(g["Nz"])
            ],
        }],
    )
    return fig


def make_fault_figure(res: dict):
    go = plotly_go()
    fig = go.Figure()
    add_fault_surface(fig, res)

    mask = res["faults"]["cell_mask"] > 0
    g = res["grid"]
    fig.add_trace(go.Scatter3d(
        x=g["X"][mask],
        y=g["Y"][mask],
        z=g["depth"][mask],
        mode="markers",
        marker={"size": 3, "color": "black"},
        name="fault-zone cells",
    ))
    add_well_traces(fig, res)
    fig.update_layout(
        **scene_layout(
            "Fault surface, cell mask and wells "
            "(face multipliers are saved separately)", g
        )
    )
    return fig


def make_fracture_figure(res: dict):
    go = plotly_go()
    fig = go.Figure()
    add_fracture_points(fig, res)
    add_fault_surface(fig, res)
    add_well_traces(fig, res)
    fig.update_layout(
        **scene_layout(
            "Equivalent-continuum fracture corridors "
            "(magenta cells)", res["grid"]
        )
    )
    return fig


def make_well_figure(res: dict):
    go = plotly_go()
    fig = go.Figure()
    add_well_traces(fig, res)
    add_fault_surface(fig, res)
    add_fracture_points(fig, res)
    add_owc_trace(fig, res)
    fig.update_layout(
        **scene_layout(
            "Injector and producer; diamonds are perforations",
            res["grid"],
        )
    )
    return fig


def make_pressure_figure(res: dict):
    go = plotly_go()
    g = res["grid"]
    P = res["initial_state"]["pressure"]
    D = g["depth"]

    fig = go.Figure()
    fig.add_trace(go.Scattergl(
        x=(P / 1e6).ravel(),
        y=D.ravel(),
        mode="markers",
        marker={"size": 4, "opacity": .45},
        name="water pressure at cell centres",
        hovertemplate=(
            "P=%{x:.3f} MPa"
            "<br>depth=%{y:.2f} m<extra></extra>"
        ),
    ))
    fig.add_hline(
        y=res["config"].owc_depth_m,
        line_dash="dash",
        line_color="cyan",
        annotation_text="OWC",
    )
    fig.update_layout(
        title="Initial hydrostatic water pressure with depth",
        xaxis_title="Pressure [MPa]",
        yaxis_title="Depth [m], positive downward",
        yaxis={"autorange": "reversed"},
        height=650,
    )
    return fig


def make_saturation_figure(res: dict):
    go = plotly_go()
    g = res["grid"]
    # Show an x-depth section through the producer's y-index.
    j = next(
        w["j"] for w in res["wells"]
        if w["kind"] == "producer"
    )
    Sw = res["initial_state"]["Sw"][:, j, :]
    D = g["depth"][:, j, :]
    X = g["X"][:, j, :]

    fig = go.Figure(go.Heatmap(
        x=X.ravel(),
        y=D.ravel(),
        z=Sw.ravel(),
        colorscale="RdBu",
        zmin=0,
        zmax=1,
        colorbar={"title": "Sw"},
        hovertemplate=(
            "x=%{x:.1f} m"
            "<br>depth=%{y:.1f} m"
            "<br>Sw=%{z:.3f}<extra></extra>"
        ),
    ))
    # Plotly Heatmap expects a regular 2D z array; structural depth is
    # not a regular y grid. Replace with a 3D section-scatter for accurate
    # cell-centred coordinates rather than misrepresenting the structure.
    fig = go.Figure(go.Scattergl(
        x=X.ravel(),
        y=D.ravel(),
        mode="markers",
        marker={
            "size": 14,
            "symbol": "square",
            "color": Sw.ravel(),
            "colorscale": "RdBu",
            "cmin": 0, "cmax": 1,
            "colorbar": {"title": "Sw"},
        },
        text=np.tile(np.arange(g["Nz"]), g["Nx"]),
        hovertemplate=(
            "x=%{x:.1f} m"
            "<br>depth=%{y:.1f} m"
            "<br>layer=%{text}"
            "<br>Sw=%{marker.color:.3f}<extra></extra>"
        ),
        name="Sw cell centres",
    ))
    fig.add_hline(
        y=res["config"].owc_depth_m,
        line_dash="dash",
        line_color="black",
        annotation_text="OWC",
    )
    fig.update_layout(
        title=f"Initial water saturation and OWC, y-index j={j}",
        xaxis_title="x [m]",
        yaxis_title="depth [m], positive downward",
        yaxis={"autorange": "reversed"},
        height=650,
    )
    return fig


def make_crossplot(res: dict):
    go = plotly_go()
    phi = res["phi"]
    kx = res["kx_matrix"]
    facies = res["facies"]
    fig = go.Figure()

    for code, name in FACIES.items():
        m = facies == code
        fig.add_trace(go.Scattergl(
            x=phi[m],
            y=kx[m],
            mode="markers",
            marker={
                "size": 5,
                "color": FACIES_COLORS[code],
                "opacity": .48,
            },
            name=name,
            hovertemplate=(
                f"{name}"
                "<br>phi=%{x:.3f}"
                "<br>matrix kx=%{y:.4g} mD"
                "<extra></extra>"
            ),
        ))

    net = np.isin(facies, [1, 2, 3])
    r2_net = logk_phi_r2(phi, kx, net)
    r2_all = logk_phi_r2(
        phi, kx, np.ones(phi.shape, dtype=bool)
    )
    fig.update_layout(
        title=(
            "Carbonate matrix porosity–permeability cross-plot "
            f"(R² net={r2_net:.3f}, all={r2_all:.3f}; "
            "linear fit to log10 k)"
        ),
        xaxis_title="Total porosity [fraction]",
        yaxis_title="Matrix kx [mD, log axis]",
        yaxis_type="log",
        height=650,
    )
    return fig


def save_interactive_figures(res: dict, directory: str) -> None:
    folder = Path(directory)
    folder.mkdir(parents=True, exist_ok=True)

    figures = {
        "01_depositional_facies_3d.html": make_cutaway_figure(
            res, res["facies"], FACIES, FACIES_COLORS,
            "3D carbonate depositional facies",
        ),
        "02_pore_types_3d.html": make_cutaway_figure(
            res, res["pore_type"], PORE_TYPES, PORE_COLORS,
            "3D dominant pore types",
        ),
        "03_porosity_slice.html": make_property_slice(
            res, "phi", logarithmic=False
        ),
        "04_permeability_slice_log.html": make_property_slice(
            res, "kx", logarithmic=True
        ),
        "05_diagenesis_3d.html": make_diagenesis_figure(res),
        "06_fault_3d.html": make_fault_figure(res),
        "07_fractures_3d.html": make_fracture_figure(res),
        "08_wells_3d.html": make_well_figure(res),
        "09_pressure_depth.html": make_pressure_figure(res),
        "10_saturation_owc.html": make_saturation_figure(res),
        "11_phi_k_crossplot.html": make_crossplot(res),
    }

    for filename, fig in figures.items():
        # Include Plotly JS in each file: independently viewable offline.
        fig.write_html(
            folder / filename,
            include_plotlyjs=True,
            full_html=True,
        )
        print("Saved:", folder / filename)


def main() -> None:
    cfg = Config()
    reservoir = build_reservoir(cfg)
    report = validate_reservoir(reservoir)

    output = Path("carbonate_output")
    npz_path, json_path = save_reservoir(reservoir, str(output))
    save_interactive_figures(
        reservoir, str(output / "figures")
    )

    print("\nSYNTHETIC CARBONATE RESERVOIR")
    print(
        f"Grid: {cfg.Nx} x {cfg.Ny} x {cfg.Nz}; "
        f"dimensions: {cfg.Lx:g} x {cfg.Ly:g} x {cfg.Lz:g} m"
    )
    print("Seed:", cfg.seed)
    print(
        "Matrix phi/log10(kx) R²:",
        f"net={report['phi_log10_matrix_kx_r2_net']:.3f},",
        f"all={report['phi_log10_matrix_kx_r2_all_cells']:.3f}",
    )
    print("Fault-crossing faces:", report["fault_faces"])
    print("Fracture-corridor cells:", report["fracture_cells"])
    for w in reservoir["wells"]:
        print(
            f"{w['name']}: cell ({w['i']},{w['j']}), "
            f"perforated layers {w['perforated_layers']}"
        )
    print("Arrays:", npz_path)
    print("Metadata:", json_path)
    print(
        "\nOpen carbonate_output/figures/"
        "01_depositional_facies_3d.html in a browser."
    )
    print(
        "In 3D: left-drag rotates; right-drag pans; "
        "scroll zooms; the slider cuts away upper layers."
    )


if __name__ == "__main__":
    main()

SyntaxError: invalid syntax. Perhaps you forgot a comma? (718808182.py, line 1586)

In [1]:
"""
carbonate_reservoir.py
Research-grade SYNTHETIC 3D carbonate reservoir initialization.
NOT A REAL FIELD. All ranges illustrative unless cited.
"""
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

try:
    from scipy.ndimage import gaussian_filter
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False
    def gaussian_filter(a, sigma=None, mode="reflect"):
        print("WARNING: scipy missing, GRF unsmoothed (white noise).")
        return a

# ---------------- constants ----------------
FACIES_NAMES = {0:"Mudstone/Wackestone",1:"Packstone",2:"Grainstone",
                3:"Boundstone/Reef",4:"Dolomitized",5:"Evaporite/Anhydrite"}
PORE_NAMES = {0:"Interparticle",1:"Intraparticle",2:"Moldic",
              3:"Vuggy",4:"Microporosity",5:"Fracture"}
# facies: (phi_mean,phi_std,phi_min,phi_max)
FACIES_PHI = {
 0:(0.07,0.02,0.03,0.12), 1:(0.11,0.025,0.05,0.18),
 2:(0.15,0.03,0.08,0.22), 3:(0.13,0.05,0.04,0.25),
 4:(0.16,0.03,0.08,0.254),5:(0.02,0.01,0.005,0.05)}
# facies: (mean_log10k[mD],std,min_log,max_log,kvkh)
FACIES_LOGK = {
 0:(-0.30,0.60,-2.22,0.70,0.05), 1:(0.70,0.50,-1.00,1.48,0.10),
 2:(1.40,0.40,0.00,2.16,0.25),  3:(1.18,0.80,-1.30,2.16,0.15),
 4:(1.54,0.40,0.00,2.16,0.20),  5:(-1.70,0.40,-2.30,-1.00,0.05)}
SWC_PORE = {0:0.20,1:0.35,2:0.30,3:0.28,4:0.50,5:0.05}
FACIES_CMAP_COLORS = ["#8c8c8c","#f2e394","#f2b134","#e94e34","#6a4c93","#00a8b5"]
PORE_CMAP_COLORS = ["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#000000"]

def _sigmoid(x): return 1.0/(1.0+np.exp(-x))

# ---------------- grid ----------------
def create_grid(Nx=40,Ny=40,Nz=10,Lx=1000.,Ly=1000.,Lz=50.,
                x0=0.,y0=0.,top_depth=2500.):
    dx,dy,dz = Lx/Nx, Ly/Ny, Lz/Nz
    xc_1d = x0+(np.arange(Nx)+0.5)*dx
    yc_1d = y0+(np.arange(Ny)+0.5)*dy
    zc_1d = top_depth+(np.arange(Nz)+0.5)*dz
    Xc,Yc,Zc = np.meshgrid(xc_1d,yc_1d,zc_1d,indexing="ij")
    return {"Nx":Nx,"Ny":Ny,"Nz":Nz,"Lx":Lx,"Ly":Ly,"Lz":Lz,
            "dx":dx,"dy":dy,"dz":dz,"x0":x0,"y0":y0,"top_depth":top_depth,
            "xc_1d":xc_1d,"yc_1d":yc_1d,"zc_1d":zc_1d,"Xc":Xc,"Yc":Yc,"Zc":Zc}

def gaussian_random_field(shape, sigma_cells, rng, mean=0., std=1.):
    white = rng.standard_normal(shape)
    smooth = gaussian_filter(white, sigma=sigma_cells, mode="reflect") if HAS_SCIPY else white
    s = np.std(smooth); s = s if s>1e-12 else 1.0
    return mean+std*(smooth-np.mean(smooth))/s

# ---------------- facies ----------------
def build_depositional_facies(grid, rng):
    Nx,Ny,Nz = grid["Nx"],grid["Ny"],grid["Nz"]
    Xc,Yc = grid["Xc"],grid["Yc"]; Lx = grid["Lx"]
    noise_reef = gaussian_random_field((Nx,Ny,Nz),(4,4,1),rng)
    noise_lith = gaussian_random_field((Nx,Ny,Nz),(3,3,1),rng)
    noise_evap = gaussian_random_field((Nx,Ny,Nz),(5,5,1),rng)
    shoals = [(250,450,110,55),(450,500,140,60),(650,450,120,55),
              (350,380,90,45),(750,550,80,50)]
    facies_dep = np.zeros((Nx,Ny,Nz), dtype=np.int8)
    for k in range(Nz):
        y_reef_k = 600.0+(Nz-1-k)*8.0
        Xk, Yk = Xc[:,:,k], Yc[:,:,k]
        y_line = y_reef_k+60.*np.sin(2*np.pi*Xk/Lx)+noise_reef[:,:,k]*25.
        d = Yk-y_line
        shoal_mask = np.zeros((Nx,Ny),bool)
        grf_shoal = np.zeros((Nx,Ny),bool)
        if k in [1,2,3,4,5,6]:
            for cx,cy,rx,ry in shoals:
                shoal_mask |= ((Xk-cx)/rx)**2+((Yk-cy)/ry)**2 < 1.0
            grf_shoal = noise_lith[:,:,k]>0.7
        fk = np.zeros((Nx,Ny),dtype=np.int8)
        m_basin=d>120; m_fore=(d>40)&(d<=120); m_reef=(d>-40)&(d<=40)
        m_back=(d>-260)&(d<=-40); m_lag=d<=-260
        fk[m_basin]=0
        nl = noise_lith[:,:,k]
        fk[m_fore]=np.where(nl[m_fore]<-0.3,0,1)
        fk[m_reef]=np.where(nl[m_reef]<-0.8,1,3)
        fk[m_back]=1
        is_shoal=(shoal_mask|grf_shoal)&m_back
        fk[is_shoal]=2
        fk[m_back&(~is_shoal)&(nl<-1.0)]=0
        fk[m_lag]=np.where(nl[m_lag]>0.2,1,0)
        if k==0:
            fk[(m_lag|m_back)&(noise_evap[:,:,k]>0.2)]=5
        if k==5:
            fk[m_lag&(noise_evap[:,:,k]>0.8)]=5
        facies_dep[:,:,k]=fk
    return facies_dep, {"shoals":shoals}

# ---------------- faults ----------------
def create_faults(grid, facies=None, x0=650., dip_deg=70., amp=30.):
    Nx,Ny,Nz=grid["Nx"],grid["Ny"],grid["Nz"]
    Xc,Yc,Zc=grid["Xc"],grid["Yc"],grid["Zc"]
    cot=1.0/np.tan(np.radians(dip_deg))
    x_f=x0+(Zc-grid["top_depth"])*cot+amp*np.sin(2*np.pi*Yc/grid["Ly"])
    sdf=Xc-x_f
    cell_mask=np.abs(sdf)<(grid["dx"]*0.6)
    face_x_mask=(sdf[:-1,:,:]*sdf[1:,:,:]<0)
    face_y_mask=(sdf[:,:-1,:]*sdf[:,1:,:]<0)
    face_x_mult=np.ones((Nx-1,Ny,Nz)); face_y_mult=np.ones((Nx,Ny-1,Nz))
    Yfx=0.5*(Yc[:-1,:,:]+Yc[1:,:,:])
    base=np.where(Yfx<500,0.05,2.0)
    face_x_mult[face_x_mask]=base[face_x_mask]
    if facies is not None:
        f1,f2=facies[:-1,:,:],facies[1:,:,:]
        evap=((f1==5)|(f2==5))&face_x_mask
        face_x_mult[evap]=0.01
    face_y_mult[face_y_mask]=0.5
    cell_trans=np.ones((Nx,Ny,Nz)); cell_trans[cell_mask]=np.where(Yc[cell_mask]<500,0.05,2.0)
    params=[{"name":"F-01","type":"normal","x0":float(x0),"dip_deg":float(dip_deg),
             "strike":"N-S","amplitude":float(amp),"trans_south":0.05,"trans_north":2.0,
             "equation":"x_f(y,z)=x0+(z-ztop)*cot(dip)+A*sin(2*pi*y/Ly)"}]
    return {"cell_mask":cell_mask,"face_x_mask":face_x_mask,"face_y_mask":face_y_mask,
            "face_x_mult":face_x_mult,"face_y_mult":face_y_mult,
            "cell_trans_mult":cell_trans,"params":params,"sdf":sdf}

# ---------------- fractures ----------------
def create_fracture_corridors(grid):
    Nx,Ny,Nz=grid["Nx"],grid["Ny"],grid["Nz"]
    Xc,Yc,Zc=grid["Xc"],grid["Yc"],grid["Zc"]
    corridors=[
      {"name":"FC-01","center":[300,650],"azimuth_deg":30,"length":600,"width":25,
       "z_top":2510,"z_bottom":2540,"perm_mult":500,"phi_add":0.004,"aperture_mm":2.0},
      {"name":"FC-02","center":[650,350],"azimuth_deg":120,"length":450,"width":25,
       "z_top":2505,"z_bottom":2535,"perm_mult":300,"phi_add":0.003,"aperture_mm":1.5},
      {"name":"FC-03","center":[500,500],"azimuth_deg":75,"length":350,"width":50,
       "z_top":2520,"z_bottom":2550,"perm_mult":800,"phi_add":0.005,"aperture_mm":3.0}]
    cell_mask=np.zeros((Nx,Ny,Nz),bool); perm_mult=np.ones((Nx,Ny,Nz))
    phi_add=np.zeros((Nx,Ny,Nz)); aperture=np.zeros((Nx,Ny,Nz))
    X2d,Y2d=Xc[:,:,0],Yc[:,:,0]
    for c in corridors:
        az=np.radians(c["azimuth_deg"]); dxd, dyd=np.sin(az),np.cos(az)
        cx,cy=c["center"]; hl=c["length"]/2
        p0x,p0y=cx-dxd*hl,cy-dyd*hl; p1x,p1y=cx+dxd*hl,cy+dyd*hl
        Dx,Dy=p1x-p0x,p1y-p0y; L2=Dx*Dx+Dy*Dy+1e-12
        t=np.clip(((X2d-p0x)*Dx+(Y2d-p0y)*Dy)/L2,0,1)
        dist=np.sqrt((X2d-(p0x+t*Dx))**2+(Y2d-(p0y+t*Dy))**2)
        in_w=dist<(c["width"]/2)
        c["p0"]=[float(p0x),float(p0y)]; c["p1"]=[float(p1x),float(p1y)]
        for k in range(Nz):
            zc=grid["zc_1d"][k]
            if c["z_top"]<=zc<=c["z_bottom"]:
                m=in_w
                cell_mask[:,:,k]|=m
                sl=perm_mult[:,:,k]; sl[m]=np.maximum(sl[m],c["perm_mult"]); perm_mult[:,:,k]=sl
                sl=phi_add[:,:,k]; sl[m]=np.maximum(sl[m],c["phi_add"]); phi_add[:,:,k]=sl
                sl=aperture[:,:,k]; sl[m]=np.maximum(sl[m],c["aperture_mm"]); aperture[:,:,k]=sl
    return {"corridors":corridors,"cell_mask":cell_mask,"perm_mult":perm_mult,
            "phi_add":phi_add,"aperture_mm":aperture,"model":"equivalent-continuum, NOT DFM"}

# ---------------- diagenesis ----------------
def generate_diagenesis(grid, facies_dep, faults, fractures, rng):
    Nx,Ny,Nz=grid["Nx"],grid["Ny"],grid["Nz"]; Zc=grid["Zc"]
    if HAS_SCIPY:
        fprox=np.clip(gaussian_filter(faults["cell_mask"].astype(float),(2,2,1))*3,0,1)
        frprox=np.clip(gaussian_filter(fractures["cell_mask"].astype(float),(3,3,1))*2.5,0,1)
    else:
        fprox=faults["cell_mask"].astype(float); frprox=fractures["cell_mask"].astype(float)
    g_d=gaussian_random_field((Nx,Ny,Nz),(4,4,1.2),rng)
    g_c=gaussian_random_field((Nx,Ny,Nz),(5,5,1.0),rng)
    g_o=gaussian_random_field((Nx,Ny,Nz),(6,6,1.5),rng)
    is_good=(facies_dep==3).astype(float)+(facies_dep==2).astype(float)+0.5*(facies_dep==1).astype(float)
    depth_n=(Zc-grid["top_depth"])/grid["Lz"]; upper=1-depth_n
    bedding=np.zeros((Nx,Ny,Nz))
    for kb in [2,6]:
        for k in range(Nz): bedding[:,:,k]+=np.exp(-((k-kb)**2)/1.0)
    bedding=np.clip(bedding,0,1)
    stylo=np.zeros((Nx,Ny,Nz)); stylo[:,:,4]=1.0; stylo[:,:,7]=1.0
    diss=_sigmoid(0.9*g_d+2.2*frprox+1.0*fprox+0.8*is_good+0.6*upper+0.8*bedding-0.5)
    cem=_sigmoid(0.9*g_c+1.5*stylo+0.6*(facies_dep==0).astype(float)+0.4*depth_n-0.3)
    dolo=_sigmoid(0.8*g_o+1.8*fprox+0.9*frprox+0.9*((facies_dep==2).astype(float)+(facies_dep==1).astype(float)+0.7*(facies_dep==3).astype(float))+0.5*upper-0.8)
    gain=np.where(np.isin(facies_dep,[1,2,3]),0.030,-0.010)
    dphi=0.06*diss-0.05*cem+gain*dolo
    f=lambda a:a.astype(np.float64)
    return {"dissolution":f(diss),"cementation":f(cem),"dolomitization":f(dolo),
            "dphi":f(dphi),"fault_proximity":f(fprox),"fracture_proximity":f(frprox),
            "bedding_factor":f(bedding),"stylolite_factor":f(stylo)}

def apply_dolomitization(facies_dep, diagenesis, threshold=0.62):
    f=facies_dep.copy()
    f[(diagenesis["dolomitization"]>threshold)&np.isin(facies_dep,[1,2,3])]=4
    return f

# ---------------- pore types ----------------
def assign_pore_types(grid, facies, diagenesis, fractures, rng):
    Nx,Ny,Nz=grid["Nx"],grid["Ny"],grid["Nz"]
    base=np.array([[0.05,0.25,0.05,0.05,0.60,0.0],[0.30,0.25,0.10,0.10,0.25,0.0],
                   [0.50,0.10,0.20,0.15,0.05,0.0],[0.25,0.10,0.20,0.35,0.10,0.0],
                   [0.45,0.05,0.15,0.25,0.10,0.0],[0.0,0.0,0.0,0.0,1.0,0.0]])
    diss,cem,dolo=diagenesis["dissolution"],diagenesis["cementation"],diagenesis["dolomitization"]
    fmask=fractures["cell_mask"]; pt=np.zeros((Nx,Ny,Nz),dtype=np.int8)
    for idx in np.ndindex(Nx,Ny,Nz):
        f=int(facies[idx]); p=base[f].copy()
        d,c,o=float(diss[idx]),float(cem[idx]),float(dolo[idx])
        if d>0.4: p[3]+=(d-0.4)*0.6; p[2]+=(d-0.4)*0.4; p[4]=max(0.02,p[4]-(d-0.4)*0.3)
        if c>0.4: p[4]+=(c-0.4)*0.5; p[0]=max(0.01,p[0]-(c-0.4)*0.25); p[3]=max(0.01,p[3]-(c-0.4)*0.15)
        if o>0.5: p[0]+=0.10; p[3]+=0.08
        if fmask[idx]:
            if rng.random()<0.75: pt[idx]=5; continue
            else: p[3]+=0.20
        p=np.clip(p,0,None); s=p.sum(); p=p/s if s>0 else base[f]
        pt[idx]=rng.choice(6,p=p)
    return pt

# ---------------- porosity / perm ----------------
def generate_porosity(grid, facies, diagenesis, fractures, rng):
    shape=(grid["Nx"],grid["Ny"],grid["Nz"])
    grf=gaussian_random_field(shape,(5,5,1.2),rng)
    phi=np.zeros(shape); dphi=diagenesis["dphi"]
    for f,(mean,std,vmin,vmax) in FACIES_PHI.items():
        m=(facies==f)
        if not np.any(m): continue
        white=rng.standard_normal(np.count_nonzero(m))
        vals=mean+std*grf[m]+0.005*white+dphi[m]
        phi[m]=np.clip(vals,vmin,vmax)
    phi=np.clip(phi+fractures["phi_add"],0.005,0.30)
    return phi.astype(np.float64)

def generate_permeability(grid, facies, pore_type, diagenesis, fractures, phi, rng):
    shape=(grid["Nx"],grid["Ny"],grid["Nz"])
    gk=gaussian_random_field(shape,(4,4,1.0),rng)
    ga=gaussian_random_field(shape,(4,4,1.0),rng)
    gz=gaussian_random_field(shape,(3,3,1.0),rng)
    diss,cem,dolo=diagenesis["dissolution"],diagenesis["cementation"],diagenesis["dolomitization"]
    is_v=(pore_type==3); conn=is_v&((diss>0.4)|(diagenesis["fracture_proximity"]>0.3))
    pshift=np.zeros(shape); pshift[pore_type==1]=-0.5; pshift[pore_type==2]=-0.3
    pshift[conn]=0.7; pshift[is_v&(~conn)]=-0.2; pshift[pore_type==4]=-1.0
    dia=1.0*diss-0.8*cem+0.4*dolo
    logk=np.zeros(shape); kvkh_arr=np.zeros(shape); b=2.0
    for f,(mlog,stdlog,minlog,maxlog,kvkh) in FACIES_LOGK.items():
        m=(facies==f)
        if not np.any(m): continue
        n=np.count_nonzero(m); white=rng.standard_normal(n); extra=rng.standard_normal(n)*0.4
        base=mlog+stdlog*(0.75*gk[m]+0.25*white)
        vals=base+b*(phi[m]-FACIES_PHI[f][0])+pshift[m]+dia[m]+extra
        vals=np.clip(vals,minlog-0.3,maxlog+0.5)
        logk[m]=vals; kvkh_arr[m]=kvkh
    kx_m=np.clip(10**logk,0.005,500)
    ky_m=np.clip(kx_m*10**(0.08*ga),0.005,500)
    sty=1-0.95*diagenesis["stylolite_factor"]; sty[fractures["cell_mask"]]=1.0
    kz_m=np.clip(kx_m*kvkh_arr*10**(0.15*gz)*sty,0.001,500)
    fm=fractures["perm_mult"]
    kx=np.clip(kx_m*fm,0.005,8000); ky=np.clip(ky_m*fm,0.005,8000); kz=np.clip(kz_m*fm,0.001,8000)
    kz[fractures["cell_mask"]]=np.maximum(kz[fractures["cell_mask"]],0.8*kx[fractures["cell_mask"]])
    return kx,ky,kz,kx_m,ky_m,kz_m,logk

def compute_dpdp_arrays(grid, phi, kx,ky,kz, kxm,kym,kzm, fractures, pore_type, diagenesis):
    phi_frac=fractures["phi_add"].astype(float).copy()
    conn=(pore_type==3)&(diagenesis["dissolution"]>0.45)
    phi_frac[conn]+=0.15*phi[conn]
    phi_frac=np.minimum(np.maximum(phi_frac,0.0),0.9*phi)
    phi_mat=phi-phi_frac
    kxf=np.zeros_like(kx); kyf=np.zeros_like(ky); kzf=np.zeros_like(kz)
    m=fractures["cell_mask"]; kxf[m]=kx[m]; kyf[m]=ky[m]; kzf[m]=kz[m]
    sigma=np.full_like(phi,0.001); sigma[m]=0.0432
    sigma[conn]=np.maximum(sigma[conn],0.01)
    return {"phi_matrix":phi_mat,"phi_fracture":phi_frac,"kx_matrix":kxm,"ky_matrix":kym,
            "kz_matrix":kzm,"kx_fracture":kxf,"ky_fracture":kyf,"kz_fracture":kzf,
            "shape_factor":sigma}

# ---------------- pressure / saturations ----------------
def initialize_pressure(grid, Pref=25e6, z_ref=None, rho=900., g=9.81):
    if z_ref is None: z_ref=grid["top_depth"]
    return (Pref+rho*g*(grid["Zc"]-z_ref)).astype(np.float64)

def initialize_saturations(grid, facies, pore_type, phi, OWC, rng, lam=5.0):
    shape=(grid["Nx"],grid["Ny"],grid["Nz"]); Zc=grid["Zc"]
    Swc=np.zeros(shape)
    for k,v in SWC_PORE.items(): Swc[pore_type==k]=v
    Swc[facies==0]+=0.08; Swc[facies==4]-=0.03; Swc[facies==5]=0.90
    Swc=np.clip(Swc+rng.standard_normal(shape)*0.02,0.05,0.95)
    h=OWC-Zc; Sw=np.zeros(shape)
    Sw[h<=0]=1.0; a=h>0
    Sw[a]=Swc[a]+(1-Swc[a])*np.exp(-h[a]/lam)
    Sw=np.clip(Sw,0,1); Sw[facies==5]=1.0
    So=1-Sw; Sg=np.zeros(shape)
    return {"Sw":Sw.astype(np.float64),"So":So.astype(np.float64),
            "Sg":Sg.astype(np.float64),"Swc":Swc.astype(np.float64),"OWC":float(OWC)}

# ---------------- wells ----------------
def define_wells(grid, facies, phi, kx, diagenesis, sats, OWC):
    Nx,Ny,Nz=grid["Nx"],grid["Ny"],grid["Nz"]; Zc=grid["Zc"]; dz=grid["dz"]
    safe=OWC-12.0
    pay=np.isin(facies,[1,2,3,4])&(phi>0.08)&(kx>1.0)&(diagenesis["cementation"]<0.6)&(Zc<=safe)
    score=np.sum(phi*np.log10(kx+1)*pay*dz,axis=2)
    X2d,Y2d=grid["Xc"][:,:,0],grid["Yc"][:,:,0]
    prod_reg=(X2d>=200)&(X2d<=800)&(Y2d>=450)&(Y2d<=800)
    inj_reg=(X2d>=200)&(X2d<=800)&(Y2d>=100)&(Y2d<=450)
    pi=np.unravel_index(np.argmax(np.where(prod_reg,score,-1)),(Nx,Ny))
    ii=np.unravel_index(np.argmax(np.where(inj_reg,score,-1)),(Nx,Ny))
    def pick(ij, pos):
        i,j=ij; cand=np.where(pay[i,j,:])[0]
        if len(cand)==0:
            rel=np.isin(facies[i,j,:],[1,2,3,4])&(Zc[i,j,:]<=safe)&(phi[i,j,:]>0.06)
            cand=np.where(rel)[0]
        if len(cand)==0: raise ValueError(f"No pay at {ij}")
        cand=np.sort(cand)
        return sorted((cand[:3] if pos=="upper" else cand[-3:]).tolist())
    pp, ipick=pick(pi,"upper"),pick(ii,"lower")
    mk=lambda n,t,ij,pp,ctl:{"name":n,"type":t,"ij":[int(ij[0]),int(ij[1])],
        "x":float(grid["xc_1d"][ij[0]]),"y":float(grid["yc_1d"][ij[1]]),
        "perforations":pp,"perforation_depths":[float(grid["zc_1d"][k]) for k in pp],
        "control":ctl,"radius_m":0.1,"skin":0.0}
    return [mk("PROD-01","producer",pi,pp,{"mode":"BHP","BHP_bar":150.0,"oil_rate_limit_m3day":200.0}),
            mk("INJ-01","injector",ii,ipick,{"mode":"RATE","water_rate_m3day":300.0,"BHP_limit_bar":320.0})]

# ---------------- build ----------------
def build_reservoir(seed=42,Nx=40,Ny=40,Nz=10,Lx=1000.,Ly=1000.,Lz=50.,
                    top_depth=2500.,OWC=2535.,Pref_MPa=25.0):
    rng=np.random.default_rng(seed)
    grid=create_grid(Nx,Ny,Nz,Lx,Ly,Lz,top_depth=top_depth)
    fdep,_=build_depositional_facies(grid,rng)
    faults=create_faults(grid,facies=fdep)
    fractures=create_fracture_corridors(grid)
    diagen=generate_diagenesis(grid,fdep,faults,fractures,rng)
    facies=apply_dolomitization(fdep,diagen)
    pore=assign_pore_types(grid,facies,diagen,fractures,rng)
    phi=generate_porosity(grid,facies,diagen,fractures,rng)
    kx,ky,kz,kxm,kym,kzm,_=generate_permeability(grid,facies,pore,diagen,fractures,phi,rng)
    dpdp=compute_dpdp_arrays(grid,phi,kx,ky,kz,kxm,kym,kzm,fractures,pore,diagen)
    P=initialize_pressure(grid,Pref=Pref_MPa*1e6,z_ref=top_depth)
    sats=initialize_saturations(grid,facies,pore,phi,OWC,rng)
    wells=define_wells(grid,facies,phi,kx,diagen,sats,OWC)
    meta={"synthetic_warning":"SYNTHETIC - NOT A REAL FIELD. Illustrative ranges.",
        "seed":seed,"grid":{"Nx":Nx,"Ny":Ny,"Nz":Nz,"Lx":Lx,"Ly":Ly,"Lz":Lz,
        "dx":grid["dx"],"dy":grid["dy"],"dz":grid["dz"],"top_depth_m":top_depth},
        "units":{"length":"m","porosity":"fraction","permeability":"mD, 1mD=9.869233e-16m2",
                 "pressure":"Pa","saturation":"fraction"},
        "facies_definitions":{str(k):v for k,v in FACIES_NAMES.items()},
        "pore_definitions":{str(k):v for k,v in PORE_NAMES.items()},
        "OWC_m":float(OWC),"reference_pressure_Pa":float(Pref_MPa*1e6),
        "reference_depth_m":float(top_depth),"formation_compressibility_1Pa":7.0e-10,
        "relperm_params":{"type":"Corey-carbonate-illustrative","Sor_avg":0.342,
         "krw_max":0.6,"kro_max":0.9,"nw":3.5,"no":2.5},
        "capillary_params":{"type":"Brooks-Corey-illustrative","lambda":2.0},
        "faults":faults["params"],"fractures":fractures["corridors"],
        "dpdp":{"enabled":False,"fracture_model":"equivalent-continuum, NOT DFM"},
        "honesty_notes":["Diagenesis-facies interaction poorly understood.",
         "Carbonate ranges wider than sandstone, basin-dependent.",
         "Equivalent continuum is numerical simplification."]}
    return {"grid":grid,"facies":facies,"facies_depositional":fdep,"diagenesis":diagen,
      "pore_type":pore,"phi":phi,"phi_matrix":dpdp["phi_matrix"],"phi_fracture":dpdp["phi_fracture"],
      "kx":kx,"ky":ky,"kz":kz,"kx_matrix":kxm,"ky_matrix":kym,"kz_matrix":kzm,
      "kx_fracture":dpdp["kx_fracture"],"ky_fracture":dpdp["ky_fracture"],"kz_fracture":dpdp["kz_fracture"],
      "shape_factor":dpdp["shape_factor"],"faults":faults,"fractures":fractures,
      "initial_state":{"pressure":P,"Sw":sats["Sw"],"So":sats["So"],"Sg":sats["Sg"],"Swc":sats["Swc"]},
      "wells":wells,"metadata":meta}

# ---------------- validation ----------------
def validate_reservoir(res):
    rep=[]; ok=True
    def chk(c,m):
        nonlocal ok; rep.append(("PASS: " if c else "FAIL: ")+m); ok=ok and bool(c)
    phi,kx,ky,kz=res["phi"],res["kx"],res["ky"],res["kz"]
    Sw,So,Sg=res["initial_state"]["Sw"],res["initial_state"]["So"],res["initial_state"]["Sg"]
    P=res["initial_state"]["pressure"]; fac=res["facies"]
    for f,(mn,st,lo,hi) in FACIES_PHI.items():
        m=(fac==f)
        if np.any(m): chk(np.all(phi[m]>=lo-1e-9)&np.all(phi[m]<=hi+0.006+1e-9),f"phi bounds facies {f}")
    chk(np.allclose(phi,res["phi_matrix"]+res["phi_fracture"]),"phi=matrix+fracture")
    chk(np.all(kx>0)&np.all(ky>0)&np.all(kz>0),"k>0")
    chk(np.allclose(Sw+So+Sg,1.0),"Sw+So+Sg=1")
    chk(np.all((Sw>=0)&(Sw<=1)&(So>=0)&(So<=1)),"saturations [0,1]")
    chk(np.all(np.diff(P,axis=2)>0),"P increasing with depth")
    Nx,Ny,Nz=res["grid"]["Nx"],res["grid"]["Ny"],res["grid"]["Nz"]
    OWC=res["metadata"]["OWC_m"]
    for w in res["wells"]:
        i,j=w["ij"]; chk(0<=i<Nx and 0<=j<Ny,f"{w['name']} inside")
        for k in w["perforations"]:
            f=fac[i,j,k]; zc=res["grid"]["zc_1d"][k]
            chk(f!=5,f"{w['name']} k={k} not evaporite"); chk(f!=0,f"{w['name']} k={k} not mudstone")
            chk((OWC-zc)>=10,f"{w['name']} k={k} >=10m above OWC")
            chk(res["diagenesis"]["cementation"][i,j,k]<0.65,f"{w['name']} k={k} not cemented")
    chk(np.any(res["faults"]["cell_mask"]),"fault non-empty")
    chk(np.any(res["fractures"]["cell_mask"]),"fracture non-empty")
    R2=float(np.corrcoef(phi.flatten(),np.log10(kx.flatten()))[0,1]**2)
    chk(R2<0.5,f"weak phi-k R2={R2:.3f}<0.5")
    for k in ["dissolution","cementation","dolomitization"]:
        a=res["diagenesis"][k]; chk(np.all((a>=0)&(a<=1)),f"{k} in [0,1]")
    return ok,rep,{"R2":R2}

# ---------------- I/O ----------------
def save_reservoir(res, prefix="carbonate_model"):
    p=Path(str(prefix)); npz=str(p)+".npz"; js=str(p)+"_meta.json"
    g=res["grid"]
    np.savez_compressed(npz, facies=res["facies"],facies_depositional=res["facies_depositional"],
      pore_type=res["pore_type"],phi=res["phi"],phi_matrix=res["phi_matrix"],phi_fracture=res["phi_fracture"],
      kx=res["kx"],ky=res["ky"],kz=res["kz"],kx_matrix=res["kx_matrix"],ky_matrix=res["ky_matrix"],
      kz_matrix=res["kz_matrix"],pressure=res["initial_state"]["pressure"],Sw=res["initial_state"]["Sw"],
      So=res["initial_state"]["So"],Sg=res["initial_state"]["Sg"],Swc=res["initial_state"]["Swc"],
      dissolution=res["diagenesis"]["dissolution"],cementation=res["diagenesis"]["cementation"],
      dolomitization=res["diagenesis"]["dolomitization"],shape_factor=res["shape_factor"],
      fault_cell=res["faults"]["cell_mask"],fault_fx=res["faults"]["face_x_mult"],
      frac_cell=res["fractures"]["cell_mask"],frac_mult=res["fractures"]["perm_mult"],
      xc_1d=g["xc_1d"],yc_1d=g["yc_1d"],zc_1d=g["zc_1d"])
    meta=dict(res["metadata"]); meta["wells"]=res["wells"]
    json.dump(meta,open(js,"w"),indent=2)
    print(f"Saved {npz} and {js}"); return npz,js

# ---------------- visualization ----------------
def _imshow(ax,d,title,cmap,vmin=None,vmax=None,label="",extent=None):
    im=ax.imshow(d.T,origin="lower",cmap=cmap,vmin=vmin,vmax=vmax,extent=extent,aspect="auto")
    ax.set_title(title); plt.colorbar(im,ax=ax,label=label)

def plot_all(res,outdir="figs"):
    import os; os.makedirs(outdir,exist_ok=True)
    g=res["grid"]; ex=[0,g["Lx"],0,g["Ly"]]; kmid=g["Nz"]//2
    fc=ListedColormap(FACIES_CMAP_COLORS); pc=ListedColormap(PORE_CMAP_COLORS)
    fig,a=plt.subplots(); _imshow(a,res["facies"][:,:,kmid],f"Facies XY k={kmid}",fc,-0.5,5.5,"facies",ex)
    for w in res["wells"]: a.plot(w["x"],w["y"],"k*" if "PROD" in w["name"] else "b^",ms=12)
    fig.savefig(f"{outdir}/facies_xy.png",dpi=150); plt.close(fig)
    fig,a=plt.subplots(); _imshow(a,res["pore_type"][:,:,kmid],f"Pore type XY k={kmid}",pc,-0.5,5.5,"pore",ex)
    fig.savefig(f"{outdir}/pore_xy.png",dpi=150); plt.close(fig)
    fig,a=plt.subplots(); _imshow(a,res["phi"][:,:,kmid],"Porosity XY","viridis",0,0.3,"phi",ex)
    fig.savefig(f"{outdir}/phi_xy.png",dpi=150); plt.close(fig)
    fig,a=plt.subplots(); _imshow(a,np.log10(res["kx"][:,:,kmid]),"log10 kx XY","plasma",-2,3.9,"log10 mD",ex)
    fig.savefig(f"{outdir}/perm_xy.png",dpi=150); plt.close(fig)
    fig,a=plt.subplots(); _imshow(a,res["diagenesis"]["dissolution"][:,:,kmid]-res["diagenesis"]["cementation"][:,:,kmid],
        "Dissolution-Cementation","seismic",-1,1,"intensity",ex)
    fig.savefig(f"{outdir}/diagenesis_xy.png",dpi=150); plt.close(fig)
    fig=plt.figure(); ax=fig.add_subplot(111,projection="3d")
    m=res["faults"]["cell_mask"]; X,Y,Z=g["Xc"][m],g["Yc"][m],g["Zc"][m]
    ax.scatter(X,Y,Z,c="r",s=20,label="fault"); ax.set_title("Fault geometry"); ax.legend()
    fig.savefig(f"{outdir}/fault_3d.png",dpi=150); plt.close(fig)
    fig=plt.figure(); ax=fig.add_subplot(111,projection="3d")
    m=res["fractures"]["cell_mask"]; ax.scatter(g["Xc"][m],g["Yc"][m],g["Zc"][m],c="b",s=20,label="fracture corridor")
    for w in res["wells"]:
        zs=[g["zc_1d"][k] for k in w["perforations"]]
        ax.plot([w["x"]]*len(zs),[w["y"]]*len(zs),zs,"k-" if "PROD" in w["name"] else "g-",lw=3)
        ax.scatter([w["x"]],[w["y"]],[zs[0]],c="k" if "PROD" in w["name"] else "g",s=80)
    ax.set_title("Fractures + wells"); ax.legend(); fig.savefig(f"{outdir}/fracture_wells_3d.png",dpi=150); plt.close(fig)
    fig,a=plt.subplots(); a.plot(res["initial_state"]["pressure"][20,20,:]/1e5,g["zc_1d"]); a.invert_yaxis()
    a.set_xlabel("P (bar)"); a.set_ylabel("Depth (m)"); a.set_title("Hydrostatic P(z)"); a.grid()
    fig.savefig(f"{outdir}/pressure_depth.png",dpi=150); plt.close(fig)
    fig,a=plt.subplots(); a.plot(res["initial_state"]["Sw"][20,20,:],g["zc_1d"],label="Sw col 20,20")
    a.axhline(res["metadata"]["OWC_m"],c="b",ls="--",label="OWC"); a.invert_yaxis(); a.legend(); a.grid()
    a.set_title("Sw vs depth"); fig.savefig(f"{outdir}/saturation_depth.png",dpi=150); plt.close(fig)
    fig,a=plt.subplots(); sc=a.scatter(res["phi"].flatten(),res["kx"].flatten(),c=res["facies"].flatten(),cmap=fc,vmin=-0.5,vmax=5.5,s=4,alpha=0.5)
    a.set_yscale("log"); a.set_xlabel("phi"); a.set_ylabel("kx (mD)"); plt.colorbar(sc,ax=a,label="facies")
    R2=np.corrcoef(res["phi"].flatten(),np.log10(res["kx"].flatten()))[0,1]**2
    a.set_title(f"phi-k crossplot R2={R2:.3f} (weak=carbonate-like)"); a.grid(True,which="both",alpha=0.3)
    fig.savefig(f"{outdir}/phi_k.png",dpi=150); plt.close(fig)
    print(f"Figures in {outdir}/")

if __name__=="__main__":
    r=build_reservoir(seed=42)
    ok,rep,m=validate_reservoir(r)
    print("\n".join(rep)); print("R2:",m)
    save_reservoir(r,"carbonate_40x40x10")
    plot_all(r,"figs")

PASS: phi bounds facies 0
PASS: phi bounds facies 1
PASS: phi bounds facies 2
PASS: phi bounds facies 3
PASS: phi bounds facies 4
PASS: phi bounds facies 5
PASS: phi=matrix+fracture
PASS: k>0
PASS: Sw+So+Sg=1
PASS: saturations [0,1]
PASS: P increasing with depth
PASS: PROD-01 inside
PASS: PROD-01 k=0 not evaporite
PASS: PROD-01 k=0 not mudstone
PASS: PROD-01 k=0 >=10m above OWC
PASS: PROD-01 k=0 not cemented
PASS: PROD-01 k=1 not evaporite
PASS: PROD-01 k=1 not mudstone
PASS: PROD-01 k=1 >=10m above OWC
PASS: PROD-01 k=1 not cemented
PASS: PROD-01 k=2 not evaporite
PASS: PROD-01 k=2 not mudstone
PASS: PROD-01 k=2 >=10m above OWC
PASS: PROD-01 k=2 not cemented
PASS: INJ-01 inside
PASS: INJ-01 k=1 not evaporite
PASS: INJ-01 k=1 not mudstone
PASS: INJ-01 k=1 >=10m above OWC
PASS: INJ-01 k=1 not cemented
PASS: INJ-01 k=2 not evaporite
PASS: INJ-01 k=2 not mudstone
PASS: INJ-01 k=2 >=10m above OWC
PASS: INJ-01 k=2 not cemented
PASS: INJ-01 k=3 not evaporite
PASS: INJ-01 k=3 not mudstone
PAS